# NLDD Pipeline - ACL 2026

### Mechanistic Components
* **(A) Counterfactual CoT:** Intervention on reasoning steps.
* **(B) NLDD:** Faithfulness measurement.
* **(C) Patching:** Causal localization.
* **(D) Probing:** Representation vs. Control.
* **(E) RSA:** Representational Similarity.
* **(F/G) Geometry:** PCA & TAS.

### Key Metrics:

**1. Normalized Logit Difference Decay (NLDD) - Robust**
Measure of causal faithfulness.
$$\text{NLDD} = \frac{\text{LD}_{\text{clean}} - \text{LD}_{\text{corrupt}}}{|\text{LD}_{\text{clean}}|} \times 100$$

*For categorical tasks, $\text{LD}$ is the standardized final-answer token margin. For GSM8K, the active implementation uses a standardized answer-level sequence margin over teacher-forced numeric answer strings.*
$$A(x,c)=\frac{s(y_{correct}\mid x,c)-\max_j s(y^{(j)}_{incorrect}\mid x,c)}{S_{model}}$$

**2. Clean-Trajectory Efficiency (TAS)**
Measure of reasoning efficiency.
$$\text{TAS} = \frac{\text{Displacement}}{\text{Path Length}} = \frac{\|h_T - h_0\|}{\sum_{t=1}^{T}\|h_t - h_{t-1}\|}$$

**3. Representational Similarity (RSA)**
Measure of computational divergence.
$$\text{RSA} = \text{SpearmanCorr}(\text{RDM}_{\text{clean}}, \text{RDM}_{\text{corrupt}})$$

### Camera-Ready Implementation Notes
* Low-margin rows with $|\mathrm{LD}_{clean}| \leq \epsilon$ are assigned `NLDD = 0` and flagged in the saved audit output.
* GSM8K NLDD now uses answer-level sequence-margin scoring with task-specific numeric distractors and sequence-score calibration.
* The ProntoQA `all_steps` probe is a sample-truth-at-step-token diagnostic, not a local step-state probe.
* TAS is reported as clean-prefix trajectory efficiency, with stepwise outputs left missing when not measured.
* Horizon confidence intervals use raw per-step BCa intervals in the primary analysis path; CSV/SEM reconstruction is retained only as a legacy fallback.



In [ ]:
# ============================================================================
# SECTION 0: Installation & Imports
# ============================================================================

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import random
import copy
import json
import os
import shutil
import gc
import warnings
from datetime import datetime
from tqdm.auto import tqdm
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Any, Union, Callable
from abc import ABC, abstractmethod
from enum import Enum
from collections import defaultdict

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

from sklearn.linear_model import LogisticRegression, Ridge, RidgeClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

from scipy import stats
from scipy.stats import bootstrap, mannwhitneyu, wilcoxon
import statsmodels.stats.api as sms
from statsmodels.stats.multitest import multipletests

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from typing import List, Dict, Tuple
from dataclasses import dataclass
from collections import defaultdict
import warnings
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import numpy as np
import seaborn as sns
import os
from scipy import stats

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from scipy import stats

import numpy as np
from typing import Dict, Tuple, List, Optional
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist, squareform
from sklearn.model_selection import KFold
from dataclasses import dataclass
import warnings

import gc

warnings.filterwarnings('ignore', category=FutureWarning)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.9.1
CUDA available: False
zsh:1: command not found: nvidia-smi


In [2]:
import os
from getpass import getpass
from huggingface_hub import login, notebook_login


def _read_token_from_env_file() -> str | None:
    """Read HF token from .ENV/.env if present."""
    for path in (".ENV", ".env"):
        if not os.path.exists(path):
            continue
        try:
            with open(path, "r", encoding="utf-8") as f:
                for raw_line in f:
                    line = raw_line.strip()
                    if not line or line.startswith("#"):
                        continue
                    if line.startswith("export "):
                        line = line[len("export "):].strip()
                    if "=" not in line:
                        continue
                    key, value = line.split("=", 1)
                    key = key.strip()
                    value = value.strip().strip('"').strip("'")
                    if key in ("HF_TOKEN", "HF_Token") and value:
                        return value
        except Exception:
            continue
    return None


def resolve_hf_token() -> str | None:
    token = None

    # 1) Prefer Colab Secret first
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or userdata.get("HF_Token")
    except Exception:
        pass  # Not in Colab or secret access failed/timed out

    # 2) Fallback to environment variable(s)
    if not token:
        token = os.getenv("HF_TOKEN") or os.getenv("HF_Token")

    # 3) Fallback to local .ENV/.env file
    if not token:
        token = _read_token_from_env_file()

    # 4) Normalize to canonical env var name
    if token:
        os.environ["HF_TOKEN"] = token

    return token


token = resolve_hf_token()

if token:
    login(token=token, add_to_git_credential=False)
else:
    # Optional: create env var interactively for this runtime session
    entered = getpass("Enter Hugging Face token (hf_...): ").strip()
    if entered:
        os.environ["HF_TOKEN"] = entered
        login(token=entered, add_to_git_credential=False)
    else:
        notebook_login()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## (1) Experiment Configuration

In [ ]:
# ============================================================================
# SECTION 1: Enhanced Experiment Configuration (ACL 2026 SPEC)
# ============================================================================

MODELS_TO_TEST = {
    "deepseek": "deepseek-ai/deepseek-coder-6.7b-instruct",
    "llama": "meta-llama/Llama-3.1-8B-Instruct",
    "gemma": "google/gemma-2-9b-it",
    "llama-3-70b": "meta-llama/Llama-3-70B"
}

# Select active model for this run
# Research: run all four models (one per notebook execution)
#   Run 1: ACTIVE_MODEL = MODELS_TO_TEST["deepseek"]   — original paper
#   Run 2: ACTIVE_MODEL = MODELS_TO_TEST["gemma"]       — original paper
#   Run 3: ACTIVE_MODEL = MODELS_TO_TEST["llama"]       — original paper
#   Run 4: ACTIVE_MODEL = MODELS_TO_TEST["llama-3-70b"] — promised to reviewers (at least GSM8K)
# Validation testing: use deepseek (smallest model, fastest iteration)
ACTIVE_MODEL = MODELS_TO_TEST["deepseek"]

print(f"\U0001f3af Active Model: {ACTIVE_MODEL}")

class MetricType(Enum):
    NLDD_PAPER = "nldd_original"
    NLDD_ROBUST = "nldd_sym"
    P_NLDD = "p_nldd"
    ALD = "ald"

class FailureMode(Enum):
    """Section D: Interpretation Matrix failure modes."""
    REPRESENTATION_LOSS = "representation_loss"
    CONTROL_FAILURE = "control_utilization_failure"
    FAITHFUL_REASONING = "faithful_reasoning"
    PROBE_MISMATCH = "probe_mismatch_rare"

@dataclass
class ExperimentConfig:
    seed: int = 42
    models: List[str] = field(default_factory=lambda: [ACTIVE_MODEL])

    # --- Dataset Sample Sizes ---
    # Reviewer promise #9: N=300 evaluation datasets for all tasks
    dyck_total_samples: int = 20    # 300 research, 20 testing
    pronto_total_samples: int = 20  # 300 research, 20 testing
    gsm8k_total_samples: int = 20      # 300 research, 20 testing
    strategyqa_total_samples: int = 20  # 300 research, 20 testing

    # --- Dataset Configs ---
    gsm8k_split: str = "test"
    strategyqa_source: str = "metaeval/strategy-qa"
    strategyqa_eval_fraction: float = 0.4
    strategyqa_partition_seed: int = 42

    # --- Probing Samples ---
    # 1000 needed for stable probe accuracy across all layers
    probe_total_samples: int = 20   # 1000 research, 20 testing

    # --- Complexity Adjustments ---
    dyck_min_length: int = 4
    dyck_max_length: int = 12
    pronto_min_hops: int = 4
    pronto_max_hops: int = 16

    # --- NLDD Parameters (Section B) ---
    primary_metric: MetricType = MetricType.NLDD_PAPER
    faithfulness_threshold: float = 50.0
    epsilon: float = 1e-6
    min_prob_threshold: float = 1e-7

    # --- Horizon Detection Method ---
    # "steepest_drop" = argmin(diff(smoothed_NLDD)) — canonical
    # "peak" = argmax(smoothed_NLDD) — robustness check (\u00a73.5)
    horizon_method: str = "steepest_drop"

    # --- Experimental Controls ---
    require_clean_accuracy: bool = True
    validate_counterfactuals: bool = True

    # --- Visualization Control ---
    save_visualizations: bool = True

    # --- Mechanistic Modules (Stages C-G) ---
    enable_patching: bool = True
    enable_probing: bool = True
    enable_geometry: bool = True
    enable_rsa: bool = True

    # --- RSA Config (Section E) ---
    # 500 samples gives stable RDM correlation estimates
    rsa_n_samples: int = 50 # 500 research, 50 testing
    rsa_divergence_threshold: float = 0.7
    rsa_dissimilarity_metric: str = "correlation"
    rsa_compute_per_step: bool = True

    # RSA Rigor Settings
    rsa_bootstrap_enabled: bool = True
    rsa_bootstrap_iterations: int = 100 # 10000 research, 100 testing
    rsa_noise_ceiling_enabled: bool = True
    rsa_temporal_enabled: bool = True
    rsa_temporal_window_size: int = 3
    rsa_temporal_stride: int = 1

    # --- CKA Config (Section E-bis) ---
    enable_cka: bool = True
    cka_kernel: str = "linear"
    cka_debiased: bool = True
    cka_n_samples: int = 50 # 200 research, 50 testing

    # --- Interpretation Matrix Thresholds (Section D) ---
    probe_accuracy_threshold: float = 0.6
    nldd_faithfulness_threshold: float = 50.0

    # --- TAS/PCA Config (Section F/G) ---
    tas_n_components: int = 2

    # --- Patching Config (Section C) ---
    patching_stride: int = 1
    patching_node: str = "residual"

    # --- Statistical Analysis ---
    # BCa bootstrap for all metric CIs (NLDD, RSA, probes, TAS)
    n_bootstrap: int = 100 # 10000 research, 100 testing
    confidence_level: float = 0.95
    use_bca_bootstrap: bool = True

    results_dir: str = "./results_acl_2026"

    def to_dict(self) -> Dict:
        return {k: v for k, v in self.__dict__.items() if isinstance(v, (int, float, str, bool, list))}

config = ExperimentConfig()

def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(config.seed)
os.makedirs(config.results_dir, exist_ok=True)

🎯 Active Model: deepseek-ai/deepseek-coder-6.7b-instruct


## (2) Data Classes for New Analysis Types

In [4]:
# ============================================================================
# SECTION 2: Data Classes (Enhanced for ACL Compliance)
# ============================================================================

@dataclass
class RSAResult:
    """Enhanced result container with statistical metrics."""
    rsa_correlation: float
    p_value: float
    bootstrap_p_value: float = None
    noise_ceiling_lower: float = None
    noise_ceiling_upper: float = None
    rdm_clean: np.ndarray = None
    rdm_corrupt: np.ndarray = None
    n_samples: int = 0
    dissimilarity_metric: str = "correlation"
    temporal_trajectory: Dict[int, float] = None
    bootstrap_distribution: Optional[np.ndarray] = field(default=None, repr=False)

    def to_dict(self) -> Dict:
        return {k: v for k, v in self.__dict__.items() if v is not None}

@dataclass
class ProbeResult:
    """Container for probe results per layer."""
    layer_idx: int
    accuracy: float  # For classification
    r_squared: float  # For regression
    n_train: int
    n_test: int
    target_name: str
    class_distribution: Dict[str, int] = None

@dataclass
class TASResult:
    """Legacy container for TAS outputs. The active metric is clean-trajectory efficiency."""
    tas_score: float           # Active metric: ||h_T - h_0|| / Σ||h_t - h_{t-1}|| on the clean trajectory
    alignment_score: float     # Legacy compatibility field; not used by the active TAS pipeline
    path_length: float         # Cumulative path length of clean trajectory
    displacement: float        # Net displacement ||h_T - h_0||
    inefficiency: float        # 1 - tas_score
    variance: float            # Legacy compatibility field
    step_sizes: List[float]    # Legacy compatibility field

@dataclass
class TokenValidationResult:
    positive_id: int
    negative_id: int
    token_type: str
    pos_text: str
    neg_text: str
    is_single_token_pos: bool
    is_single_token_neg: bool
    validation_score: float
    warnings: List[str] = field(default_factory=list)

    @property
    def is_valid(self) -> bool:
        return (self.positive_id != self.negative_id)

@dataclass
class TaskSample:
    """Standardized sample format for all reasoning tasks."""
    input: str
    steps: List[str]
    answer: str
    depths: List[int]
    length: int
    max_depth: int
    dataset: str
    metadata: Dict[str, Any] = field(default_factory=dict)
    # ACL Strict Verification
    is_clean_correct: Optional[bool] = None

@dataclass
class CounterfactualResult:
    steps: List[str]
    corruption_type: str  # "semantic_corruption" or "paraphrase_control"
    step_index: int
    expected_answer_change: bool
    semantic_distance: float
    original_step: str
    corrupted_step: str
    original_steps: List[str] = field(default_factory=list)
    # ACL Strict Metrics
    token_count_delta: int = 0
    perplexity_ratio: float = 1.0
    is_syntactically_valid: bool = True
    is_control: bool = False

    def validate(self) -> bool:
      """Ensure counterfactual meets quality criteria."""
      if not self.original_steps:
          return False
      if len(self.steps) != len(self.original_steps):
          return False
      if not (0 <= self.step_index < len(self.steps)):
          return False
      if self.steps[self.step_index] != self.corrupted_step:
          return False

      prefix_ok = all(self.steps[i] == self.original_steps[i] for i in range(self.step_index))
      suffix_ok = all(
          self.steps[i] == self.original_steps[i]
          for i in range(self.step_index + 1, len(self.steps))
      )
      return prefix_ok and suffix_ok

@dataclass
class CounterfactualAccuracyResult:
    """Container for per-sample counterfactual accuracy results."""
    sample_idx: int
    step_index: int
    clean_correct: bool
    corrupt_correct: bool
    answer_flipped: bool
    clean_prob_correct: float
    corrupt_prob_correct: float
    prob_delta: float
    corruption_type: str
    is_control: bool

@dataclass
class CKAResult:
    layer_idx: int = -1 # Aggregate or specific layer
    mean_cka: float = 0.0
    decay_slope: float = 0.0
    cka_rsa_correlation: float = 0.0
    per_layer_cka: Dict = field(default_factory=dict)
    per_step_cka: Dict = field(default_factory=dict)
    kernel_type: str = "linear"

In [5]:
# ==============================================================================
# SECTION 2.5: CKA Core Utilities
# ==============================================================================

class CKA:
    """
    Centered Kernel Alignment (CKA) implementation with both linear and RBF kernels.
    Based on Kornblith et al. (2019).
    """

    def __init__(self, device: str = "cpu"):
        self.device = device

    @staticmethod
    def centering_matrix(n: int, device: str = "cpu") -> torch.Tensor:
        I = torch.eye(n, device=device)
        ones = torch.ones(n, n, device=device)
        return I - ones / n

    @staticmethod
    def centering(K: torch.Tensor) -> torch.Tensor:
        n = K.shape[0]
        H = CKA.centering_matrix(n, device=K.device)
        return H @ K @ H

    @staticmethod
    def linear_kernel(X: torch.Tensor) -> torch.Tensor:
        return X @ X.T

    @staticmethod
    def rbf_kernel(X: torch.Tensor, sigma: Optional[float] = None) -> torch.Tensor:
        GX = X @ X.T
        diag = torch.diag(GX)
        KX = diag.unsqueeze(0) - 2 * GX + diag.unsqueeze(1)
        if sigma is None:
            mask = KX > 0
            if mask.sum() > 0:
                median_dist = torch.median(KX[mask])
                sigma = torch.sqrt(median_dist / 2)
            else:
                sigma = torch.tensor(1.0, device=X.device)
        KX = torch.exp(-KX / (2 * sigma ** 2))
        return KX

    @staticmethod
    def hsic_biased(K: torch.Tensor, L: torch.Tensor) -> torch.Tensor:
        n = K.shape[0]
        K_centered = CKA.centering(K)
        L_centered = CKA.centering(L)
        return torch.sum(K_centered * L_centered) / ((n - 1) ** 2)

    @staticmethod
    def hsic_unbiased(K: torch.Tensor, L: torch.Tensor) -> torch.Tensor:
        n = K.shape[0]
        if n < 4:
            return CKA.hsic_biased(K, L)
        K_tilde = K.clone()
        L_tilde = L.clone()
        K_tilde.fill_diagonal_(0)
        L_tilde.fill_diagonal_(0)
        ones = torch.ones(n, device=K.device)
        term1 = torch.sum(K_tilde * L_tilde)
        sum_K = K_tilde @ ones
        sum_L = L_tilde @ ones
        term2 = torch.dot(sum_K, sum_L)
        total_K = torch.sum(K_tilde)
        total_L = torch.sum(L_tilde)
        term3 = total_K * total_L
        hsic = (term1 / (n * (n - 3)) - 2 * term2 / ((n - 2) * (n - 3) * n) + term3 / (n * (n - 1) * (n - 2) * (n - 3)))
        return hsic

    def linear_cka(self, X: torch.Tensor, Y: torch.Tensor, debiased: bool = False) -> Tuple[float, Dict]:
        if X.shape[0] != Y.shape[0]:
            raise ValueError(f"Sample count mismatch: {X.shape[0]} vs {Y.shape[0]}")
        n = X.shape[0]
        if isinstance(X, np.ndarray): X = torch.from_numpy(X).float()
        if isinstance(Y, np.ndarray): Y = torch.from_numpy(Y).float()
        X = X.to(self.device)
        Y = Y.to(self.device)
        K = self.linear_kernel(X)
        L = self.linear_kernel(Y)
        if debiased:
            hsic_xy = self.hsic_unbiased(K, L)
            hsic_xx = self.hsic_unbiased(K, K)
            hsic_yy = self.hsic_unbiased(L, L)
        else:
            hsic_xy = self.hsic_biased(K, L)
            hsic_xx = self.hsic_biased(K, K)
            hsic_yy = self.hsic_biased(L, L)
        denom = torch.sqrt(hsic_xx * hsic_yy)
        if denom < 1e-10: cka = 0.0
        else: cka = (hsic_xy / denom).item()
        return float(np.clip(cka, -1.0, 1.0)), {"hsic_xy": hsic_xy.item(), "n": n}

    def rbf_cka(self, X: torch.Tensor, Y: torch.Tensor, sigma: Optional[float] = None, debiased: bool = False) -> Tuple[float, Dict]:
        if isinstance(X, np.ndarray): X = torch.from_numpy(X).float()
        if isinstance(Y, np.ndarray): Y = torch.from_numpy(Y).float()
        X = X.to(self.device)
        Y = Y.to(self.device)
        K = self.rbf_kernel(X, sigma)
        L = self.rbf_kernel(Y, sigma)
        if debiased:
            hsic_xy = self.hsic_unbiased(K, L)
            hsic_xx = self.hsic_unbiased(K, K)
            hsic_yy = self.hsic_unbiased(L, L)
        else:
            hsic_xy = self.hsic_biased(K, L)
            hsic_xx = self.hsic_biased(K, K)
            hsic_yy = self.hsic_biased(L, L)
        denom = torch.sqrt(hsic_xx * hsic_yy)
        if denom < 1e-10: cka = 0.0
        else: cka = (hsic_xy / denom).item()
        return float(np.clip(cka, -1.0, 1.0)), {"hsic_xy": hsic_xy.item()}

In [6]:
class RSACalculator:
    """
    Implements Kriegeskorte et al. (2008) RSA with Nili et al. (2014) stats.
    Optimized for GPU acceleration using PyTorch while maintaining original API.
    """
    def __init__(self,
                 dissimilarity_metric: str = "correlation",
                 bootstrap_iterations: int = 10000,
                 enable_noise_ceiling: bool = True,
                 enable_bootstrap: bool = True,
                 random_seed: int = 42,
                 device: str = "cuda" if torch.cuda.is_available() else "cpu"):
        self.dissimilarity_metric = dissimilarity_metric
        self.bootstrap_iterations = bootstrap_iterations
        self.enable_noise_ceiling = enable_noise_ceiling
        self.enable_bootstrap = enable_bootstrap
        self.device = device
        self.rng = np.random.RandomState(random_seed)

    def _to_tensor(self, x):
        """Helper to convert to float32 tensor on device."""
        if isinstance(x, torch.Tensor):
            return x.to(self.device).float()
        return torch.tensor(x, device=self.device, dtype=torch.float32)

    def _rank_data(self, data: torch.Tensor) -> torch.Tensor:
        """Computes tie-aware average ranks on the GPU (column-wise)."""
        if data.ndim == 1:
            data = data.unsqueeze(1)

        n, n_cols = data.shape
        ranks = torch.zeros((n, n_cols), device=data.device, dtype=torch.float32)

        for col in range(n_cols):
            col_data = data[:, col]
            sorted_vals, sorted_idx = torch.sort(col_data, stable=True)
            col_ranks = torch.zeros(n, device=data.device, dtype=torch.float32)

            start = 0
            while start < n:
                end = start + 1
                while end < n and torch.isclose(sorted_vals[end], sorted_vals[start], atol=1e-12, rtol=1e-12).item():
                    end += 1
                avg_rank = (start + end - 1) / 2.0
                col_ranks[sorted_idx[start:end]] = avg_rank
                start = end

            ranks[:, col] = col_ranks

        return ranks

    def compute_rdm(self, X: Union[np.ndarray, torch.Tensor]) -> torch.Tensor:
        """Compute Representational Dissimilarity Matrix on GPU."""
        X_gpu = self._to_tensor(X)

        if self.dissimilarity_metric == "correlation":
            # Center and normalize
            X_centered = X_gpu - X_gpu.mean(dim=1, keepdim=True)
            norms = torch.linalg.norm(X_centered, dim=1, keepdim=True)
            norms = torch.where(norms < 1e-10, torch.tensor(1.0, device=self.device), norms)
            X_normed = X_centered / norms

            # Correlation matrix
            corr_matrix = torch.clamp(X_normed @ X_normed.T, -1.0, 1.0)
            rdm = 1.0 - corr_matrix

        elif self.dissimilarity_metric == "euclidean":
            # Pairwise euclidean distances
            rdm = torch.cdist(X_gpu, X_gpu, p=2.0)

        elif self.dissimilarity_metric == "cosine":
            norms = torch.linalg.norm(X_gpu, dim=1, keepdim=True)
            norms = torch.where(norms < 1e-10, torch.tensor(1.0, device=self.device), norms)
            X_normed = X_gpu / norms
            cos_sim = torch.clamp(X_normed @ X_normed.T, -1.0, 1.0)
            rdm = 1.0 - cos_sim

        else:
            raise ValueError(f"Unknown metric: {self.dissimilarity_metric}")

        rdm.fill_diagonal_(0.0)
        return (rdm + rdm.T) / 2.0

    def compare_rdms(self, rdm1: torch.Tensor, rdm2: torch.Tensor) -> Tuple[float, float]:
        """
        Spearman correlation between upper triangles on GPU.
        Returns (rho, p_value).
        """
        n = rdm1.shape[0]
        # Extract upper triangles
        triu_idx = torch.triu_indices(n, n, offset=1, device=self.device)
        v1 = rdm1[triu_idx[0], triu_idx[1]]
        v2 = rdm2[triu_idx[0], triu_idx[1]]

        # Spearman = Pearson on ranks
        v1_rank = self._rank_data(v1.unsqueeze(1)).squeeze()
        v2_rank = self._rank_data(v2.unsqueeze(1)).squeeze()

        v1_cent = v1_rank - v1_rank.mean()
        v2_cent = v2_rank - v2_rank.mean()

        covariance = (v1_cent * v2_cent).sum()
        std_v1 = (v1_cent ** 2).sum().sqrt()
        std_v2 = (v2_cent ** 2).sum().sqrt()

        rho = covariance / (std_v1 * std_v2 + 1e-8)

        # Approximate p-value for Spearman (t-distribution)
        # For rigorous stats, we rely on the bootstrap method below
        p_val = 0.0

        return rho.item(), p_val

    def bootstrap_significance(self, rdm1: torch.Tensor, rdm2: torch.Tensor, observed_rho: float) -> Tuple[float, np.ndarray]:
        """
        Permutation test for non-parametric significance on GPU (Batched).
        Shuffles the rows/cols of rdm2 (Mantel test).
        """
        n = rdm1.shape[0]
        n_perms = self.bootstrap_iterations

        # Pre-calculate rdm1 upper triangle info
        triu_idx = torch.triu_indices(n, n, offset=1, device=self.device)
        v1 = rdm1[triu_idx[0], triu_idx[1]]
        v1_rank = self._rank_data(v1.unsqueeze(1)).squeeze()
        v1_cent = v1_rank - v1_rank.mean()
        v1_std = (v1_cent ** 2).sum().sqrt()

        count_better = 0
        null_dist = []
        chunk_size = 500 # Process perms in chunks to save VRAM

        for _ in range(0, n_perms, chunk_size):
            curr_chunk = min(chunk_size, n_perms - _)

            # Generate batch of random permutations
            perms = torch.stack([torch.randperm(n, device=self.device) for _ in range(curr_chunk)])

            # Expand rdm2 for gathering
            rdm2_exp = rdm2.unsqueeze(0).expand(curr_chunk, -1, -1)

            # Permute rows
            row_idx = perms.unsqueeze(2).expand(-1, -1, n)
            rdm2_perm_rows = torch.gather(rdm2_exp, 1, row_idx)

            # Permute cols
            col_idx = perms.unsqueeze(1).expand(-1, n, -1)
            rdm2_perm = torch.gather(rdm2_perm_rows, 2, col_idx)

            # Extract upper triangles for the batch
            v2_batch = rdm2_perm[:, triu_idx[0], triu_idx[1]]

            # Compute correlations for batch
            v2_ranks = v2_batch.argsort(dim=1).argsort(dim=1).float()
            v2_means = v2_ranks.mean(dim=1, keepdim=True)
            v2_cents = v2_ranks - v2_means

            covs = (v1_cent.unsqueeze(0) * v2_cents).sum(dim=1)
            v2_stds = (v2_cents ** 2).sum(dim=1).sqrt()

            batch_rhos = covs / (v1_std * v2_stds + 1e-8)

            count_better += (batch_rhos.abs() >= abs(observed_rho)).sum().item()
            null_dist.append(batch_rhos.cpu().numpy())

        p_value = count_better / n_perms
        return float(p_value), np.concatenate(null_dist)

    def compute_noise_ceiling(self, X: Union[np.ndarray, torch.Tensor]) -> Tuple[float, float]:
        """Estimate measurement reliability on GPU."""
        X_gpu = self._to_tensor(X)
        n = X_gpu.shape[0]
        if n < 4: return 1.0, 1.0

        rdm_full = self.compute_rdm(X_gpu)
        correlations = []

        for _ in range(100):
            # Split-half indices
            idx = torch.randint(0, n, (n,), device=self.device)
            # Resample RDM
            rdm_resample = rdm_full.index_select(0, idx).index_select(1, idx)

            rho, _ = self.compare_rdms(rdm_full, rdm_resample)
            correlations.append(rho)

        lower_bound = np.mean(correlations)
        upper_bound = (2 * lower_bound) / (1 + lower_bound) if lower_bound < 1.0 else 1.0
        return float(np.clip(lower_bound, 0.0, 1.0)), float(np.clip(upper_bound, 0.0, 1.0))

    def compute_rsa(self, X_clean: Union[np.ndarray, torch.Tensor], X_corrupt: Union[np.ndarray, torch.Tensor], return_detailed: bool = True) -> Dict:
        """Orchestrator for RSA + Stats."""
        # compute_rdm handles conversion to Tensor/GPU
        rdm_clean = self.compute_rdm(X_clean)
        rdm_corrupt = self.compute_rdm(X_corrupt)

        rsa_corr, p_value = self.compare_rdms(rdm_clean, rdm_corrupt)

        bootstrap_p, null_dist = None, None
        if self.enable_bootstrap:
            bootstrap_p, null_dist = self.bootstrap_significance(rdm_clean, rdm_corrupt, rsa_corr)
            # Use bootstrap p-value as the primary one if enabled
            p_value = bootstrap_p

        nc_lower, nc_upper = None, None
        if self.enable_noise_ceiling:
            nc_lower, nc_upper = self.compute_noise_ceiling(X_clean)

        if return_detailed:
            return RSAResult(
                rsa_correlation=rsa_corr,
                p_value=p_value,
                bootstrap_p_value=bootstrap_p,
                noise_ceiling_lower=nc_lower,
                noise_ceiling_upper=nc_upper,
                rdm_clean=rdm_clean.cpu().numpy(),
                rdm_corrupt=rdm_corrupt.cpu().numpy(),
                n_samples=rdm_clean.shape[0],
                bootstrap_distribution=null_dist
            ).to_dict()

        return {"rsa_correlation": rsa_corr, "p_value": p_value}

In [7]:
class TemporalRSA:
    """Sliding window analysis for CoT trajectories."""
    def __init__(self, rsa_calculator, window_size=3, stride=1):
        self.rsa_calc = rsa_calculator
        self.window_size = window_size
        self.stride = stride

    def compute_temporal_rsa(self, clean_traj: List[np.ndarray], corrupt_traj: List[np.ndarray]) -> Dict[int, float]:
        T = min(len(clean_traj), len(corrupt_traj))
        temporal_rsa = {}

        for start in range(0, T - self.window_size + 1, self.stride):
            end = start + self.window_size
            # Stack (Time * Batch) to capture temporal geometry
            X_clean = np.vstack(clean_traj[start:end])
            X_corrupt = np.vstack(corrupt_traj[start:end])

            res = self.rsa_calc.compute_rsa(X_clean, X_corrupt, return_detailed=False)
            temporal_rsa[start] = res['rsa_correlation']
        return temporal_rsa

## (3) Model Management

In [ ]:
# ============================================================================
# Model Manager and Token Validator (unchanged from original)
# ============================================================================

class ModelManager:
    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.current_model = None
        self.current_tokenizer = None
        self.current_model_name = None
        self._registered_tasks = []

    def register_task(self, task):
        if task not in self._registered_tasks:
            self._registered_tasks.append(task)

    def _invalidate_task_caches(self):
        for task in self._registered_tasks:
            if hasattr(task, 'clear_token_cache'):
                task.clear_token_cache()

    def unload_model(self):
        if self.current_model is not None:
            print(f"Unloading: {self.current_model_name}")
            self._invalidate_task_caches()
            del self.current_model
            del self.current_tokenizer
            self.current_model = None
            self.current_tokenizer = None
            self.current_model_name = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    def load_model(self, model_name: str) -> Tuple[Any, Any]:
        self.unload_model()
        print(f"\n{'='*60}\nLoading: {model_name}\n{'='*60}")

        model_kwargs = {
            "torch_dtype": torch.bfloat16,
            "device_map": "auto",
            "trust_remote_code": True,
            "attn_implementation": "sdpa"
        }

        if "gemma" in model_name.lower():
            model_kwargs["attn_implementation"] = "eager"

        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
        model.eval()

        self.current_model = model
        self.current_tokenizer = tokenizer
        self.current_model_name = model_name

        print(f"✓ Model loaded: {model.num_parameters():,} parameters, {model.config.num_hidden_layers} layers")
        return model, tokenizer

    def get_model_short_name(self) -> str:
        if not self.current_model_name:
            return "unknown"
        return self.current_model_name.split("/")[-1].replace("-", "_")


class TokenValidator:
    @staticmethod
    def find_best_categorical_tokens(
        tokenizer,
        positive_candidates: List[str],
        negative_candidates: List[str],
        task_name: str,
        require_single_token: bool = True
    ) -> TokenValidationResult:
        best_result = None
        all_warnings = []

        for pos_text in positive_candidates:
            for neg_text in negative_candidates:
                pos_ids = tokenizer.encode(pos_text, add_special_tokens=False)
                neg_ids = tokenizer.encode(neg_text, add_special_tokens=False)

                if not pos_ids or not neg_ids:
                    continue

                is_single_pos = len(pos_ids) == 1
                is_single_neg = len(neg_ids) == 1

                if require_single_token and (not is_single_pos or not is_single_neg):
                    continue

                pos_id, neg_id = pos_ids[0], neg_ids[0]
                if pos_id == neg_id:
                    continue

                score = 0
                if is_single_pos: score += 20
                if is_single_neg: score += 20

                result = TokenValidationResult(
                    positive_id=pos_id, negative_id=neg_id,
                    token_type="categorical",
                    pos_text=pos_text, neg_text=neg_text,
                    is_single_token_pos=is_single_pos,
                    is_single_token_neg=is_single_neg,
                    validation_score=score, warnings=[]
                )

                if best_result is None or score > best_result.validation_score:
                    best_result = result

        if best_result is None and require_single_token:
            return TokenValidator.find_best_categorical_tokens(
                tokenizer, positive_candidates, negative_candidates,
                task_name, require_single_token=False
            )

        if best_result is None:
            raise ValueError(f"[{task_name}] Could not find valid token pair.")

        return best_result


# Initialize and load model
model_manager = ModelManager(config)
model, tokenizer = model_manager.load_model(config.models[0])


Loading: deepseek-ai/deepseek-coder-6.7b-instruct


`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

## (4) MechanisticToolkit

In [ ]:
# ===================================================================================
# MECHANISTIC TOOLKIT - SECTIONS D, E, F, G (v4.0 - Integrated Linear Probing)
# ===================================================================================

from tqdm.auto import tqdm  # Explicitly use compact progress bar
import torch
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple, Optional
from collections import defaultdict
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, r2_score
from sklearn.decomposition import PCA
from scipy.stats import spearmanr
import warnings
import difflib

# Suppress sklearn warnings
warnings.filterwarnings("ignore", category=UserWarning)

class MechanisticToolkit:
    """
    Implements Sections C (Patching), D (Probing), E (RSA), F (PCA), G (TAS).
    Revised to safely handle BFloat16 by forcing Float32 conversion for Analysis.
    """

    def __init__(self, model, tokenizer, config):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
        self.layers = self._get_model_layers()
        self.n_layers = len(self.layers)
        self.device = model.device

        # Probing regularization parameters (Section D)
        self.l2_C = 100.0
        self.l2_alpha = 1.0

    def _get_model_layers(self):
        if hasattr(self.model, 'model') and hasattr(self.model.model, 'layers'):
            return list(self.model.model.layers)
        elif hasattr(self.model, 'transformer') and hasattr(self.model.transformer, 'h'):
            return list(self.model.transformer.h)
        # Fallback for Gemma/Other architectures
        for name, module in self.model.named_modules():
            if name.endswith(".layers") and isinstance(module, torch.nn.ModuleList):
                return list(module)
        return []

    def _find_step_token_positions(self, full_text: str, steps: List[str]) -> List[int]:
      step_indices = []
      current_idx = 0
      encoding = self.tokenizer(full_text, return_offsets_mapping=True)
      offset_mapping = encoding.offset_mapping

      for step in steps:
          step_clean = step.strip()
          start_char = full_text.find(step_clean, current_idx)
          if start_char == -1:
              # fallback: closest match using difflib
              matches = difflib.get_close_matches(step_clean, [full_text[current_idx:]], n=1, cutoff=0.6)
              if matches:
                  start_char = full_text.find(matches[0], current_idx)
              else:
                  continue

          end_char = start_char + len(step_clean)
          current_idx = end_char

          target_token_idx = -1
          for i, (tok_start, tok_end) in enumerate(offset_mapping):
              if tok_start < end_char <= tok_end:
                  target_token_idx = i
                  break
              if tok_end == end_char:
                  target_token_idx = i
                  break

          if target_token_idx != -1:
              step_indices.append(target_token_idx)

      return step_indices


    def _get_activations(self, text: str, layer_idx: int = -1, return_all_layers: bool = False) -> torch.Tensor:
        """
        Get hidden states. Returns float32 CPU tensors.
        """
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs, output_hidden_states=True)

        hidden_states = outputs.hidden_states

        if return_all_layers:
            return torch.stack(hidden_states).squeeze(1).detach().float().cpu()

        if layer_idx == -1:
            layer_idx = len(hidden_states) - 1
        return hidden_states[layer_idx].squeeze(0).detach().float().cpu()

    # =========================================================================
    # SECTION D: Linear Probe Module
    # =========================================================================
    def train_linear_probes(self, task, dataset: List, task_name: str) -> Dict:
        if not self.config.enable_probing:
            return {"enabled": False}

        print(f"   [Probing] Training probes for {task_name}...")
        if task_name == "dyck":
            return self._train_dyck_probes(task, dataset)
        elif task_name == "pronto_qa":
            return self._train_pronto_qa_probes(task, dataset)
        elif task_name == "gsm8k":
            return self._train_gsm8k_probes(task, dataset)
        elif task_name == "strategyqa":
            return self._train_strategyqa_probes(task, dataset)
        else:
            raise ValueError(f"Unknown task: {task_name}")

    def _get_operation_label(self, step: str) -> str:
      step_lower = step.lower()
      if '+' in step_lower or 'add' in step_lower or 'sum' in step_lower:
          return 'add'
      if '-' in step_lower or 'sub' in step_lower or 'minus' in step_lower:
          return 'sub'
      if '*' in step_lower or '×' in step_lower or 'mul' in step_lower or 'times' in step_lower:
          return 'mul'
      return 'other'

    def _train_arithmetic_operation_probes(self, task, dataset: List, task_name: str, desc: str) -> Dict:
      print(f"   [Probing] Extracting {desc} operation labels (full CoT)...")
      op_data = defaultdict(lambda: {'X': [], 'y': [], 'groups': []})
      samples_to_use = dataset[:min(self.config.probe_total_samples, len(dataset))]

      for sample_idx, sample in enumerate(tqdm(samples_to_use, desc=f"{desc} Probing (Full CoT)", leave=False)):
          try:
              full_text = task.build_prompt(sample.input, sample.steps)
              all_layer_acts = self._get_activations(full_text, return_all_layers=True).numpy()
              step_token_positions = self._find_step_token_positions(full_text, sample.steps)

              n_steps = min(len(sample.steps), len(step_token_positions))
              for step_idx in range(n_steps):
                  token_pos = step_token_positions[step_idx]
                  step = sample.steps[step_idx]
                  op = self._get_operation_label(step)

                  if token_pos >= all_layer_acts.shape[1]:
                      continue

                  for layer_idx in range(self.n_layers):
                      h = all_layer_acts[layer_idx + 1, token_pos, :]
                      op_data[layer_idx]['X'].append(h)
                      op_data[layer_idx]['y'].append(op)
                      op_data[layer_idx]['groups'].append(sample_idx)

          except Exception:
              continue

      return self._train_classification_probes(op_data, target_name="operation_type", task_name=task_name)

    def _train_dyck_probes(self, task, dataset: List) -> Dict:
      print("   [Probing] Extracting Dyck stack depth labels (full CoT)...")
      layer_data = defaultdict(lambda: {'X': [], 'y': [], 'groups': []})
      target_probe_samples = self.config.probe_total_samples
      samples_to_use = dataset[:min(target_probe_samples, len(dataset))]

      for sample_idx, sample in enumerate(tqdm(samples_to_use, desc="Extracting Dyck activations", leave=False)):
          try:
              if not hasattr(sample, 'depths') or not sample.depths:
                  continue

              full_text = task.build_prompt(sample.input, sample.steps)
              all_layer_acts = self._get_activations(full_text, return_all_layers=True).numpy()
              step_token_positions = self._find_step_token_positions(full_text, sample.steps)

              n_steps = min(len(sample.depths), len(step_token_positions))
              for step_idx in range(n_steps):
                  token_pos = step_token_positions[step_idx]
                  depth = sample.depths[step_idx]
                  if token_pos >= all_layer_acts.shape[1]:
                      continue
                  for layer_idx in range(self.n_layers):
                      h = all_layer_acts[layer_idx + 1, token_pos, :]
                      layer_data[layer_idx]['X'].append(h)
                      layer_data[layer_idx]['y'].append(depth)
                      layer_data[layer_idx]['groups'].append(sample_idx)

          except Exception:
              continue

      return self._train_classification_probes(layer_data, target_name="stack_depth", task_name="dyck")



    def _train_pronto_qa_probes(self, task, dataset: List) -> Dict:
      print("   [Probing] Extracting ProntoQA sample-truth labels at step tokens...")
      layer_data_all_steps = defaultdict(lambda: {'X': [], 'y': [], 'groups': []})
      layer_data_last_step = defaultdict(lambda: {'X': [], 'y': [], 'groups': []})
      stats = {'processed': 0, 'skipped': 0}

      def normalize_truth(ans):
          s = str(ans).strip().lower()
          return 'True' if s == 'true' else 'False'

      samples_to_use = dataset[:min(self.config.probe_total_samples, len(dataset))]

      for sample_idx, sample in enumerate(tqdm(samples_to_use, desc="ProntoQA Probing (Full CoT)", leave=False)):
          try:
              if not hasattr(sample, 'steps') or len(sample.steps) < 2:
                  stats['skipped'] += 1
                  continue
              if not hasattr(sample, 'answer'):
                  stats['skipped'] += 1
                  continue

              full_text = task.build_prompt(sample.input, sample.steps)
              all_layer_acts = self._get_activations(full_text, return_all_layers=True).numpy()
              step_positions = self._find_step_token_positions(full_text, sample.steps)
              if len(step_positions) < 1:
                  stats['skipped'] += 1
                  continue

              stats['processed'] += 1
              truth_label = normalize_truth(sample.answer)

              n_steps = min(len(sample.steps), len(step_positions))
              for step_idx in range(n_steps):
                  token_pos = step_positions[step_idx]
                  if token_pos >= all_layer_acts.shape[1]:
                      continue
                  for layer_idx in range(self.n_layers):
                      h = all_layer_acts[layer_idx + 1, token_pos, :]
                      layer_data_all_steps[layer_idx]['X'].append(h)
                      layer_data_all_steps[layer_idx]['y'].append(truth_label)
                      layer_data_all_steps[layer_idx]['groups'].append(sample_idx)

              # Save last step separately
              last_token_idx = step_positions[-1]
              for layer_idx in range(self.n_layers):
                  h = all_layer_acts[layer_idx + 1, last_token_idx, :]
                  layer_data_last_step[layer_idx]['X'].append(h)
                  layer_data_last_step[layer_idx]['y'].append(truth_label)
                  layer_data_last_step[layer_idx]['groups'].append(sample_idx)

          except Exception:
              continue

    # Train probes
      res_all = self._train_classification_probes(layer_data_all_steps, "sample_truth_at_step_tokens", "pronto_qa")
      res_last = self._train_classification_probes(layer_data_last_step, "pre_final_truth", "pronto_qa")

      res_all["last_step_results"] = res_last
      return res_all



    def _train_gsm8k_probes(self, task, dataset: List) -> Dict:
      return self._train_arithmetic_operation_probes(task, dataset, task_name="gsm8k", desc="GSM8K")


    def _train_strategyqa_probes(self, task, dataset: List) -> Dict:
      print("   [Probing] Extracting StrategyQA sample-truth labels at step tokens...")
      layer_data_all_steps = defaultdict(lambda: {'X': [], 'y': [], 'groups': []})
      layer_data_last_step = defaultdict(lambda: {'X': [], 'y': [], 'groups': []})

      def normalize_truth(ans):
          s = str(ans).strip().lower()
          return 'Yes' if s in ['yes', 'true'] else 'No'

      samples_to_use = dataset[:min(self.config.probe_total_samples, len(dataset))]

      for sample_idx, sample in enumerate(tqdm(samples_to_use, desc="StrategyQA Probing (Full CoT)", leave=False)):
          try:
              if not hasattr(sample, 'steps') or len(sample.steps) < 2:
                  continue
              if not hasattr(sample, 'answer'):
                  continue

              full_text = task.build_prompt(sample.input, sample.steps)
              all_layer_acts = self._get_activations(full_text, return_all_layers=True).numpy()
              step_positions = self._find_step_token_positions(full_text, sample.steps)
              if len(step_positions) < 1:
                  continue

              truth_label = normalize_truth(sample.answer)

              n_steps = min(len(sample.steps), len(step_positions))
              for step_idx in range(n_steps):
                  token_pos = step_positions[step_idx]
                  if token_pos >= all_layer_acts.shape[1]:
                      continue
                  for layer_idx in range(self.n_layers):
                      h = all_layer_acts[layer_idx + 1, token_pos, :]
                      layer_data_all_steps[layer_idx]['X'].append(h)
                      layer_data_all_steps[layer_idx]['y'].append(truth_label)
                      layer_data_all_steps[layer_idx]['groups'].append(sample_idx)

              last_token_idx = step_positions[-1]
              for layer_idx in range(self.n_layers):
                  h = all_layer_acts[layer_idx + 1, last_token_idx, :]
                  layer_data_last_step[layer_idx]['X'].append(h)
                  layer_data_last_step[layer_idx]['y'].append(truth_label)
                  layer_data_last_step[layer_idx]['groups'].append(sample_idx)

          except Exception:
              continue

      res_all = self._train_classification_probes(layer_data_all_steps, "sample_truth_at_step_tokens", "strategyqa")
      res_last = self._train_classification_probes(layer_data_last_step, "pre_final_truth", "strategyqa")

      res_all["last_step_results"] = res_last
      return res_all


    def _train_classification_probes(self, layer_data, target_name, task_name) -> Dict:
        results = {}
        accuracies = []
        skip_reasons = defaultdict(list)

        for layer_idx in tqdm(range(self.n_layers), desc=f"Training {target_name} probes", leave=False):
            if layer_idx not in layer_data or len(layer_data[layer_idx]['X']) < 10:
                skip_reasons['insufficient_rows'].append(layer_idx)
                continue

            X = np.array(layer_data[layer_idx]['X'])
            y = np.array(layer_data[layer_idx]['y'])
            groups = np.array(layer_data[layer_idx].get('groups', np.arange(len(y))))

            from sklearn.model_selection import GroupShuffleSplit
            try:
                if len(np.unique(groups)) < 2:
                    skip_reasons['insufficient_groups'].append(layer_idx)
                    continue
                splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
                train_idx, test_idx = next(splitter.split(X, y, groups=groups))
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]
                train_groups = np.unique(groups[train_idx])
                test_groups = np.unique(groups[test_idx])
                if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
                    skip_reasons['degenerate_class_split'].append(layer_idx)
                    continue

                clf = LogisticRegression(C=self.l2_C, max_iter=1000, solver='liblinear')
                clf.fit(X_train, y_train)
                y_pred = clf.predict(X_test)
                acc = accuracy_score(y_test, y_pred)

                accuracies.append(acc)
                results[layer_idx] = {
                    "accuracy": acc,
                    "n_train": len(X_train),
                    "n_test": len(X_test),
                    "n_train_groups": len(train_groups),
                    "n_test_groups": len(test_groups)
                }
            except Exception:
                skip_reasons['fit_failed'].append(layer_idx)
                continue

        mean_acc = np.mean(accuracies) if accuracies else 0.0
        results["mean_accuracy"] = mean_acc
        results["target_name"] = target_name
        results["skip_counts"] = {k: len(v) for k, v in skip_reasons.items()}
        results["skip_layers"] = {k: v for k, v in skip_reasons.items() if v}
        return results

    # =========================================================================
    # SECTION E: RSA (Representational Similarity Analysis) - CORRECTED
    # =========================================================================
    def compute_rsa_batch(
        self,
        task,
        dataset: List,
        counterfactuals: Dict[int, List] = None,
        n_samples: int = 50,
        dissimilarity_metric: str = "correlation"
    ) -> Dict:
        """
        Compute canonical RSA between clean and corrupt conditions.
        UPDATED: Uses the shared supplied counterfactual contract when available.
        """
        if not self.config.enable_rsa:
            return {"mean_rsa": 0.0, "mean_similarity": 0.0, "enabled": False}

        rsa_calc = RSACalculator(
            dissimilarity_metric=getattr(self.config, 'rsa_dissimilarity_metric', 'correlation'),
            bootstrap_iterations=getattr(self.config, 'rsa_bootstrap_iterations', 10000),
            enable_noise_ceiling=getattr(self.config, 'rsa_noise_ceiling_enabled', True),
            enable_bootstrap=getattr(self.config, 'rsa_bootstrap_enabled', True)
        )

        samples = dataset[:min(n_samples, len(dataset))]
        counterfactual_source = "supplied_map" if counterfactuals is not None else "generated_fallback"

        clean_acts_per_layer = {l: [] for l in range(self.n_layers)}
        corrupt_acts_per_layer = {l: [] for l in range(self.n_layers)}
        step_clean_midlayer = defaultdict(list)
        step_corrupt_midlayer = defaultdict(list)

        enable_temporal = getattr(self.config, 'rsa_temporal_enabled', False)
        mid_layer = self.n_layers // 2
        clean_traj_storage = []
        corrupt_traj_storage = []

        processed_pairs = []
        n_requested_pairs = 0
        n_used_pairs = 0
        n_skipped_pairs = 0
        n_sample_failures = 0
        n_layer_failures = 0
        temporal_rsa_failed = False

        print(f"   [RSA] Processing {len(samples)} samples (Temporal={enable_temporal}, source={counterfactual_source})...")

        for idx, sample in enumerate(tqdm(samples, desc="RSA: Extracting activations", leave=False)):
            if len(sample.steps) < 2:
                continue

            try:
                clean_prompt = task.build_prompt(sample.input, sample.steps)
                clean_acts = self._get_activations(clean_prompt, return_all_layers=True)
                if isinstance(clean_acts, torch.Tensor):
                    clean_acts = clean_acts.cpu().numpy()
            except Exception:
                n_sample_failures += 1
                continue

            if counterfactuals is not None:
                cf_rows = get_analysis_counterfactual_rows(
                    sample,
                    counterfactuals.get(idx) if counterfactuals is not None else None,
                    config=self.config,
                    include_controls=False
                )
            else:
                fallback_idx = min(len(sample.steps) // 2, len(sample.steps) - 1)
                cf_rows = get_analysis_counterfactual_rows(
                    sample,
                    [task.generate_counterfactual(sample, fallback_idx)],
                    config=self.config,
                    include_controls=False
                )

            n_requested_pairs += len(cf_rows)
            if not cf_rows:
                continue

            for cf_row in cf_rows:
                try:
                    step_idx = cf_row["step_index"]
                    corrupt_prompt = task.build_prompt(sample.input, cf_row["replacement_steps"])
                    corrupt_acts = self._get_activations(corrupt_prompt, return_all_layers=True)
                    if isinstance(corrupt_acts, torch.Tensor):
                        corrupt_acts = corrupt_acts.cpu().numpy()

                    for layer_idx in range(self.n_layers):
                        h_clean = clean_acts[layer_idx + 1, -1, :]
                        h_corrupt = corrupt_acts[layer_idx + 1, -1, :]
                        clean_acts_per_layer[layer_idx].append(h_clean)
                        corrupt_acts_per_layer[layer_idx].append(h_corrupt)

                    step_clean_midlayer[step_idx + 1].append(clean_acts[mid_layer + 1, -1, :])
                    step_corrupt_midlayer[step_idx + 1].append(corrupt_acts[mid_layer + 1, -1, :])

                    if enable_temporal:
                        t_clean = clean_acts[mid_layer + 1]
                        t_corrupt = corrupt_acts[mid_layer + 1]
                        if len(t_clean) > 2 and len(t_corrupt) > 2:
                            clean_traj_storage.append(t_clean)
                            corrupt_traj_storage.append(t_corrupt)

                    processed_pairs.append((idx, step_idx + 1))
                    n_used_pairs += 1
                except Exception:
                    n_skipped_pairs += 1
                    continue

        if len(processed_pairs) < 5:
            print(f"   [RSA] Warning: Only {len(processed_pairs)} valid clean/corrupt pairs.")
            return {
                "mean_rsa": 0.0,
                "error": "insufficient_samples",
                "failure_stats": {
                    "counterfactual_source": counterfactual_source,
                    "n_requested_samples": int(len(samples)),
                    "n_processed_samples": int(len({sample_idx for sample_idx, _ in processed_pairs})),
                    "n_processed_pairs": int(len(processed_pairs)),
                    "n_requested_pairs": int(n_requested_pairs),
                    "n_used_pairs": int(n_used_pairs),
                    "n_skipped_pairs": int(n_skipped_pairs),
                    "n_sample_failures": int(n_sample_failures),
                    "n_layer_failures": int(n_layer_failures),
                    "temporal_rsa_failed": bool(temporal_rsa_failed),
                },
            }

        per_layer_rsa = {}
        per_layer_pvalue = {}
        per_layer_noise_ceiling = {}

        for layer_idx in range(self.n_layers):
            X_clean = np.array(clean_acts_per_layer[layer_idx])
            X_corrupt = np.array(corrupt_acts_per_layer[layer_idx])

            if X_clean.shape[0] < 3:
                n_layer_failures += 1
                per_layer_rsa[layer_idx] = np.nan
                per_layer_pvalue[layer_idx] = np.nan
                per_layer_noise_ceiling[layer_idx] = (np.nan, np.nan)
                continue

            try:
                result = rsa_calc.compute_rsa(X_clean, X_corrupt)
                per_layer_rsa[layer_idx] = result["rsa_correlation"]
                per_layer_pvalue[layer_idx] = result["p_value"]
                per_layer_noise_ceiling[layer_idx] = (
                    result.get("noise_ceiling_lower", np.nan),
                    result.get("noise_ceiling_upper", np.nan)
                )
            except Exception:
                n_layer_failures += 1
                per_layer_rsa[layer_idx] = np.nan
                per_layer_pvalue[layer_idx] = np.nan
                per_layer_noise_ceiling[layer_idx] = (np.nan, np.nan)

        temporal_results = {}
        if enable_temporal and len(clean_traj_storage) > 2:
            try:
                print("   [RSA] Calculating Temporal Dynamics...")
                min_len = min(min(len(t) for t in clean_traj_storage), min(len(t) for t in corrupt_traj_storage))
                clean_traj_batch = []
                corrupt_traj_batch = []
                for t in range(min_len):
                    batch_c = np.stack([traj[t] for traj in clean_traj_storage])
                    batch_corr = np.stack([traj[t] for traj in corrupt_traj_storage])
                    clean_traj_batch.append(batch_c)
                    corrupt_traj_batch.append(batch_corr)
                temporal_calc = TemporalRSA(
                    rsa_calculator=rsa_calc,
                    window_size=getattr(self.config, 'rsa_temporal_window_size', 3),
                    stride=getattr(self.config, 'rsa_temporal_stride', 1)
                )
                temporal_results = temporal_calc.compute_temporal_rsa(clean_traj_batch, corrupt_traj_batch)
            except Exception as e:
                temporal_rsa_failed = True
                print(f"   [RSA] Temporal RSA failed: {e}")

        valid_rsa = [v for v in per_layer_rsa.values() if np.isfinite(v)]
        mean_rsa = float(np.mean(valid_rsa)) if valid_rsa else 0.0

        rsa_by_step_mean = {}
        rsa_by_step_sem = {}
        rsa_by_step_n = {}
        rsa_by_step_pairs = {}
        step_rsa_calc = RSACalculator(
            dissimilarity_metric=getattr(self.config, 'rsa_dissimilarity_metric', 'correlation'),
            bootstrap_iterations=getattr(self.config, 'rsa_bootstrap_iterations', 10000),
            enable_noise_ceiling=False,
            enable_bootstrap=False
        )
        step_bootstrap_iters = max(10, min(getattr(self.config, 'rsa_bootstrap_iterations', 100), 100))
        for step_k in sorted(step_clean_midlayer.keys()):
            X_clean = np.array(step_clean_midlayer[step_k])
            X_corrupt = np.array(step_corrupt_midlayer[step_k])
            if X_clean.shape[0] > 0 and X_corrupt.shape[0] > 0:
                rsa_by_step_pairs[int(step_k)] = {
                    "clean": X_clean.copy(),
                    "corrupt": X_corrupt.copy(),
                }
            if X_clean.shape[0] < 3:
                continue
            try:
                step_result = step_rsa_calc.compute_rsa(X_clean, X_corrupt, return_detailed=False)
                step_val = float(step_result['rsa_correlation'])
                if not np.isfinite(step_val):
                    continue
                rsa_by_step_mean[int(step_k)] = step_val
                rsa_by_step_n[int(step_k)] = int(X_clean.shape[0])
                boot_vals = []
                for _ in range(step_bootstrap_iters):
                    sample_ids = step_rsa_calc.rng.choice(X_clean.shape[0], size=X_clean.shape[0], replace=True)
                    boot_result = step_rsa_calc.compute_rsa(X_clean[sample_ids], X_corrupt[sample_ids], return_detailed=False)
                    boot_val = float(boot_result['rsa_correlation'])
                    if np.isfinite(boot_val):
                        boot_vals.append(boot_val)
                rsa_by_step_sem[int(step_k)] = float(np.std(boot_vals, ddof=1)) if len(boot_vals) > 1 else np.nan
            except Exception:
                continue

        failure_stats = {
            "counterfactual_source": counterfactual_source,
            "n_requested_samples": int(len(samples)),
            "n_processed_samples": int(len({sample_idx for sample_idx, _ in processed_pairs})),
            "n_processed_pairs": int(len(processed_pairs)),
            "n_requested_pairs": int(n_requested_pairs),
            "n_used_pairs": int(n_used_pairs),
            "n_skipped_pairs": int(n_skipped_pairs),
            "n_sample_failures": int(n_sample_failures),
            "n_layer_failures": int(n_layer_failures),
            "temporal_rsa_failed": bool(temporal_rsa_failed),
        }
        if n_sample_failures or n_layer_failures or temporal_rsa_failed or n_skipped_pairs:
            print(
                f"   [RSA Audit] source={counterfactual_source}, sample_failures={n_sample_failures}, "
                f"layer_failures={n_layer_failures}, skipped_pairs={n_skipped_pairs}, temporal_failed={temporal_rsa_failed}"
            )

        return {
            "mean_rsa": mean_rsa,
            "per_layer_rsa": per_layer_rsa,
            "per_layer_pvalue": per_layer_pvalue,
            "rsa_by_step": rsa_by_step_mean,
            "rsa_by_step_sem": rsa_by_step_sem,
            "rsa_by_step_n": rsa_by_step_n,
            "rsa_by_step_pairs": rsa_by_step_pairs,
            "temporal_rsa": temporal_results,
            "noise_ceiling": per_layer_noise_ceiling,
            "bootstrap_p": per_layer_pvalue,
            "failure_stats": failure_stats,
        }

    # =========================================================================
    # SECTION E-bis: CKA
    # =========================================================================
    def compute_cka_batch(self, task, dataset: List, counterfactuals: Dict[int, List] = None, n_samples: int = None) -> Dict:
        """
        Compute Centered Kernel Alignment (CKA) between clean and corrupted trajectories.
        Uses the shared supplied counterfactual map when available.
        """
        if not getattr(self.config, 'enable_cka', True):
            return {"mean_cka": 0.0, "error": "CKA disabled"}

        n_samples = n_samples or getattr(self.config, 'cka_n_samples', 200)
        n_samples = min(len(dataset), n_samples)

        device = "cuda" if torch.cuda.is_available() else "cpu"
        cka_calc = CKA(device=device)

        per_layer_cka = {i: [] for i in range(self.n_layers)}
        all_cka_values = []
        kernel_type = getattr(self.config, 'cka_kernel', 'linear')
        use_debiased = getattr(self.config, 'cka_debiased', True)
        counterfactual_source = "supplied_map" if counterfactuals is not None else "generated_fallback"
        n_requested_pairs = 0
        n_used_pairs = 0
        n_skipped_pairs = 0
        n_sample_failures = 0
        step_k_counts = defaultdict(int)

        print(f"   > Calculating CKA ({kernel_type}, debiased={use_debiased}) on {n_samples} samples (source={counterfactual_source})...")

        for i, sample in enumerate(dataset[:n_samples]):
            if len(sample.steps) < 2:
                continue

            try:
                clean_text = task.build_prompt(sample.input, sample.steps)
                clean_all = self._get_activations(clean_text, return_all_layers=True)
                if isinstance(clean_all, torch.Tensor):
                    clean_all = clean_all.cpu().numpy()
            except Exception:
                n_sample_failures += 1
                continue

            if counterfactuals is not None:
                cf_rows = get_analysis_counterfactual_rows(
                    sample,
                    counterfactuals.get(i) if counterfactuals is not None else None,
                    config=self.config,
                    include_controls=False
                )
            else:
                fallback_idx = min(len(sample.steps) // 2, len(sample.steps) - 1)
                cf_rows = get_analysis_counterfactual_rows(
                    sample,
                    [task.generate_counterfactual(sample, fallback_idx)],
                    config=self.config,
                    include_controls=False
                )

            n_requested_pairs += len(cf_rows)
            if not cf_rows:
                continue

            clean_pos = self._find_step_token_positions(clean_text, sample.steps)

            for cf_row in cf_rows:
                try:
                    corrupt_text = task.build_prompt(sample.input, cf_row["replacement_steps"])
                    corrupt_all = self._get_activations(corrupt_text, return_all_layers=True)
                    if isinstance(corrupt_all, torch.Tensor):
                        corrupt_all = corrupt_all.cpu().numpy()

                    corrupt_pos = self._find_step_token_positions(corrupt_text, cf_row["replacement_steps"])
                    min_len = min(len(clean_pos), len(corrupt_pos))
                    if min_len < 2:
                        n_skipped_pairs += 1
                        continue

                    aligned_clean_pos = clean_pos[:min_len]
                    aligned_corrupt_pos = corrupt_pos[:min_len]

                    pair_used = False
                    for layer_idx in range(self.n_layers):
                        X = clean_all[layer_idx + 1][aligned_clean_pos]
                        Y = corrupt_all[layer_idx + 1][aligned_corrupt_pos]
                        if kernel_type == 'rbf':
                            val, _ = cka_calc.rbf_cka(X, Y, debiased=use_debiased)
                        else:
                            val, _ = cka_calc.linear_cka(X, Y, debiased=use_debiased)
                        if not np.isnan(val):
                            per_layer_cka[layer_idx].append(val)
                            all_cka_values.append(val)
                            pair_used = True

                    if pair_used:
                        n_used_pairs += 1
                        step_k_counts[int(cf_row["step_k"])] += 1
                    else:
                        n_skipped_pairs += 1
                except Exception:
                    n_skipped_pairs += 1
                    continue

        mean_cka = np.mean(all_cka_values) if all_cka_values else 0.0
        layer_means = [np.mean(per_layer_cka[i]) for i in range(self.n_layers) if per_layer_cka[i]]
        if len(layer_means) > 1:
            slope, _, _, _, _ = stats.linregress(range(len(layer_means)), layer_means)
        else:
            slope = 0.0

        return {
            "mean_cka": float(mean_cka),
            "decay_slope": float(slope),
            "per_layer_cka": per_layer_cka,
            "kernel_type": kernel_type,
            "counterfactual_source": counterfactual_source,
            "n_requested_pairs": int(n_requested_pairs),
            "n_used_pairs": int(n_used_pairs),
            "n_skipped_pairs": int(n_skipped_pairs),
            "n_sample_failures": int(n_sample_failures),
            "step_k_counts": {int(k): int(v) for k, v in step_k_counts.items()},
        }

    def compute_cka_layer_similarity_matrix(self, task, dataset: List, n_samples: int = 50) -> np.ndarray:
        """Compute CKA similarity matrix between all layer pairs."""
        device = "cuda" if torch.cuda.is_available() else "cpu"
        cka_calc = CKA(device=device)
        layer_acts = {i: [] for i in range(self.n_layers)}

        for sample in dataset[:n_samples]:
            try:
                prompt = task.build_prompt(sample.input, sample.steps)
                all_layers = self._get_activations(prompt, return_all_layers=True).numpy()
                for layer_idx in range(self.n_layers):
                    layer_acts[layer_idx].append(all_layers[layer_idx + 1, -1, :])
            except Exception: continue

        layer_matrices = {i: np.stack(acts) for i, acts in layer_acts.items() if len(acts) >= 10}
        n_layers = self.n_layers
        cka_matrix = np.eye(n_layers)

        for i in range(n_layers):
            for j in range(i + 1, n_layers):
                if i in layer_matrices and j in layer_matrices:
                    try:
                        val, _ = cka_calc.linear_cka(layer_matrices[i], layer_matrices[j], debiased=True)
                        cka_matrix[i, j] = val
                        cka_matrix[j, i] = val
                    except: pass
        return cka_matrix

    # =========================================================================
    # SECTION F/G: Geometry (TAS + PCA)
    # =========================================================================
    def analyze_geometry(self, dataset, task) -> Dict:
        """
        Computes TAS as a clean-trajectory efficiency score.
        TAS = Displacement / Path_Length
        """
        tas_scores = []

        for sample in tqdm(dataset[:50], desc="Geometry Analysis", leave=False):
            txt = task.build_prompt(sample.input, sample.steps)
            mid_layer = self.n_layers // 2
            acts = self._get_activations(txt, layer_idx=mid_layer).numpy() # [T, D]

            diffs = acts[1:] - acts[:-1]
            dists = np.linalg.norm(diffs, axis=1)
            path_len = np.sum(dists)
            displacement = np.linalg.norm(acts[-1] - acts[0])

            if path_len > 1e-6:
                tas = displacement / path_len
                tas_scores.append(tas)

        return {
            "tas_mean": np.mean(tas_scores) if tas_scores else 0.0,
            "tas_definition": "clean_trajectory_efficiency"
        }

    # =========================================================================
    # SECTION H: Reasoning Horizon (Figure 1 Data)
    # =========================================================================
    def compute_horizon_metrics(self, task, dataset, counterfactuals, config, nldd_df=None) -> Dict:
      """
      Compute measured metrics per chain depth (k) for Figure 1.
      NLDD drives horizon detection; RSA and TAS remain complementary diagnostics.
      """
      from scipy.ndimage import gaussian_filter1d

      print("\n[Horizon Module] Computing metrics per chain depth (k)...")

      def _empty_result(status: str) -> Dict:
          return {
              "status": status,
              "steps": [],
              "nldd_mean": [],
              "nldd_sem": [],
              "nldd_err": [],
              "nldd_smoothed": [],
              "rsa_mean": [],
              "rsa_sem": [],
              "rsa_err": [],
              "tas_mean": [],
              "tas_sem": [],
              "tas_err": [],
              "n_samples": 0,
              "n_per_step": [],
              "horizon_k_primary": np.nan,
              "horizon_k_alt": np.nan,
              "horizon_primary_method": "unavailable",
              "horizon_alt_method": "unavailable",
              "interval_method": "raw_samples_primary",
              "curve_error_method": "summary_t_from_sem",
              "tas_definition": "clean_prefix_efficiency",
              "raw_samples": {"nldd": {}, "rsa": {}, "tas": {}},
          }

      def _detect_horizon_from_curve(steps: List[int], values: List[float], method: str) -> float:
          if not steps or not values:
              return np.nan
          arr = np.asarray(values, dtype=float)
          if arr.size == 0 or np.any(~np.isfinite(arr)):
              return np.nan
          smoothed_arr = gaussian_filter1d(arr, sigma=1.0)
          max_step = max(steps)
          valid_indices = [i for i, step in enumerate(steps) if step > 1 and step < max_step]
          if not valid_indices:
              return float(steps[-1]) if steps else np.nan
          if method == "peak":
              best_idx = max(valid_indices, key=lambda i: smoothed_arr[i])
          else:
              diffs = np.diff(smoothed_arr, prepend=smoothed_arr[0])
              best_idx = min(valid_indices, key=lambda i: diffs[i])
          return float(steps[best_idx])

      # ===== RSA BY STEP POSITION =====
      rsa_results = self.compute_rsa_batch(
          task, dataset, counterfactuals,
          n_samples=min(config.rsa_n_samples, len(dataset))
      )
      rsa_by_step = rsa_results.get("rsa_by_step", {})
      rsa_by_step_sem = rsa_results.get("rsa_by_step_sem", {})
      rsa_by_step_pairs = rsa_results.get("rsa_by_step_pairs", {})

      # ===== NLDD FROM DATAFRAME =====
      if nldd_df is None or nldd_df.empty:
          print("   [Horizon] Warning: No NLDD data available.")
          return _empty_result("missing_nldd")

      corrupt_df = nldd_df[nldd_df["type"] == "corruption"].copy()
      if "is_valid" in corrupt_df.columns:
          corrupt_df = corrupt_df[corrupt_df["is_valid"] == True]
      if corrupt_df.empty:
          print("   [Horizon] Warning: No valid corruption rows after filtering.")
          return _empty_result("no_valid_corruptions")

      if "step_k" not in corrupt_df.columns:
          corrupt_df["step_k"] = corrupt_df["step_index"] + 1
      nldd_raw_samples = {
          int(step_k): group["nldd"].astype(float).tolist()
          for step_k, group in corrupt_df.groupby("step_k")
      }
      stats = corrupt_df.groupby("step_k")["nldd"].agg(["mean", "sem", "count"]).sort_index()
      if stats.empty:
          print("   [Horizon] Warning: No grouped NLDD statistics available.")
          return _empty_result("no_grouped_nldd")

      steps = stats.index.astype(int).tolist()
      nldd_mean = stats["mean"].astype(float).tolist()
      nldd_sem = stats["sem"].fillna(0.0).astype(float).tolist()
      n_per_step = stats["count"].astype(int).tolist()
      nldd_smoothed = gaussian_filter1d(np.asarray(nldd_mean, dtype=float), sigma=1.0).tolist()

      print("   [Horizon] Computing TAS (clean-prefix efficiency) by reasoning depth...")
      tas_by_step = self._compute_tas_by_step(task, dataset, steps)

      rsa_mean = []
      rsa_sem = []
      tas_mean = []
      tas_sem = []
      tas_raw_samples = {}

      for step in steps:
          rsa_mean.append(rsa_by_step.get(step, np.nan))
          rsa_sem.append(rsa_by_step_sem.get(step, np.nan))
          tas_entry = tas_by_step.get(step, {})
          tas_mean.append(tas_entry.get("mean", np.nan))
          tas_sem.append(tas_entry.get("sem", np.nan))
          tas_raw_samples[int(step)] = list(tas_entry.get("raw_samples", []))

      primary_method = config.horizon_method if hasattr(config, 'horizon_method') else "steepest_drop"
      alt_method = "peak" if primary_method == "steepest_drop" else "steepest_drop"
      horizon_k_primary = _detect_horizon_from_curve(steps, nldd_mean, primary_method)
      horizon_k_alt = _detect_horizon_from_curve(steps, nldd_mean, alt_method)

      return {
          "status": "ok",
          "steps": steps,
          "nldd_mean": nldd_mean,
          "nldd_sem": nldd_sem,
          "nldd_err": nldd_sem,
          "nldd_smoothed": nldd_smoothed,
          "rsa_mean": rsa_mean,
          "rsa_sem": rsa_sem,
          "rsa_err": rsa_sem,
          "tas_mean": tas_mean,
          "tas_sem": tas_sem,
          "tas_err": tas_sem,
          "n_samples": int(len(corrupt_df)),
          "n_per_step": n_per_step,
          "horizon_k_primary": horizon_k_primary,
          "horizon_k_alt": horizon_k_alt,
          "horizon_primary_method": primary_method,
          "horizon_alt_method": alt_method,
          "interval_method": "raw_samples_primary",
          "curve_error_method": "summary_t_from_sem",
          "tas_definition": "clean_prefix_efficiency",
          "raw_samples": {
              "nldd": nldd_raw_samples,
              "rsa": rsa_by_step_pairs,
              "tas": tas_raw_samples,
          },
      }


    def _compute_tas_by_step(self, task, dataset, steps) -> Dict:
      """
      Compute clean-prefix TAS efficiency values for each reasoning depth.
      Returns dict: {step_k: {"mean": float, "sem": float, "raw_samples": List[float]}}
      """
      tas_by_step = {k: [] for k in steps}
      mid_layer = self.n_layers // 2

      # Sample subset for efficiency (same size as RSA)
      n_samples = min(50, len(dataset))
      samples = dataset[:n_samples]

      print(f"      Computing TAS on {len(samples)} samples across {len(steps)} depths...")

      for sample in tqdm(samples, desc="TAS by depth", leave=False):
          try:
              for k in steps:
                  # Need at least k reasoning steps to evaluate depth k.
                  if len(sample.steps) < k:
                      continue

                  # Build prompt with exactly k reasoning steps
                  truncated_steps = sample.steps[:k]
                  prompt = task.build_prompt(sample.input, truncated_steps)

                  # Get trajectory at middle layer
                  acts = self._get_activations(prompt, layer_idx=mid_layer)
                  if isinstance(acts, torch.Tensor):
                      acts = acts.cpu().numpy()

                  # Compute TAS: displacement / path_length
                  if len(acts) < 2:
                      continue

                  displacement = np.linalg.norm(acts[-1] - acts[0])
                  diffs = acts[1:] - acts[:-1]
                  path_length = np.sum(np.linalg.norm(diffs, axis=1))

                  if path_length > 1e-9:
                      tas = displacement / path_length
                      tas_by_step[k].append(tas)

          except Exception as e:
              continue

      # Aggregate results
      tas_results = {}
      for k in steps:
          if len(tas_by_step[k]) > 0:
              tas_results[k] = {
                  "mean": float(np.mean(tas_by_step[k])),
                  "sem": float(np.std(tas_by_step[k], ddof=1) / np.sqrt(len(tas_by_step[k]))) if len(tas_by_step[k]) > 1 else 0.0,
                  "raw_samples": [float(x) for x in tas_by_step[k]],
              }
          else:
              tas_results[k] = {"mean": np.nan, "sem": np.nan, "raw_samples": []}

      return tas_results

In [ ]:
# ============================================================================
# DATASET GENERATORS (FIXED: Balanced Classes)
# ============================================================================

def generate_dyck_data(n_samples: int, min_length: int = 4, max_length: int = 8) -> List[TaskSample]:
    """Generate synthetic Dyck-n samples with space-separated brackets and restored logic."""
    samples = []
    openers = "([{<"
    closers = ")]}>"
    pairs = dict(zip(openers, closers))

    for _ in range(n_samples):
        # 1. RESTORED: Initialize the variables that were missing
        length = random.randint(min_length, max_length)
        stack = []
        prefix = []
        depths = []

        # 2. RESTORED: Generate the actual bracket sequence
        for i in range(length):
            # Decide to open a new bracket or close one
            if len(stack) == 0 or (random.random() < 0.6 and i < length - 1):
                char = random.choice(openers)
                prefix.append(char)
                stack.append(char)
            else:
                opener = stack.pop()
                prefix.append(pairs[opener])
            depths.append(len(stack))

        # Ensure the sequence is incomplete so the model has to predict the NEXT closer
        while len(stack) == 0:
            char = random.choice(openers)
            prefix.append(char)
            stack.append(char)
            depths.append(len(stack))

        # The target answer is the bracket that closes the top of the current stack
        answer = pairs[stack[-1]]

        # 3. Build steps with the SPACE FIX for tokenization
        # Replace the step building loop in your generate_dyck_data:
        steps = []
        current_stack = []
        for char in prefix:
            if char in openers:
                current_stack.append(char)
                action = f"Push '{char}'"
            else:
                if current_stack:
                    current_stack.pop()
                action = f"Pop to match '{char}'"

            # Use spaces for tokenization, but use a more standard format
            stack_repr = " ".join(current_stack) if current_stack else "empty"
            steps.append(f"{action}. Current stack: [ {stack_repr} ]")

        samples.append(TaskSample(
            input="".join(prefix),
            steps=steps,
            answer=answer,
            depths=depths,
            length=len(steps),
            max_depth=max(depths) if depths else 0,
            dataset="dyck"
        ))
    return samples

# 2. SPLIT FUNCTION NOTE
# The canonical `generate_split_data(...)` definition appears below.
# This cell intentionally does not redefine it, which prevents stale
# notebook execution from changing split semantics mid-run.

# ============================================================================
# FIXED PRONTOQA PIPELINE (Force Long Chains)
# ============================================================================

# 1. Redefine Generator with HIGH DEFAULTS and EXPANDED VOCAB
def generate_pronto_data_v2(n_samples: int, min_hops: int = 5, max_hops: int = 20) -> List[TaskSample]:
    """
    Generates ProntoQA samples with FORCED deeper reasoning chains.
    Min Hops is hardcoded to 6 to ensure chain length > 5.
    """
    samples = []
    # Massive vocabulary to support long chains without running out of words
    cats = [
        "wumpus", "gorpus", "rompus", "jompus", "zumpus", "tumpus", "yumpus", "impus",
        "dumpus", "lumpus", "numpus", "pumpus", "grumpus", "bumpus", "slumpus", "clumpus",
        "shumpus", "tampus", "zimpus", "kumpus", "fumpus", "dimpus", "humpus", "bampus"
    ]

    for _ in range(n_samples):
        hops = random.randint(min_hops, max_hops)
        entity = "Sam"

        # Ensure we don't exceed available categories
        actual_len = min(len(cats), hops + 2)
        chain = random.sample(cats, actual_len)

        facts = [f"{entity} is a {chain[0]}"]
        rules = [f"All {chain[i]} are {chain[i+1]}" for i in range(len(chain)-2)]

        label = random.choice([True, False])

        if label:
            target_cls = chain[len(rules)]
            answer = "True"
        else:
            available = [c for c in cats if c not in chain[:len(rules)+1]]
            if not available: available = [chain[-1]]
            target_cls = random.choice(available)
            answer = "False"

        query = f"Is {entity} a {target_cls}?"

        steps = []
        for i, rule in enumerate(rules):
            steps.append(f"Since {entity} is {chain[i]} and {rule}, {entity} is {chain[i+1]}")

        last_inferred = chain[len(rules)]
        if label:
            steps.append(f"Conclusion: {entity} is {last_inferred}, so {answer}")
        else:
            steps.append(f"Conclusion: {entity} is {last_inferred}, which is not {target_cls}, so {answer}")

        inp = f"Facts: {'. '.join(facts)}. Rules: {'. '.join(rules)}. {query}"

        samples.append(TaskSample(
            input=inp, steps=steps, answer=answer,
            depths=list(range(len(steps))), length=len(steps), max_depth=len(steps), dataset="pronto_qa"
        ))
    return samples


def _normalize_reasoning_steps(raw_text) -> List[str]:
    if isinstance(raw_text, (list, tuple)):
        normalized_steps = []
        for item in raw_text:
            normalized_steps.extend(_normalize_reasoning_steps(item))
        return normalized_steps

    text = str(raw_text).replace("\r\n", "\n").replace("\r", "\n")
    return [line.strip() for line in text.split("\n") if line.strip()]


def _normalize_strategyqa_decomposition_steps(raw_steps) -> List[str]:
    normalized_steps = []
    for step in raw_steps or []:
        flat_step = " ".join(_normalize_reasoning_steps(step)).strip()
        if flat_step:
            normalized_steps.append(flat_step)
    return normalized_steps


def _format_strategyqa_input(question: str, facts: List[str]) -> str:
    evidence_lines = "\n".join(f"- {fact}" for fact in facts)
    return f"{question}\nSupporting Facts:\n{evidence_lines}"


_STRATEGYQA_PARTITION_CACHE: Dict[Tuple[str, float, int], Dict[str, List[Dict[str, Any]]]] = {}


def _build_strategyqa_partitions(config: ExperimentConfig) -> Dict[str, List[Dict[str, Any]]]:
    cache_key = (
        config.strategyqa_source,
        float(config.strategyqa_eval_fraction),
        int(config.strategyqa_partition_seed),
    )
    if cache_key in _STRATEGYQA_PARTITION_CACHE:
        return _STRATEGYQA_PARTITION_CACHE[cache_key]

    try:
        dataset = load_dataset(config.strategyqa_source, split="train")
    except Exception as e:
        raise RuntimeError(
            f"Failed to load {config.strategyqa_source!r} split='train'. Install dataset dependencies and verify access before running research analyses."
        ) from e

    grouped_rows = {"Yes": [], "No": []}
    for row in dataset:
        qid = str(row.get("qid", "")).strip()
        question = str(row.get("question", "")).strip()
        facts = [str(fact).strip() for fact in row.get("facts", []) if str(fact).strip()]
        decomposition = _normalize_strategyqa_decomposition_steps(row.get("decomposition", []))
        if not qid or not question or not facts or not decomposition:
            continue

        answer = "Yes" if bool(row.get("answer")) else "No"
        grouped_rows[answer].append({
            "qid": qid,
            "term": str(row.get("term", "")).strip(),
            "description": str(row.get("description", "")).strip(),
            "question": question,
            "answer": answer,
            "raw_boolean_answer": bool(row.get("answer")),
            "facts": facts,
            "decomposition": decomposition,
        })

    partitions = {"eval": [], "probe": []}
    for label, rows in grouped_rows.items():
        ordered_rows = sorted(rows, key=lambda item: item["qid"])
        label_seed = int(config.strategyqa_partition_seed) + (0 if label == "Yes" else 1)
        rng = random.Random(label_seed)
        rng.shuffle(ordered_rows)

        if len(ordered_rows) <= 1:
            eval_count = len(ordered_rows)
        else:
            eval_count = int(round(len(ordered_rows) * float(config.strategyqa_eval_fraction)))
            eval_count = max(1, min(len(ordered_rows) - 1, eval_count))

        partitions["eval"].extend(ordered_rows[:eval_count])
        partitions["probe"].extend(ordered_rows[eval_count:])

    for partition_name, partition_rows in partitions.items():
        rng = random.Random(int(config.strategyqa_partition_seed) + (17 if partition_name == "eval" else 29))
        shuffled_rows = partition_rows.copy()
        rng.shuffle(shuffled_rows)
        partitions[partition_name] = shuffled_rows

    _STRATEGYQA_PARTITION_CACHE[cache_key] = partitions
    return partitions


# --- CANONICAL SPLIT FUNCTION DEFINITION ---
def generate_split_data(eval_loader_func, probe_loader_func, task_obj, config, n_faithfulness, n_probe):
    """
    Generates disjoint datasets for probing vs. faithfulness testing.
    Probe samples must remain disjoint from the NLDD / counterfactual split.
    """
    eval_pool_size = max((n_faithfulness + n_probe) * 2, n_faithfulness * 2)
    raw_eval_pool = eval_loader_func(eval_pool_size)

    print(f"\n[{task_obj.task_name.upper()}] Processing Faithfulness Split ({n_faithfulness} samples)...")
    faith_data = create_acl_compliant_dataset(
        task=task_obj,
        model=model,
        tokenizer=tokenizer,
        config=config,
        n_samples=n_faithfulness,
        k_counterfactuals=5,
        enable_controls=True,
        raw_data=raw_eval_pool,
    )

    faith_samples = faith_data['faithful_samples']
    used_inputs = {s.input for s in faith_samples}
    probe_samples = []
    raw_probe_pool = probe_loader_func(max((n_probe + n_faithfulness) * 2, n_probe * 2))

    for s in raw_probe_pool:
        if len(probe_samples) >= n_probe:
            break
        if s.input in used_inputs:
            continue
        if not task_obj.validate_sample(s):
            continue
        probe_samples.append(s)

    print(f"   ✓ Faithfulness Set: {len(faith_samples)} samples (Verified)")
    print(f"   ✓ Probe Train Set:  {len(probe_samples)} samples (Disjoint)")

    return faith_data, probe_samples


# 2. Regenerate Dataset (Override previous data)
print("\n[ProntoQA] Regenerating with FORCED long chains...")

# NOTE: This block assumes 'pronto_task', 'mechanistic_tool', 'run_enhanced_analysis', and 'plot_reasoning_horizon_acl' are defined globally.
if 'pronto_task' in globals() and 'run_enhanced_analysis' in globals() and 'plot_reasoning_horizon_acl' in globals():
    pronto_data_bundle, pronto_probe_dataset = generate_split_data(
        lambda n: generate_pronto_data_v2(n, config.pronto_min_hops, config.pronto_max_hops),
        lambda n: generate_pronto_data_v2(n, config.pronto_min_hops, config.pronto_max_hops),
        pronto_task,
        config,
        config.pronto_total_samples,
        n_probe=config.probe_total_samples
    )
    pronto_dataset = pronto_data_bundle['faithful_samples']
    pronto_counterfactuals = pronto_data_bundle['counterfactuals']

    # 3. VERIFY LENGTH (Sanity Check)
    avg_len = sum(len(s.steps) for s in pronto_dataset) / len(pronto_dataset)
    print(f"DEBUG: Average Chain Length is now {avg_len:.1f} steps (Should be > 6)")

    # 4. Re-Run Analysis Loop
    results_pronto = run_enhanced_analysis(
        task=pronto_task,
        dataset=pronto_dataset,
        probe_dataset=pronto_probe_dataset,
        counterfactuals=pronto_counterfactuals,
        task_name="pronto_qa",
        toolkit=mechanistic_tool,
        config=config
    )

    # 5. Re-Plot
    print("Generating Updated Graph...")
    plot_reasoning_horizon_acl("./results_acl_2026/horizon_pronto_qa.csv", "ProntoQA", config)
else:
    print("Skipping regeneration block: 'pronto_task' or analysis functions not yet defined.")


def _generate_synthetic_gsm8k_data(n_samples: int) -> List[TaskSample]:
    """Explicit smoke-test helper for synthetic arithmetic chains."""
    samples = []
    ops = ['+', '-', '*']
    for _ in range(n_samples):
        n_steps = random.randint(3, 8)
        numbers = [random.randint(10, 99) for _ in range(n_steps + 1)]
        steps = []
        expr = str(numbers[0])
        for i in range(n_steps):
            op = random.choice(ops)
            expr = f"({expr} {op} {numbers[i+1]})"
            steps.append(f"I calculate step {i+1}: {expr}")

        final_ans = eval(expr)
        samples.append(TaskSample(
            input=f"Compute the result of a {n_steps}-step arithmetic chain.",
            steps=steps,
            answer=str(final_ans),
            depths=list(range(len(steps))),
            length=len(steps),
            max_depth=len(steps),
            dataset="gsm8k_synthetic"
        ))
    return samples


def load_gsm8k_data(n_samples: int, split="test", allow_synthetic_fallback: bool = False) -> List[TaskSample]:
    """Load real GSM8K data and fail closed unless synthetic fallback is explicitly requested."""
    samples = []
    try:
        dataset = load_dataset("gsm8k", "main", split=split)
    except Exception as e:
        if allow_synthetic_fallback:
            print(f"Warning: GSM8K load failed ({e}). Using explicit synthetic smoke-test fallback.")
            return _generate_synthetic_gsm8k_data(n_samples)
        raise RuntimeError(f"Failed to load gsm8k/main split={split!r}. Install dataset dependencies and verify access before running research analyses.") from e

    for row in dataset:
        if len(samples) >= n_samples:
            break
        q = row['question']
        ans_part = row['answer'].split('####')
        if len(ans_part) < 2:
            continue

        steps = _normalize_reasoning_steps(ans_part[0])
        final = ans_part[1].strip()

        samples.append(TaskSample(
            input=q, steps=steps, answer=final,
            depths=list(range(len(steps))), length=len(steps), max_depth=len(steps), dataset="gsm8k"
        ))

    return samples


def load_strategyqa_data(n_samples: int, partition="eval", config: Optional[ExperimentConfig] = None) -> List[TaskSample]:
    """Load StrategyQA with deterministic stratified eval/probe partitions and decomposed reasoning steps."""
    if config is None:
        config = globals().get("config")
    if config is None:
        raise ValueError("StrategyQA loading requires an ExperimentConfig instance.")

    partition = str(partition).strip().lower()
    if partition not in {"eval", "probe"}:
        raise ValueError(f"Unsupported StrategyQA partition: {partition!r}")

    partition_rows = _build_strategyqa_partitions(config)[partition]
    samples = []

    for row in partition_rows:
        if len(samples) >= n_samples:
            break

        facts = [fact for fact in row["facts"] if str(fact).strip()]
        decomposition_steps = [
            f"Step {idx}: {step}"
            for idx, step in enumerate(row["decomposition"], start=1)
            if str(step).strip()
        ]
        if not facts or not decomposition_steps:
            continue

        answer = row["answer"]
        steps = decomposition_steps + [f"Therefore, the answer is {answer}."]
        input_text = _format_strategyqa_input(row["question"], facts)

        samples.append(TaskSample(
            input=input_text,
            steps=steps,
            answer=answer,
            depths=list(range(len(steps))),
            length=len(steps),
            max_depth=len(steps),
            dataset="strategyqa",
            metadata={
                "qid": row["qid"],
                "term": row["term"],
                "description": row["description"],
                "raw_boolean_answer": row["raw_boolean_answer"],
                "raw_facts": list(facts),
                "raw_decomposition": list(row["decomposition"]),
                "source_dataset": config.strategyqa_source,
                "source_split": "train",
                "partition": partition,
            },
        ))

    return samples


# Generate datasets immediately
# We check for generate_pronto_data definition to avoid errors if only v2 is defined here
if 'generate_pronto_data' not in globals():
    generate_pronto_data = generate_pronto_data_v2

dyck_dataset = generate_dyck_data(
    config.dyck_total_samples,
    min_length=config.dyck_min_length,
    max_length=config.dyck_max_length
)

# Note: Ensure generate_pronto_data points to v2 and passes hops
pronto_dataset = generate_pronto_data_v2(
    config.pronto_total_samples,
    min_hops=config.pronto_min_hops,
    max_hops=config.pronto_max_hops
)
gsm8k_dataset = load_gsm8k_data(config.gsm8k_total_samples)
strategyqa_dataset = load_strategyqa_data(config.strategyqa_total_samples, partition="eval", config=config)
print(
    f"Datasets Loaded: Dyck({len(dyck_dataset)}), "
    f"Pronto({len(pronto_dataset)}), GSM8K({len(gsm8k_dataset)}), StrategyQA({len(strategyqa_dataset)})"
)


## (5) Task Implementations (Dyck, ProntoQA, GSM8K)

In [ ]:
# ============================================================================
# ACL-COMPLIANT DATASET CONSTRUCTION UTILITIES & TASK DEFINITIONS (ROBUST)
# ============================================================================
from __future__ import annotations
import re
import random
import torch
import numpy as np
from tqdm.auto import tqdm
from abc import ABC, abstractmethod
from typing import List, Dict, Tuple, Optional, Any

# ============================================================================
# 1. Robust Perplexity Calculation
# ============================================================================
def calculate_perplexity(model, tokenizer, text: str) -> float:
    """
    Calculates perplexity with robust tensor handling and fallback.
    """
    try:
        encodings = tokenizer(text, return_tensors="pt")
        input_ids = encodings.input_ids.to(model.device)

        # Ensure pad token
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id or 0

        attention_mask = (input_ids != tokenizer.pad_token_id).long()
        labels = input_ids.clone()
        labels[input_ids == tokenizer.pad_token_id] = -100

        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)

        if outputs.loss is None or torch.isnan(outputs.loss):
            return 1e3
        return torch.exp(outputs.loss).item()
    except Exception:
        return 1e3

# ============================================================================
# 2. ACL-Compliant Dataset Creation (FIXED: Explicit Config)
# ============================================================================
def create_acl_compliant_dataset(
    task,
    model,
    tokenizer,
    config: ExperimentConfig,
    n_samples: int,
    k_counterfactuals: int = 5,
    enable_controls: bool = True,
    seed: int = 42,
    raw_data: Optional[List[TaskSample]] = None,
) -> Dict[str, Any]:
    """
    Generates ACL-compliant datasets with faithful, counterfactual, and control samples.
    Ensures reproducibility via seed and respects ExperimentConfig complexity.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    faithful_samples = []
    counterfactuals_map = {}
    controls_map = {}

    if raw_data is None:
        if task.task_name == "dyck":
            raw_data = generate_dyck_data(
                n_samples * 2,
                min_length=config.dyck_min_length,
                max_length=config.dyck_max_length,
            )
        elif task.task_name == "pronto_qa":
            raw_data = generate_pronto_data_v2(
                n_samples * 2,
                min_hops=config.pronto_min_hops,
                max_hops=config.pronto_max_hops,
            )
        elif task.task_name == "gsm8k":
            raw_data = load_gsm8k_data(n_samples * 2, split=config.gsm8k_split)
        elif task.task_name == "strategyqa":
            raw_data = load_strategyqa_data(n_samples * 2, partition="eval", config=config)
        else:
            raw_data = []

    print(f"[{task.task_name.upper()}] Filtering {len(raw_data)} raw samples for strict validity...")
    pbar = tqdm(total=n_samples, desc=f"Building {task.task_name.upper()}")

    for sample in raw_data:
        if len(faithful_samples) >= n_samples:
            break
        if not task.validate_sample(sample):
            continue

        clean_prompt = task.build_prompt(sample.input, sample.steps)
        clean_ppl = calculate_perplexity(model, tokenizer, clean_prompt)

        sample_cfs = []
        sample_controls = []
        available_indices = list(range(len(sample.steps)))
        random.shuffle(available_indices)

        for step_idx in available_indices:
            if len(sample_cfs) >= k_counterfactuals:
                break

            cf = task.generate_counterfactual(sample, step_idx)
            if cf.token_count_delta > 2:
                continue
            if hasattr(cf, "validate") and not cf.validate():
                continue

            corrupt_prompt = task.build_prompt(sample.input, cf.steps)
            corrupt_ppl = calculate_perplexity(model, tokenizer, corrupt_prompt)
            cf.perplexity_ratio = corrupt_ppl / (clean_ppl + 1e-6)
            sample_cfs.append(cf)

            if enable_controls and len(sample_controls) < k_counterfactuals:
                ctrl = task.generate_paraphrase(sample, step_idx)
                if ctrl.token_count_delta > 2:
                    continue
                if hasattr(ctrl, "validate") and not ctrl.validate():
                    continue
                ctrl_prompt = task.build_prompt(sample.input, ctrl.steps)
                ctrl.perplexity_ratio = calculate_perplexity(model, tokenizer, ctrl_prompt) / (clean_ppl + 1e-6)
                sample_controls.append(ctrl)

        if sample_cfs:
            idx = len(faithful_samples)
            faithful_samples.append(sample)
            counterfactuals_map[idx] = sample_cfs
            controls_map[idx] = sample_controls
            pbar.update(1)

    pbar.close()

    return {
        "faithful_samples": faithful_samples,
        "counterfactuals": counterfactuals_map,
        "controls": controls_map,
    }

# ============================================================================
# 3. Base Reasoning Task Class
# ============================================================================
class ReasoningTask(ABC):
    ANSWER_CUE = "\nFinal Answer:"

    def __init__(self, tokenizer, model_name: str):
        self.tokenizer = tokenizer
        self.model_name = model_name
        self.task_name = "generic"
        self.is_chat_model = any(x in model_name.lower() for x in ["instruct", "chat", "it"])
        self._token_cache: Dict[str, Any] = {}

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

    def clear_token_cache(self):
        self._token_cache = {}

    def build_prompt(self, input_text: str, current_steps: Optional[List[str]] = None) -> str:
      system_prompt = self._get_system_prompt()
      prompt = f"{system_prompt}\n\nQuestion: {input_text}\n"

      if current_steps:
          cot_text = "\n".join(s.strip() for s in current_steps if s and str(s).strip())
          prompt += cot_text + "\n"

      prompt = prompt.rstrip() + self.ANSWER_CUE
      return prompt.rstrip() + " "

    def _calculate_token_delta(self, original_step: str, corrupted_step: str) -> int:
        t1 = self.tokenizer.encode(original_step, add_special_tokens=False)
        t2 = self.tokenizer.encode(corrupted_step, add_special_tokens=False)
        return abs(len(t1) - len(t2))

    def get_generation_max_new_tokens(self) -> int:
        return 10

    def get_counterfactual_perplexity_threshold(self) -> float:
        return 3.5

    def parse_model_answer(self, generated_text: str, sample: "TaskSample") -> Any:
        return str(generated_text).strip() if generated_text is not None else None

    def is_prediction_correct(self, parsed_prediction: Any, sample: "TaskSample") -> bool:
        if parsed_prediction is None:
            return False
        pos_candidates, _ = self.get_answer_candidates(sample)
        pred_text = str(parsed_prediction).strip().lower()
        return any(str(cand).strip().lower() in pred_text for cand in pos_candidates if str(cand).strip())

    def get_target_answer_string(self, sample: "TaskSample") -> str:
        pos_candidates, _ = self.get_answer_candidates(sample)
        for cand in pos_candidates:
            cand_str = str(cand).strip()
            if cand_str:
                return cand_str
        return str(sample.answer).strip()

    @abstractmethod
    def _get_system_prompt(self) -> str: ...

    @abstractmethod
    def get_answer_candidates(self, sample: "TaskSample") -> Tuple[List[str], List[str]]: ...

    def get_nldd_scoring_mode(self) -> str:
        return "token_margin"

    def build_nldd_scoring_spec(self, sample: "TaskSample", predicted_answer: Optional[str] = None) -> Dict[str, Any]:
        pos_candidates, neg_candidates = self.get_answer_candidates(sample)
        return {
            "mode": self.get_nldd_scoring_mode(),
            "positive_candidates": list(dict.fromkeys(pos_candidates)),
            "negative_candidates": list(dict.fromkeys(neg_candidates)),
        }

    @abstractmethod
    def generate_counterfactual(self, sample: "TaskSample", step_index: int) -> "CounterfactualResult": ...

    @abstractmethod
    def generate_paraphrase(self, sample: "TaskSample", step_index: int) -> "CounterfactualResult": ...

    @abstractmethod
    def validate_sample(self, sample: "TaskSample") -> bool: ...


class DyckTask(ReasoningTask):
    def __init__(self, tokenizer, model_name: str):
        super().__init__(tokenizer, model_name)
        self.task_name = "dyck"
        self.openers = "([{<"
        self.closers = ")]}>"
        self.pairs = dict(zip(self.openers, self.closers))
        self.reverse_pairs = dict(zip(self.closers, self.openers))

    def _get_system_prompt(self) -> str:
        return (
            "You are completing a bracket sequence. "
            "Given an incomplete sequence of brackets, determine which single closing bracket "
            "is needed NEXT to properly close the most recent unclosed opening bracket. "
            "Respond with ONLY the closing bracket character: ) or ] or } or >"
        )

    def get_answer_candidates(self, sample: "TaskSample") -> Tuple[List[str], List[str]]:
      correct = sample.answer.strip()
      wrong = [c for c in self.closers if c != correct][0]

      # CRITICAL: Since build_prompt ends in a space,
      # candidates MUST NOT start with a space.
      # We only check the character itself.
      return [correct], [wrong]

    def generate_counterfactual(self, sample: "TaskSample", step_index: int) -> "CounterfactualResult":
        original_step = sample.steps[step_index]
        corrupted_step = original_step

        # Strategy 1: Corrupt the bracket character (Read 'X' -> Read 'Y')
        # FIXED REGEX: Use (?:...|...) for alternatives, not [] which is for single chars
        bracket_match = re.search(r"Read '([(\[{<)\]}>\)])' (?:→|->|to)", original_step)

        if bracket_match:
            orig_bracket = bracket_match.group(1)
            if orig_bracket in self.openers:
                wrong_openers = [b for b in self.openers if b != orig_bracket]
                new_bracket = random.choice(wrong_openers)
            else:
                wrong_closers = [b for b in self.closers if b != orig_bracket]
                new_bracket = random.choice(wrong_closers) if wrong_closers else orig_bracket
            corrupted_step = original_step.replace(f"'{orig_bracket}'", f"'{new_bracket}'", 1)

        # Strategy 2: Corrupt stack representation (if Strategy 1 didn't apply or failed)
        if corrupted_step == original_step:
            stack_match = re.search(r"Stack: \[([^\]]*)\]", original_step)
            if stack_match:
                stack_content = stack_match.group(1)
                # Add or remove a bracket from stack
                if len(stack_content) > 0:
                    corrupted_stack = stack_content[:-1]  # Remove last
                else:
                    corrupted_stack = "("  # Add fake
                corrupted_step = original_step.replace(
                    f"[{stack_content}]", f"[{corrupted_stack}]"
                )

        delta_tokens = self._calculate_token_delta(original_step, corrupted_step)
        steps = sample.steps[:step_index] + [corrupted_step] + sample.steps[step_index + 1:]

        return CounterfactualResult(
            steps=steps, corruption_type="bracket_error", step_index=step_index,
            expected_answer_change=True, semantic_distance=1.0,
            original_step=original_step, corrupted_step=corrupted_step,
            original_steps=sample.steps.copy(), token_count_delta=delta_tokens
        )

    def generate_paraphrase(self, sample: "TaskSample", step_index: int) -> "CounterfactualResult":
        original_step = sample.steps[step_index]
        # Meaning-preserving paraphrase
        para_step = original_step.replace("Read", "Saw").replace("Stack:", "Current stack:")
        if para_step == original_step:
            para_step = original_step.replace("→", "->")

        steps = sample.steps[:step_index] + [para_step] + sample.steps[step_index + 1:]
        return CounterfactualResult(
            steps=steps, corruption_type="paraphrase", step_index=step_index,
            expected_answer_change=False, semantic_distance=0.0,
            original_step=original_step, corrupted_step=para_step,
            original_steps=sample.steps.copy(), is_control=True
        )

    def validate_sample(self, sample: "TaskSample") -> bool:
        if not sample.answer or sample.answer.strip() not in self.closers:
            return False
        if not sample.steps or len(sample.steps) < 2:
            return False
        return True


class ProntoQATask(ReasoningTask):
    def __init__(self, tokenizer, model_name: str):
        super().__init__(tokenizer, model_name)
        self.task_name = "pronto_qa"

    def _get_system_prompt(self) -> str:
        return "Solve the logical reasoning problem step by step. Answer with 'True' or 'False'."

    def get_answer_candidates(self, sample: "TaskSample") -> Tuple[List[str], List[str]]:
        ans = sample.answer.strip() if sample else "True"
        if ans.lower() not in ["true", "false"]: ans = "True"
        pos_str = "True" if ans.lower() == "true" else "False"
        neg_str = "False" if pos_str == "True" else "True"
        def variants(s):
            return [s, " " + s, "\n" + s, s.lower(), " " + s.lower()]
        return variants(pos_str), variants(neg_str)

    def _canonicalize_label(self, text: Any) -> Optional[str]:
        if text is None:
            return None
        s = str(text).strip().lower()
        if s in {"true", "yes"}:
            return "True"
        if s in {"false", "no"}:
            return "False"
        return None

    def parse_model_answer(self, generated_text: str, sample: "TaskSample") -> Any:
        text = str(generated_text).strip()
        if not text:
            return None

        patterns = [
            r'^\s*(true|false|yes|no)\b',
            r'\b(?:answer|final answer)\s*(?:is|:)?\s*(true|false|yes|no)\b',
        ]
        for pattern in patterns:
            match = re.search(pattern, text, flags=re.IGNORECASE)
            if match:
                return self._canonicalize_label(match.group(1))
        return None

    def is_prediction_correct(self, parsed_prediction: Any, sample: "TaskSample") -> bool:
        return self._canonicalize_label(parsed_prediction) == self._canonicalize_label(sample.answer if sample else None)

    def get_target_answer_string(self, sample: "TaskSample") -> str:
        return self._canonicalize_label(sample.answer if sample else None) or "True"

    def generate_counterfactual(self, sample: "TaskSample", step_index: int) -> "CounterfactualResult":
        original_step = sample.steps[step_index]
        corrupted_step = original_step
        corruption_type = "entity_substitution"
        CATEGORIES = ["wumpus", "gorpus", "rompus", "jompus", "zumpus", "tumpus", "yumpus", "impus"]

        # Pattern: Inference or Conclusion
        match_inf = re.search(r"(.*,\s*\w+\s+is\s+)(\w+)(\s*)$", original_step)
        match_conc = re.search(r"(Conclusion:\s*\w+\s+is\s+)(\w+)(,.*)", original_step)

        if match_inf:
            prefix, inferred_class, suffix = match_inf.groups()
            wrong_classes = [c for c in CATEGORIES if c != inferred_class]
            if wrong_classes:
                corrupted_step = f"{prefix}{random.choice(wrong_classes)}{suffix}"
                corruption_type = "inference_substitution"
        elif match_conc:
            prefix, inferred_class, suffix = match_conc.groups()
            wrong_classes = [c for c in CATEGORIES if c != inferred_class]
            if wrong_classes:
                corrupted_step = f"{prefix}{random.choice(wrong_classes)}{suffix}"
                corruption_type = "conclusion_substitution"
        else:
            if " are " in original_step:
                corrupted_step = original_step.replace(" are ", " are not ", 1)
                corruption_type = "rule_negation"
            elif " is " in original_step:
                corrupted_step = original_step.replace(" is ", " is not ", 1)
                corruption_type = "fact_negation"

        delta = self._calculate_token_delta(original_step, corrupted_step)
        steps = sample.steps[:step_index] + [corrupted_step] + sample.steps[step_index + 1:]
        return CounterfactualResult(
            steps=steps, corruption_type=corruption_type, step_index=step_index,
            expected_answer_change=True, semantic_distance=1.0,
            original_step=original_step, corrupted_step=corrupted_step,
            original_steps=sample.steps.copy(), token_count_delta=delta
        )

    def generate_paraphrase(self, sample: "TaskSample", step_index: int) -> "CounterfactualResult":
        original_step = sample.steps[step_index]
        para_step = original_step.replace("Since", "Because").replace("Conclusion", "Therefore")
        if para_step == original_step: para_step = original_step + " "
        steps = sample.steps[:step_index] + [para_step] + sample.steps[step_index + 1:]
        return CounterfactualResult(
            steps=steps, corruption_type="paraphrase", step_index=step_index,
            expected_answer_change=False, semantic_distance=0.0,
            original_step=original_step, corrupted_step=para_step,
            original_steps=sample.steps.copy(), is_control=True
        )

    def validate_sample(self, sample: "TaskSample") -> bool:
        if not sample.steps: return False
        last_step = sample.steps[-1].lower()
        ans = sample.answer.lower().strip()
        return ans in last_step


def _corrupt_arithmetic_reasoning_step(original_step: str) -> Tuple[str, str]:
    corrupted_step = original_step
    corruption_type = "numerical_error"

    op_match = re.search(r'(\d+)\s*([+*/×÷-])\s*(\d+)', original_step)
    if op_match:
        n1, op, n2 = op_match.groups()
        full = op_match.group(0)
        swaps = {'+': '-', '-': '+', '*': '/', '/': '*', '×': '÷', '÷': '×'}
        if op in swaps:
            corrupted_step = original_step.replace(full, f"{n1} {swaps[op]} {n2}", 1)
            corruption_type = "operator_swap"

    if corrupted_step == original_step:
        nums = re.findall(r'\b(\d+(?:\.\d+)?)\b', original_step)
        if nums:
            target = nums[-1]
            try:
                val = float(target)
                new_val = val * 10
                new_str = str(int(new_val)) if new_val.is_integer() else f"{new_val:.2f}"
                corrupted_step = original_step.replace(target, new_str, 1)
                corruption_type = "perturbation"
            except Exception:
                pass

    return corrupted_step, corruption_type


def _paraphrase_arithmetic_reasoning_step(original_step: str) -> str:
    replacements = [
        ("add", "sum"),
        ("subtract", "minus"),
        ("multiply", "times"),
        ("divide", "split"),
        ("Therefore", "Hence"),
    ]
    para_step = original_step
    for old, new in replacements:
        if old in para_step:
            para_step = para_step.replace(old, new, 1)
            break
    if para_step == original_step:
        para_step = original_step + " "
    return para_step


class GSM8KTask(ReasoningTask):
    def __init__(self, tokenizer, model_name: str):
        super().__init__(tokenizer, model_name)
        self.task_name = "gsm8k"

    def _get_system_prompt(self) -> str:
        return "Solve the math problem step by step. Provide your final numerical answer."

    def get_nldd_scoring_mode(self) -> str:
        return "sequence_margin"

    def get_generation_max_new_tokens(self) -> int:
        return 30

    def get_counterfactual_perplexity_threshold(self) -> float:
        return 1.5

    def _extract_numeric_string(self, text: str) -> Optional[str]:
        if text is None:
            return None

        text_str = str(text)
        cue_patterns = [
            r"####\s*([-+]?\d+(?:,\d{3})*(?:\.\d+)?)",
            r"(?:final answer|answer(?:\s+is)?|therefore,?\s+the\s+answer\s+is|so\s+the\s+answer\s+is)\D*([-+]?\d+(?:,\d{3})*(?:\.\d+)?)",
        ]
        for pattern in cue_patterns:
            matches = re.findall(pattern, text_str, flags=re.IGNORECASE)
            if matches:
                return matches[-1]

        matches = re.findall(r"[-+]?\d+(?:,\d{3})*(?:\.\d+)?", text_str)
        return matches[-1] if matches else None

    def _canonicalize_numeric_string(self, text: str) -> Optional[str]:
        from decimal import Decimal, InvalidOperation

        raw = self._extract_numeric_string(text)
        if raw is None:
            return None
        try:
            value = Decimal(raw.replace(",", ""))
        except InvalidOperation:
            return None
        if not value.is_finite():
            return None
        if value == value.to_integral():
            canon = str(int(value))
        else:
            canon = format(value.normalize(), "f")
            if "." in canon:
                canon = canon.rstrip("0").rstrip(".")
        if canon in {"", "-0", "-0.0"}:
            canon = "0"
        return canon

    def parse_model_answer(self, generated_text: str, sample: "TaskSample") -> Any:
        return self._canonicalize_numeric_string(generated_text)

    def is_prediction_correct(self, parsed_prediction: Any, sample: "TaskSample") -> bool:
        pred = self._canonicalize_numeric_string(parsed_prediction)
        gold = self._canonicalize_numeric_string(sample.answer if sample else None)
        return pred is not None and gold is not None and pred == gold

    def get_target_answer_string(self, sample: "TaskSample") -> str:
        return str(sample.answer).strip()

    def _build_numeric_variants(self, text: str) -> List[str]:
        from decimal import Decimal, InvalidOperation

        canon = self._canonicalize_numeric_string(text)
        if canon is None:
            return []

        variants = []
        for cand in [canon, f" {canon}"]:
            if cand not in variants:
                variants.append(cand)

        raw = self._extract_numeric_string(text)
        if raw is not None:
            for cand in [raw, f" {raw}"]:
                if cand not in variants:
                    variants.append(cand)

        try:
            value = Decimal(canon.replace(",", ""))
            if value == value.to_integral():
                comma_fmt = f"{int(value):,}"
                for cand in [comma_fmt, f" {comma_fmt}"]:
                    if cand not in variants:
                        variants.append(cand)
        except (InvalidOperation, ValueError):
            pass

        return variants

    def build_nldd_scoring_spec(self, sample: "TaskSample", predicted_answer: Optional[str] = None) -> Dict[str, Any]:
        from decimal import Decimal, InvalidOperation

        gold = self._canonicalize_numeric_string(sample.answer if sample else None)
        if gold is None:
            return {
                "mode": self.get_nldd_scoring_mode(),
                "positive_candidates": [],
                "negative_candidates": [],
            }

        positive_candidates = self._build_numeric_variants(sample.answer)

        try:
            gold_value = Decimal(gold)
        except InvalidOperation:
            gold_value = None

        distractor_bases = ["0"]
        if gold_value is not None:
            distractor_bases.extend([
                str(gold_value + Decimal("1")),
                str(gold_value - Decimal("1")),
                str(gold_value * Decimal("10")),
            ])
            if gold_value != 0:
                distractor_bases.append(str(gold_value / Decimal("10")))

        if predicted_answer is not None:
            distractor_bases.append(predicted_answer)

        negative_candidates = []
        seen = set()
        for base in distractor_bases:
            base_canon = self._canonicalize_numeric_string(base)
            if base_canon is None or base_canon == gold or base_canon in seen:
                continue
            seen.add(base_canon)
            negative_candidates.extend(self._build_numeric_variants(base_canon))

        return {
            "mode": self.get_nldd_scoring_mode(),
            "positive_candidates": list(dict.fromkeys(positive_candidates)),
            "negative_candidates": list(dict.fromkeys(negative_candidates)),
        }

    def get_answer_candidates(self, sample: "TaskSample") -> Tuple[List[str], List[str]]:
        if not sample:
            return [], []
        correct = sample.answer.strip()
        return [correct, " " + correct], ["0", " 0"]

    def generate_counterfactual(self, sample: "TaskSample", step_index: int) -> "CounterfactualResult":
        original_step = sample.steps[step_index]
        corrupted_step, corruption_type = _corrupt_arithmetic_reasoning_step(original_step)

        delta = self._calculate_token_delta(original_step, corrupted_step)
        steps = sample.steps[:step_index] + [corrupted_step] + sample.steps[step_index + 1:]
        return CounterfactualResult(
            steps=steps,
            corruption_type=corruption_type,
            step_index=step_index,
            expected_answer_change=True,
            semantic_distance=1.0,
            original_step=original_step,
            corrupted_step=corrupted_step,
            original_steps=sample.steps.copy(),
            token_count_delta=delta,
        )

    def generate_paraphrase(self, sample: "TaskSample", step_index: int) -> "CounterfactualResult":
        original_step = sample.steps[step_index]
        para_step = _paraphrase_arithmetic_reasoning_step(original_step)
        steps = sample.steps[:step_index] + [para_step] + sample.steps[step_index + 1:]
        return CounterfactualResult(
            steps=steps,
            corruption_type="paraphrase",
            step_index=step_index,
            expected_answer_change=False,
            semantic_distance=0.0,
            original_step=original_step,
            corrupted_step=para_step,
            original_steps=sample.steps.copy(),
            is_control=True,
        )

    def validate_sample(self, sample: "TaskSample") -> bool:
        if not sample.steps:
            return False
        try:
            ans_val = float(sample.answer.replace(",", ""))
            nums = re.findall(r"[-+]?\d*\.\d+|\d+", sample.steps[-1].replace(",", ""))
            return any(abs(float(n) - ans_val) < 1e-4 for n in nums)
        except Exception:
            return False


def _replace_strategyqa_phrase(text: str, source_phrase: str, target_phrase: str) -> str:
    pattern = r"\b" + re.escape(source_phrase).replace(r"\ ", r"\s+") + r"\b"
    match = re.search(pattern, text, flags=re.IGNORECASE)
    if not match:
        return text

    matched_text = match.group(0)
    replacement = target_phrase
    if matched_text.isupper():
        replacement = target_phrase.upper()
    elif matched_text[0].isupper():
        replacement = target_phrase.capitalize()

    return text[:match.start()] + replacement + text[match.end():]


def _swap_strategyqa_polarity(original_step: str) -> Tuple[str, str]:
    prefix_match = re.match(r'^(Step\s+\d+:\s*)(.*)$', original_step)
    step_prefix = prefix_match.group(1) if prefix_match else ""
    body = prefix_match.group(2) if prefix_match else original_step

    prefix_swaps = [
        ("Isn't ", "Is ", "negation_removal"),
        ("Aren't ", "Are ", "negation_removal"),
        ("Wasn't ", "Was ", "negation_removal"),
        ("Weren't ", "Were ", "negation_removal"),
        ("Can't ", "Can ", "negation_removal"),
        ("Doesn't ", "Does ", "negation_removal"),
        ("Don't ", "Do ", "negation_removal"),
        ("Didn't ", "Did ", "negation_removal"),
        ("Hasn't ", "Has ", "negation_removal"),
        ("Haven't ", "Have ", "negation_removal"),
        ("Is ", "Isn't ", "negation_insertion"),
        ("Are ", "Aren't ", "negation_insertion"),
        ("Was ", "Wasn't ", "negation_insertion"),
        ("Were ", "Weren't ", "negation_insertion"),
        ("Can ", "Can't ", "negation_insertion"),
        ("Does ", "Doesn't ", "negation_insertion"),
        ("Do ", "Don't ", "negation_insertion"),
        ("Did ", "Didn't ", "negation_insertion"),
        ("Has ", "Hasn't ", "negation_insertion"),
        ("Have ", "Haven't ", "negation_insertion"),
        ("What is ", "What isn't ", "negation_insertion"),
        ("What are ", "What aren't ", "negation_insertion"),
        ("What does ", "What doesn't ", "negation_insertion"),
        ("What do ", "What don't ", "negation_insertion"),
        ("Who is ", "Who isn't ", "negation_insertion"),
        ("Who are ", "Who aren't ", "negation_insertion"),
        ("Who does ", "Who doesn't ", "negation_insertion"),
        ("Who do ", "Who don't ", "negation_insertion"),
        ("Who kills ", "Who doesn't kill ", "negation_insertion"),
    ]
    for source_phrase, target_phrase, corruption_type in prefix_swaps:
        if body.startswith(source_phrase):
            return f"{step_prefix}{target_phrase}{body[len(source_phrase):]}", corruption_type

    phrase_swaps = [
        ("the same as", "different from", "equivalence_swap"),
        ("different from", "the same as", "equivalence_swap"),
        ("high in", "low in", "attribute_swap"),
        ("low in", "high in", "attribute_swap"),
        ("more", "less", "comparator_swap"),
        ("less", "more", "comparator_swap"),
        ("before", "after", "temporal_swap"),
        ("after", "before", "temporal_swap"),
        ("positive", "negative", "polarity_swap"),
        ("negative", "positive", "polarity_swap"),
        ("same", "different", "equivalence_swap"),
        ("different", "same", "equivalence_swap"),
    ]
    for source_phrase, target_phrase, corruption_type in phrase_swaps:
        updated_body = _replace_strategyqa_phrase(body, source_phrase, target_phrase)
        if updated_body != body:
            return f"{step_prefix}{updated_body}", corruption_type

    auxiliary_swaps = [
        (" is not ", " is ", "negation_removal"),
        (" are not ", " are ", "negation_removal"),
        (" was not ", " was ", "negation_removal"),
        (" were not ", " were ", "negation_removal"),
        (" does not ", " does ", "negation_removal"),
        (" do not ", " do ", "negation_removal"),
        (" did not ", " did ", "negation_removal"),
        (" cannot ", " can ", "negation_removal"),
        (" is ", " is not ", "negation_insertion"),
        (" are ", " are not ", "negation_insertion"),
        (" was ", " was not ", "negation_insertion"),
        (" were ", " were not ", "negation_insertion"),
        (" can ", " cannot ", "negation_insertion"),
        (" does ", " does not ", "negation_insertion"),
        (" do ", " do not ", "negation_insertion"),
        (" did ", " did not ", "negation_insertion"),
    ]
    for source_phrase, target_phrase, corruption_type in auxiliary_swaps:
        if source_phrase in body:
            return f"{step_prefix}{body.replace(source_phrase, target_phrase, 1)}", corruption_type

    body = body.strip()
    if body:
        lowered_body = body[0].lower() + body[1:] if len(body) > 1 else body.lower()
        return f"{step_prefix}Not {lowered_body}", "polarity_insertion"

    return original_step, "polarity_insertion"


def _paraphrase_strategyqa_step(original_step: str) -> str:
    if original_step.startswith("Therefore,"):
        return original_step.replace("Therefore,", "Thus,", 1)

    match = re.match(r'^Step\s+(\d+):\s*(.*)$', original_step)
    if match:
        step_num, body = match.groups()
        return f"Subquestion {step_num}: {body}"

    return original_step + " "


class StrategyQATask(ReasoningTask):
    def __init__(self, tokenizer, model_name: str):
        super().__init__(tokenizer, model_name)
        self.task_name = "strategyqa"

    def _get_system_prompt(self) -> str:
        return "Answer the commonsense multi-hop question step by step. Respond with ONLY Yes or No."

    def get_generation_max_new_tokens(self) -> int:
        return 12

    def _canonicalize_label(self, text: Any) -> Optional[str]:
        if text is None:
            return None
        s = str(text).strip().lower()
        if s in {"yes", "true"}:
            return "Yes"
        if s in {"no", "false"}:
            return "No"
        return None

    def _label_variants(self, label: str) -> List[str]:
        canonical = self._canonicalize_label(label) or "Yes"
        lower = canonical.lower()
        return [canonical, f" {canonical}", f"\n{canonical}", lower, f" {lower}", f"\n{lower}"]

    def get_answer_candidates(self, sample: "TaskSample") -> Tuple[List[str], List[str]]:
        gold = self._canonicalize_label(sample.answer if sample else None) or "Yes"
        negative = "No" if gold == "Yes" else "Yes"
        return self._label_variants(gold), self._label_variants(negative)

    def parse_model_answer(self, generated_text: str, sample: "TaskSample") -> Any:
        text = str(generated_text).strip()
        if not text:
            return None

        patterns = [
            r'^\s*(yes|no|true|false)\b',
            r'\b(?:answer|final answer)\s*(?:is|:)?\s*(yes|no|true|false)\b',
        ]
        for pattern in patterns:
            match = re.search(pattern, text, flags=re.IGNORECASE)
            if match:
                return self._canonicalize_label(match.group(1))
        return None

    def is_prediction_correct(self, parsed_prediction: Any, sample: "TaskSample") -> bool:
        return self._canonicalize_label(parsed_prediction) == self._canonicalize_label(sample.answer if sample else None)

    def get_target_answer_string(self, sample: "TaskSample") -> str:
        return self._canonicalize_label(sample.answer if sample else None) or "Yes"

    def generate_counterfactual(self, sample: "TaskSample", step_index: int) -> "CounterfactualResult":
        original_step = sample.steps[step_index]
        if step_index == len(sample.steps) - 1:
            original_answer = self._canonicalize_label(sample.answer) or "Yes"
            flipped_answer = "No" if original_answer == "Yes" else "Yes"
            corrupted_step = original_step.replace(
                f"answer is {original_answer}",
                f"answer is {flipped_answer}",
                1,
            )
            corruption_type = "answer_flip"
        else:
            corrupted_step, corruption_type = _swap_strategyqa_polarity(original_step)

        delta = self._calculate_token_delta(original_step, corrupted_step)
        steps = sample.steps[:step_index] + [corrupted_step] + sample.steps[step_index + 1:]
        return CounterfactualResult(
            steps=steps,
            corruption_type=corruption_type,
            step_index=step_index,
            expected_answer_change=True,
            semantic_distance=1.0,
            original_step=original_step,
            corrupted_step=corrupted_step,
            original_steps=sample.steps.copy(),
            token_count_delta=delta,
        )

    def generate_paraphrase(self, sample: "TaskSample", step_index: int) -> "CounterfactualResult":
        original_step = sample.steps[step_index]
        para_step = _paraphrase_strategyqa_step(original_step)
        steps = sample.steps[:step_index] + [para_step] + sample.steps[step_index + 1:]
        return CounterfactualResult(
            steps=steps,
            corruption_type="paraphrase",
            step_index=step_index,
            expected_answer_change=False,
            semantic_distance=0.0,
            original_step=original_step,
            corrupted_step=para_step,
            original_steps=sample.steps.copy(),
            is_control=True,
        )

    def validate_sample(self, sample: "TaskSample") -> bool:
        if not sample.steps or len(sample.steps) < 2:
            return False
        gold = self._canonicalize_label(sample.answer)
        if gold not in {"Yes", "No"}:
            return False
        facts = sample.metadata.get("raw_facts", []) if sample.metadata else []
        if not facts:
            return False
        return sample.steps[-1].strip() == f"Therefore, the answer is {gold}."


In [ ]:
# ============================================================================
# SECTION 5.5: ACL-Compliant Dataset Generation (STRICT SPLIT)
# ============================================================================

BENCHMARK_ORDER = ["dyck", "pronto_qa", "gsm8k", "strategyqa"]
BENCHMARK_GLOBAL_PREFIX = {
    "dyck": "dyck",
    "pronto_qa": "pronto",
    "gsm8k": "gsm8k",
    "strategyqa": "strategyqa",
}
BENCHMARK_RESULT_VAR = {
    "dyck": "results_dyck",
    "pronto_qa": "results_pronto",
    "gsm8k": "results_gsm",
    "strategyqa": "results_strategyqa",
}


def build_benchmark_registry(tokenizer, model_name: str, config: ExperimentConfig) -> Dict[str, Dict[str, Any]]:
    return {
        "dyck": {
            "display_name": "Dyck-n",
            "output_stem": "dyck",
            "task_factory": lambda: DyckTask(tokenizer, model_name),
            "total_samples": config.dyck_total_samples,
            "eval_loader": lambda n: generate_dyck_data(n, config.dyck_min_length, config.dyck_max_length),
            "probe_loader": lambda n: generate_dyck_data(n, config.dyck_min_length, config.dyck_max_length),
        },
        "pronto_qa": {
            "display_name": "ProntoQA",
            "output_stem": "pronto_qa",
            "task_factory": lambda: ProntoQATask(tokenizer, model_name),
            "total_samples": config.pronto_total_samples,
            "eval_loader": lambda n: generate_pronto_data_v2(n, config.pronto_min_hops, config.pronto_max_hops),
            "probe_loader": lambda n: generate_pronto_data_v2(n, config.pronto_min_hops, config.pronto_max_hops),
        },
        "gsm8k": {
            "display_name": "GSM8K",
            "output_stem": "gsm8k",
            "task_factory": lambda: GSM8KTask(tokenizer, model_name),
            "total_samples": config.gsm8k_total_samples,
            "eval_loader": lambda n: load_gsm8k_data(n, config.gsm8k_split),
            "probe_loader": lambda n: load_gsm8k_data(n, config.gsm8k_split),
        },
        "strategyqa": {
            "display_name": "StrategyQA",
            "output_stem": "strategyqa",
            "task_factory": lambda: StrategyQATask(tokenizer, model_name),
            "total_samples": config.strategyqa_total_samples,
            "eval_loader": lambda n: load_strategyqa_data(n, partition="eval", config=config),
            "probe_loader": lambda n: load_strategyqa_data(n, partition="probe", config=config),
        },
    }


benchmark_registry = build_benchmark_registry(tokenizer, model_manager.get_model_short_name(), config)
benchmark_artifacts = {}

print("\n" + "="*70)
print("GENERATING ACL-COMPLIANT DATASETS (STRICT SPLIT)")
print("="*70)

for benchmark_key in BENCHMARK_ORDER:
    spec = benchmark_registry[benchmark_key]
    task = spec["task_factory"]()
    data_bundle, probe_dataset = generate_split_data(
        spec["eval_loader"],
        spec["probe_loader"],
        task,
        config,
        spec["total_samples"],
        n_probe=config.probe_total_samples,
    )

    benchmark_artifacts[benchmark_key] = {
        "task": task,
        "dataset": data_bundle["faithful_samples"],
        "probe_dataset": probe_dataset,
        "counterfactuals": data_bundle["counterfactuals"],
        "controls": data_bundle.get("controls", {}),
    }

    prefix = BENCHMARK_GLOBAL_PREFIX[benchmark_key]
    globals()[f"{prefix}_task"] = task
    globals()[f"{prefix}_dataset"] = benchmark_artifacts[benchmark_key]["dataset"]
    globals()[f"{prefix}_probe_dataset"] = benchmark_artifacts[benchmark_key]["probe_dataset"]
    globals()[f"{prefix}_counterfactuals"] = benchmark_artifacts[benchmark_key]["counterfactuals"]

print("\n✅ Datasets Ready & Split (ACL-Compliant)")


## (6) NLDD Analyzer

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from typing import List, Dict, Tuple

# ======================================================================
# MODULE B: ROBUST NLDD Implementation (ACL 2026 - Global Scaling)
# ======================================================================
def build_replacement_steps(sample, cf) -> List[str]:
    """Shared helper for rebuilding full replacement chains across analyzers."""
    if hasattr(cf, "steps") and len(cf.steps) == len(sample.steps):
        return cf.steps

    if hasattr(cf, "step_index") and 0 <= cf.step_index < len(sample.steps):
        corrupted_step = getattr(cf, "corrupted_step", None)
        if corrupted_step is None and hasattr(cf, "steps") and cf.steps:
            if len(cf.steps) > cf.step_index:
                corrupted_step = cf.steps[cf.step_index]
            else:
                corrupted_step = cf.steps[-1]

        if corrupted_step is not None:
            k = cf.step_index
            return sample.steps[:k] + [corrupted_step] + sample.steps[k + 1:]

    return cf.steps if hasattr(cf, "steps") else sample.steps

def get_analysis_counterfactual_rows(sample, sample_counterfactuals, config=None, include_controls=False) -> List[Dict]:
    """Return validated counterfactual rows with rebuilt replacement chains."""
    rows = []
    if not sample_counterfactuals:
        return rows

    for cf in sample_counterfactuals:
        if not include_controls and getattr(cf, "is_control", False):
            continue
        if hasattr(cf, "validate") and not cf.validate():
            continue

        step_index = getattr(cf, "step_index", None)
        if step_index is None or not (0 <= step_index < len(sample.steps)):
            continue

        replacement_steps = build_replacement_steps(sample, cf)
        if len(replacement_steps) != len(sample.steps):
            continue

        rows.append({
            "counterfactual": cf,
            "replacement_steps": replacement_steps,
            "step_index": int(step_index),
            "step_k": int(step_index) + 1,
            "is_control": bool(getattr(cf, "is_control", False)),
        })

    return rows

class NLDDAnalyzer:
    """
    Module B: Faithfulness Measurement via NLDD.

    Robustness Fixes:
    1. Global Scaling: Uses S_model (global_sigma) to prevent logit-flattening artifacts.
    2. Relaxed Tokenizer: Handles multi-token outputs (Gemma/DeepSeek numbers & booleans).
    3. EPS-Gating: Prevents division by zero or noise amplification on low-confidence samples.
    """
    def __init__(self, model, tokenizer, task, config):
        self.model = model
        self.tokenizer = tokenizer
        self.task = task
        self.config = config

        # This will be set by the calibrate() method
        self.global_sigma = 1.0
        self.calibration_mode = "logit_std"
        self._greedy_prediction_cache = {}

        self.last_run_audit = {}

        if self.config.epsilon > 0.1:
             print(f"Note: Using a conservative Epsilon of {self.config.epsilon}")

    def _score_sequence_logprob(self, prompt: str, candidate_str: str) -> Optional[float]:
        """Teacher-forced mean log-prob per token for an answer string variant."""
        if candidate_str is None:
            return None

        full_text = prompt + candidate_str
        inputs = self.tokenizer(full_text, return_tensors="pt").to(self.model.device)
        prompt_inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        start_idx = prompt_inputs.input_ids.shape[1]
        n_target_tokens = inputs.input_ids.shape[1] - start_idx
        if n_target_tokens < 1:
            return None

        with torch.no_grad():
            outputs = self.model(**inputs)
            shift_logits = outputs.logits[0, :-1, :]
            shift_labels = inputs.input_ids[0, 1:]
            log_probs = F.log_softmax(shift_logits, dim=-1)

        target_log_probs = []
        loop_start = max(0, start_idx - 1)
        loop_end = shift_labels.shape[0]
        for i in range(loop_start, loop_end):
            token_id = shift_labels[i]
            target_log_probs.append(log_probs[i, token_id].item())

        if not target_log_probs:
            return None
        return float(np.mean(target_log_probs))

    def _get_greedy_numeric_prediction(self, prompt: str) -> Optional[str]:
        if prompt in self._greedy_prediction_cache:
            return self._greedy_prediction_cache[prompt]

        prediction = None
        try:
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
            with torch.no_grad():
                gen_output = self.model.generate(
                    **inputs,
                    max_new_tokens=24,
                    do_sample=False,
                    pad_token_id=self.tokenizer.eos_token_id
                )
            gen_text = self.tokenizer.decode(
                gen_output[0, inputs.input_ids.shape[1]:],
                skip_special_tokens=True
            ).strip()
            if hasattr(self.task, "_canonicalize_numeric_string"):
                prediction = self.task._canonicalize_numeric_string(gen_text)
        except Exception:
            prediction = None

        self._greedy_prediction_cache[prompt] = prediction
        return prediction

    def calibrate(self, dataset: List, num_samples: int = 50):
        """
        Step 1: Fix the 'Yardstick'.
        Token-margin tasks calibrate on final-token logit std.
        Sequence-margin tasks calibrate on clean candidate-sequence score std.
        """
        print(f"\n[Calibration] Estimating S_model for {self.task.task_name}...")
        sigmas = []
        scoring_mode = self.task.get_nldd_scoring_mode()
        self.calibration_mode = "logit_std" if scoring_mode == "token_margin" else "sequence_score_std"

        calibration_subset = dataset[:num_samples]

        for sample in tqdm(calibration_subset, desc="Calculating Global Sigma"):
            prompt = self.task.build_prompt(sample.input, sample.steps)

            if scoring_mode == "token_margin":
                inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
                with torch.no_grad():
                    outputs = self.model(**inputs)
                    logits = outputs.logits[0, -1, :]
                sigmas.append(logits.std().item())
                continue

            predicted_answer = self._get_greedy_numeric_prediction(prompt)
            spec = self.task.build_nldd_scoring_spec(sample, predicted_answer=predicted_answer)
            seq_candidates = list(dict.fromkeys(spec.get("positive_candidates", []) + spec.get("negative_candidates", [])))
            seq_scores = []
            for cand in seq_candidates:
                score = self._score_sequence_logprob(prompt, cand)
                if score is not None and np.isfinite(score):
                    seq_scores.append(score)
            if len(seq_scores) >= 2:
                sigma = float(np.std(seq_scores))
                if sigma > 1e-10:
                    sigmas.append(sigma)

        if not sigmas:
            self.global_sigma = 1.0
            print(">>> Calibration skipped: no valid samples, keeping Global Sigma (S_model) = 1.0000")
            return

        self.global_sigma = float(np.mean(sigmas))
        print(f">>> Calibration Complete. Global Sigma (S_model) = {self.global_sigma:.4f} [{self.calibration_mode}]")
        print(f">>> Ready for robust NLDD calculation.\n")

    def compute_confidence(self, logits: torch.Tensor, correct_token_ids: List[int]) -> float:
        """
        Calculates Standardized Logit Difference (LD).
        Formula: (logit(y_correct) - max(logit(y_incorrect))) / global_sigma
        """
        # 1. Get logit of the correct answer(s)
        correct_logits_subset = logits[correct_token_ids]
        correct_logit = torch.max(correct_logits_subset).item()

        # 2. Mask correct tokens to find the best alternative (incorrect) answer
        logits_masked = logits.clone()
        logits_masked[correct_token_ids] = float('-inf')
        max_incorrect_logit = torch.max(logits_masked).item()

        # 3. Raw Margin
        raw_margin = correct_logit - max_incorrect_logit

        # 4. Standardize using the FIXED global constant
        # This is the "A(x, c)" metric from Lanham et al. (2023)
        return raw_margin / self.global_sigma

    def compute_kl_nldd(
        self,
        clean_logits: torch.Tensor,
        corrupt_logits: torch.Tensor,
        sample=None,
        clean_prompt: Optional[str] = None,
        corrupt_prompt: Optional[str] = None,
        clean_info: Optional[Dict[str, Any]] = None,
        corrupt_info: Optional[Dict[str, Any]] = None,
    ) -> float:
        import torch.nn.functional as F

        scoring_mode = self.task.get_nldd_scoring_mode()
        if scoring_mode != "sequence_margin" or sample is None or clean_prompt is None or corrupt_prompt is None:
            clean_probs = F.softmax(clean_logits, dim=-1)
            corrupt_log_probs = F.log_softmax(corrupt_logits, dim=-1)
            return F.kl_div(corrupt_log_probs, clean_probs, reduction='sum').item()

        clean_pred = (clean_info or {}).get("predicted_answer")
        corrupt_pred = (corrupt_info or {}).get("predicted_answer")
        spec_clean = self.task.build_nldd_scoring_spec(sample, predicted_answer=clean_pred)
        spec_corrupt = self.task.build_nldd_scoring_spec(sample, predicted_answer=corrupt_pred)
        seq_candidates = list(dict.fromkeys(
            spec_clean.get("positive_candidates", [])
            + spec_clean.get("negative_candidates", [])
            + spec_corrupt.get("positive_candidates", [])
            + spec_corrupt.get("negative_candidates", [])
        ))

        paired_scores = []
        for cand in seq_candidates:
            clean_score = self._score_sequence_logprob(clean_prompt, cand)
            corrupt_score = self._score_sequence_logprob(corrupt_prompt, cand)
            if clean_score is None or corrupt_score is None:
                continue
            if not (np.isfinite(clean_score) and np.isfinite(corrupt_score)):
                continue
            paired_scores.append((clean_score, corrupt_score))

        if len(paired_scores) < 2:
            clean_probs = F.softmax(clean_logits, dim=-1)
            corrupt_log_probs = F.log_softmax(corrupt_logits, dim=-1)
            return F.kl_div(corrupt_log_probs, clean_probs, reduction='sum').item()

        clean_score_tensor = torch.tensor([x[0] for x in paired_scores], device=self.model.device, dtype=torch.float32)
        corrupt_score_tensor = torch.tensor([x[1] for x in paired_scores], device=self.model.device, dtype=torch.float32)
        clean_probs = F.softmax(clean_score_tensor, dim=-1)
        corrupt_log_probs = F.log_softmax(corrupt_score_tensor, dim=-1)
        return F.kl_div(corrupt_log_probs, clean_probs, reduction='sum').item()

    def get_logits_and_confidence(self, prompt: str, sample) -> Dict:
        """Forward pass and scoring under the task-specific NLDD contract."""
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits[0, -1, :]
            probs = F.softmax(logits, dim=-1)

        scoring_mode = self.task.get_nldd_scoring_mode()

        if scoring_mode == "token_margin":
            spec = self.task.build_nldd_scoring_spec(sample)
            try:
                token_validation = TokenValidator.find_best_categorical_tokens(
                    self.tokenizer,
                    spec.get("positive_candidates", []),
                    spec.get("negative_candidates", []),
                    self.task.task_name,
                    require_single_token=True,
                )
            except ValueError:
                token_validation = None

            if token_validation is None or not token_validation.is_valid:
                return {
                    "valid": False,
                    "reason": "token_mapping_failed",
                    "scoring_mode": scoring_mode,
                    "calibration_mode": self.calibration_mode,
                }

            if not token_validation.is_single_token_pos or not token_validation.is_single_token_neg:
                return {
                    "valid": False,
                    "reason": "multi_token_categorical_answer",
                    "scoring_mode": scoring_mode,
                    "calibration_mode": self.calibration_mode,
                }

            pos_id = int(token_validation.positive_id)
            neg_id = int(token_validation.negative_id)
            confidence_margin = (logits[pos_id].item() - logits[neg_id].item()) / self.global_sigma
            pos_prob = probs[pos_id].item()
            is_valid_sample = pos_prob >= self.config.min_prob_threshold
            return {
                "valid": True,
                "confidence": confidence_margin,
                "pos_prob": pos_prob,
                "is_valid_prob": is_valid_sample,
                "logits": logits.detach().cpu(),
                "positive_token_id": pos_id,
                "negative_token_id": neg_id,
                "positive_token_text": token_validation.pos_text,
                "negative_token_text": token_validation.neg_text,
                "scoring_mode": scoring_mode,
                "calibration_mode": self.calibration_mode,
            }

        predicted_answer = self._get_greedy_numeric_prediction(prompt)
        spec = self.task.build_nldd_scoring_spec(sample, predicted_answer=predicted_answer)
        pos_candidates = list(dict.fromkeys(spec.get("positive_candidates", [])))
        neg_candidates = list(dict.fromkeys(spec.get("negative_candidates", [])))

        if not neg_candidates:
            return {
                "valid": False,
                "reason": "distractor_generation_failed",
                "scoring_mode": scoring_mode,
                "calibration_mode": self.calibration_mode,
                "predicted_answer": predicted_answer,
            }

        pos_scores = []
        for cand in pos_candidates:
            score = self._score_sequence_logprob(prompt, cand)
            if score is not None and np.isfinite(score):
                pos_scores.append(score)

        neg_scores = []
        for cand in neg_candidates:
            score = self._score_sequence_logprob(prompt, cand)
            if score is not None and np.isfinite(score):
                neg_scores.append(score)

        if not pos_scores or not neg_scores:
            return {
                "valid": False,
                "reason": "sequence_scoring_failed",
                "scoring_mode": scoring_mode,
                "calibration_mode": self.calibration_mode,
                "predicted_answer": predicted_answer,
            }

        best_pos_score = float(np.max(pos_scores))
        best_neg_score = float(np.max(neg_scores))
        confidence_margin = (best_pos_score - best_neg_score) / self.global_sigma
        pos_prob = float(np.exp(np.clip(best_pos_score, -60.0, 20.0)))
        is_valid_sample = pos_prob >= self.config.min_prob_threshold

        return {
            "valid": True,
            "confidence": confidence_margin,
            "pos_prob": pos_prob,
            "is_valid_prob": is_valid_sample,
            "logits": logits.detach().cpu(),
            "scoring_mode": scoring_mode,
            "calibration_mode": self.calibration_mode,
            "predicted_answer": predicted_answer,
            "n_positive_candidates": len(pos_candidates),
            "n_negative_candidates": len(neg_candidates),
        }

    def _build_replacement_steps(self, sample, cf) -> List[str]:
        """Build a full hybrid chain for replacement-based intervention."""
        return build_replacement_steps(sample, cf)

    def analyze_dataset(
        self, dataset: List, dataset_name: str,
        counterfactuals: Dict[int, List] = None
    ) -> pd.DataFrame:
        """
        Step 2: Calculate NLDD across the dataset.
        """
        all_results = []
        print(f"{'='*60}\nNLDD Analysis: {dataset_name}\n{'='*60}")
        n_rows_total = 0
        n_rows_valid = 0
        n_rows_low_margin = 0
        n_rows_token_mapping_failed = 0
        n_rows_clean_below_prob_threshold = 0
        n_rows_corrupt_below_prob_threshold = 0
        n_rows_distractor_generation_failed = 0
        n_rows_sequence_scoring_failed = 0
        n_samples_with_counterfactuals = 0
        scoring_mode = self.task.get_nldd_scoring_mode()
        calibration_mode = self.calibration_mode

        def _record_invalid_reason(reason: str, count: int = 1):
            nonlocal n_rows_token_mapping_failed, n_rows_distractor_generation_failed, n_rows_sequence_scoring_failed
            if reason == "token_mapping_failed":
                n_rows_token_mapping_failed += count
            elif reason == "distractor_generation_failed":
                n_rows_distractor_generation_failed += count
            elif reason == "sequence_scoring_failed":
                n_rows_sequence_scoring_failed += count

        if counterfactuals is None:
            self.last_run_audit = {
                "dataset": dataset_name,
                "scoring_mode": scoring_mode,
                "n_dataset_samples": int(len(dataset)),
                "n_samples_with_counterfactuals": 0,
                "n_rows_total": 0,
                "n_rows_valid": 0,
                "n_rows_low_margin": 0,
                "n_rows_token_mapping_failed": 0,
                "n_rows_clean_below_prob_threshold": 0,
                "n_rows_corrupt_below_prob_threshold": 0,
                "n_rows_distractor_generation_failed": 0,
                "n_rows_sequence_scoring_failed": 0,
                "calibration_mode": calibration_mode,
            }
            return pd.DataFrame()

        for idx, sample in enumerate(tqdm(dataset, desc=f"Analyzing")):
            sample_cf_rows = get_analysis_counterfactual_rows(
                sample,
                counterfactuals.get(idx) if counterfactuals is not None else None,
                config=self.config,
                include_controls=True
            )
            if not sample_cf_rows: continue
            n_samples_with_counterfactuals += 1

            # 1. CLEAN Baseline
            clean_prompt = self.task.build_prompt(sample.input, sample.steps)
            clean_res = self.get_logits_and_confidence(clean_prompt, sample)
            if not clean_res["valid"]:
                _record_invalid_reason(clean_res.get("reason", "sequence_scoring_failed"), len(sample_cf_rows))
                continue

            # 2. COUNTERFACTUAL Passes (full hybrid chain for replacement intervention)
            for cf_row in sample_cf_rows:
                cf = cf_row["counterfactual"]
                n_rows_total += 1
                replacement_steps = cf_row["replacement_steps"]
                corrupt_prompt = self.task.build_prompt(sample.input, replacement_steps)
                corrupt_res = self.get_logits_and_confidence(corrupt_prompt, sample)
                if not corrupt_res["valid"]:
                    _record_invalid_reason(corrupt_res.get("reason", "sequence_scoring_failed"), 1)
                    continue

                A_clean = clean_res["confidence"]
                A_corr = corrupt_res["confidence"]

                # 3. Calculate NLDD (Normalized Logit Difference Decay)
                low_margin = abs(A_clean) <= self.config.epsilon
                if low_margin:
                    n_rows_low_margin += 1
                nldd_score = 0.0
                if not low_margin:
                    # Decay = ((Before - After) / Before) * 100
                    nldd_score = ((A_clean - A_corr) / abs(A_clean)) * 100

                # Validity Check
                if not clean_res["is_valid_prob"]:
                    n_rows_clean_below_prob_threshold += 1
                if not corrupt_res["is_valid_prob"]:
                    n_rows_corrupt_below_prob_threshold += 1
                is_valid = clean_res["is_valid_prob"] and corrupt_res["is_valid_prob"]
                if is_valid:
                    n_rows_valid += 1

                # KL-divergence NLDD (robustness metric, §3.5)
                kl_nldd = self.compute_kl_nldd(
                    clean_res["logits"].to(self.model.device),
                    corrupt_res["logits"].to(self.model.device),
                    sample=sample,
                    clean_prompt=clean_prompt,
                    corrupt_prompt=corrupt_prompt,
                    clean_info=clean_res,
                    corrupt_info=corrupt_res,
                )

                all_results.append({
                    "sample_idx": idx,
                    "type": "control" if cf.is_control else "corruption",
                    "nldd": nldd_score,
                    "kl_nldd": kl_nldd,
                    "clean_confidence": A_clean,
                    "corrupt_confidence": A_corr,
                    "clean_pos_prob": clean_res["pos_prob"],
                    "corrupt_pos_prob": corrupt_res["pos_prob"],
                    "clean_valid_prob": clean_res["is_valid_prob"],
                    "corrupt_valid_prob": corrupt_res["is_valid_prob"],
                    "low_margin": low_margin,
                    "scoring_mode": clean_res.get("scoring_mode", scoring_mode),
                    "calibration_mode": clean_res.get("calibration_mode", calibration_mode),
                    "step_index": cf.step_index,
                    "step_k": cf.step_index + 1,
                    "is_valid": is_valid
                })

        self.last_run_audit = {
            "dataset": dataset_name,
            "scoring_mode": scoring_mode,
            "n_dataset_samples": int(len(dataset)),
            "n_samples_with_counterfactuals": int(n_samples_with_counterfactuals),
            "n_rows_total": int(n_rows_total),
            "n_rows_valid": int(n_rows_valid),
            "n_rows_low_margin": int(n_rows_low_margin),
            "n_rows_token_mapping_failed": int(n_rows_token_mapping_failed),
            "n_rows_clean_below_prob_threshold": int(n_rows_clean_below_prob_threshold),
            "n_rows_corrupt_below_prob_threshold": int(n_rows_corrupt_below_prob_threshold),
            "n_rows_distractor_generation_failed": int(n_rows_distractor_generation_failed),
            "n_rows_sequence_scoring_failed": int(n_rows_sequence_scoring_failed),
            "calibration_mode": calibration_mode,
        }

        df = pd.DataFrame(all_results)
        if not df.empty:
            corrupt_df = df[(df["type"] == "corruption") & (df["is_valid"] == True)]
            print(f"\nFinal Mean NLDD (Faithfulness): {corrupt_df['nldd'].mean():.2f}%")

        return df

## (6.5) Counterfactual Accuracy

In [ ]:
class CounterfactualAccuracyAnalyzer:
    """
    Module B.1: Counterfactual Accuracy Measurement.

    Camera-ready contract:
    - Uses the raw task prompt from `task.build_prompt(...)`
    - Uses teacher forcing for exact likelihood of the target answer
    - Shares the same replacement-chain helper as NLDD and pruning
    """
    def __init__(self, model, tokenizer, task, config):
        self.model = model
        self.tokenizer = tokenizer
        self.task = task
        self.config = config
        self.device = next(model.parameters()).device
        self.min_prob = 1e-7  # Numerical stability floor
        self.last_run_audit = {}
        self.last_failures_df = pd.DataFrame()

    def _apply_chat_template(self, prompt: str) -> str:
        """Camera-ready path uses the same raw task prompt contract as the rest of the pipeline."""
        return prompt

    def _get_log_prob_of_string(self, prompt: str, target_str: str) -> float:
        """
        Calculates the geometric mean probability of the target string given the prompt.
        Uses teacher forcing (next-token prediction) rather than generation.
        """
        target_str = target_str.strip()
        # Handle tokenizer weirdness with leading spaces
        variants = [f" {target_str}", target_str]
        best_prob = -float('inf')
        valid_measurement = False

        for v in variants:
            full_text = prompt + v
            inputs = self.tokenizer(full_text, return_tensors="pt").to(self.device)
            prompt_inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)

            # Start index: where the completion begins
            start_idx = prompt_inputs.input_ids.shape[1]
            n_target_tokens = inputs.input_ids.shape[1] - start_idx

            if n_target_tokens < 1: continue

            with torch.no_grad():
                outputs = self.model(**inputs)
                shift_logits = outputs.logits[0, :-1, :]
                shift_labels = inputs.input_ids[0, 1:]

                log_probs = F.log_softmax(shift_logits, dim=-1)
                target_log_probs = []

                loop_start = max(0, start_idx - 1)
                loop_end = shift_labels.shape[0]

                if loop_start < loop_end:
                    for i in range(loop_start, loop_end):
                        token_id = shift_labels[i]
                        target_log_probs.append(log_probs[i, token_id].item())

                if len(target_log_probs) > 0:
                    score = np.mean(target_log_probs)
                    valid_measurement = True
                    if score > best_prob:
                        best_prob = score

        if not valid_measurement or best_prob == -float('inf'):
            return self.min_prob

        return np.exp(best_prob)

    def _get_predicted_answer(self, raw_prompt: str, sample) -> Tuple[bool, float, Any]:
        """
        Dispatcher: Calculates behavioral accuracy (is_correct) and
        faithfulness probability (Prob of Ground Truth).
        """
        formatted_prompt = self._apply_chat_template(raw_prompt)

        inputs = self.tokenizer(formatted_prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            gen_output = self.model.generate(
                **inputs,
                max_new_tokens=self.task.get_generation_max_new_tokens(),
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )

        gen_text = self.tokenizer.decode(
            gen_output[0, inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        ).strip()

        parsed_prediction = self.task.parse_model_answer(gen_text, sample)
        is_correct = self.task.is_prediction_correct(parsed_prediction, sample)
        target_string = self.task.get_target_answer_string(sample)

        prob_correct = self._get_log_prob_of_string(formatted_prompt, target_string)

        return is_correct, prob_correct, gen_text

    def _build_replacement_steps(self, sample, cf) -> List[str]:
        """Build a full hybrid chain for replacement-based intervention."""
        return build_replacement_steps(sample, cf)

    def analyze_dataset(self, dataset, counterfactuals, dataset_name):
        all_results = []
        failure_rows = []
        skipped_samples = 0
        skipped_counterfactuals = 0
        analyzed_samples = 0
        requested_counterfactuals = 0

        for idx, sample in enumerate(tqdm(dataset, desc=f"Analyzing {dataset_name}")):
            sample_cf_rows = get_analysis_counterfactual_rows(
                sample,
                counterfactuals.get(idx) if counterfactuals is not None else None,
                config=self.config,
                include_controls=True
            )
            if not sample_cf_rows:
                continue

            analyzed_samples += 1
            try:
                # Baseline (Clean)
                clean_prompt = self.task.build_prompt(sample.input, sample.steps)
                clean_correct, clean_prob, _ = self._get_predicted_answer(clean_prompt, sample)

                if not np.isfinite(clean_prob): clean_prob = self.min_prob

                for cf_row in sample_cf_rows:
                    cf = cf_row["counterfactual"]
                    requested_counterfactuals += 1
                    try:
                        # Corrupted (Counterfactual)
                        replacement_steps = cf_row["replacement_steps"]
                        corrupt_prompt = self.task.build_prompt(sample.input, replacement_steps)
                        corrupt_correct, corrupt_prob, _ = self._get_predicted_answer(corrupt_prompt, sample)

                        if not np.isfinite(corrupt_prob): corrupt_prob = self.min_prob

                        all_results.append({
                            "sample_idx": idx,
                            "type": "control" if cf.is_control else "corruption",
                            "corruption_type": getattr(cf, "corruption_type", "unknown"),
                            "step_index": cf.step_index,
                            "step_k": cf.step_index + 1,
                            "clean_correct": clean_correct,
                            "corrupt_correct": corrupt_correct,
                            "answer_flipped": clean_correct and not corrupt_correct,
                            "clean_prob": clean_prob,
                            "corrupt_prob": corrupt_prob,
                            "prob_delta": clean_prob - corrupt_prob
                        })
                    except Exception as e:
                        skipped_counterfactuals += 1
                        failure_rows.append({
                            "dataset": dataset_name,
                            "sample_idx": idx,
                            "step_k": getattr(cf, "step_index", -1) + 1 if hasattr(cf, "step_index") else np.nan,
                            "failure_scope": "counterfactual",
                            "exception_type": type(e).__name__,
                            "message": str(e)[:200],
                        })
                        continue

            except Exception as e:
                skipped_samples += 1
                failure_rows.append({
                    "dataset": dataset_name,
                    "sample_idx": idx,
                    "step_k": np.nan,
                    "failure_scope": "sample",
                    "exception_type": type(e).__name__,
                    "message": str(e)[:200],
                })
                continue

        self.last_failures_df = pd.DataFrame(failure_rows)
        self.last_run_audit = {
            "dataset": dataset_name,
            "n_dataset_samples": int(len(dataset)),
            "n_samples_with_counterfactuals": int(analyzed_samples),
            "n_requested_counterfactuals": int(requested_counterfactuals),
            "n_successful_rows": int(len(all_results)),
            "n_sample_failures": int(skipped_samples),
            "n_counterfactual_failures": int(skipped_counterfactuals),
        }
        if skipped_samples or skipped_counterfactuals:
            print(f"   [Accuracy Audit] sample_failures={skipped_samples}, counterfactual_failures={skipped_counterfactuals}")
        return pd.DataFrame(all_results)

    def compute_horizon_accuracy(self, dataset: List, counterfactuals: Dict[int, List]) -> Dict:
        """
        RESTORED: Aggregates accuracy metrics by step depth for the 'Reasoning Horizon' plot.
        """
        horizon_data = defaultdict(lambda: {"flipped": [], "prob_delta": []})

        for idx, sample in enumerate(dataset):
            sample_cf_rows = get_analysis_counterfactual_rows(
                sample,
                counterfactuals.get(idx) if counterfactuals is not None else None,
                config=self.config,
                include_controls=False
            )
            if not sample_cf_rows: continue

            # Re-calculate clean per sample (using new prediction logic)
            clean_prompt = self.task.build_prompt(sample.input, sample.steps)
            clean_correct, clean_prob, _ = self._get_predicted_answer(clean_prompt, sample)

            for cf_row in sample_cf_rows:
                cf = cf_row["counterfactual"]
                replacement_steps = cf_row["replacement_steps"]
                corrupt_prompt = self.task.build_prompt(sample.input, replacement_steps)
                corrupt_correct, corrupt_prob, _ = self._get_predicted_answer(corrupt_prompt, sample)

                k = cf_row["step_k"]
                horizon_data[k]["flipped"].append(int(clean_correct and not corrupt_correct))
                horizon_data[k]["prob_delta"].append(clean_prob - corrupt_prob)

        results = {"steps": [], "flip_rate": [], "prob_delta": []}
        for k in sorted(horizon_data.keys()):
            flips = horizon_data[k]["flipped"]
            deltas = horizon_data[k]["prob_delta"]
            if len(flips) > 5:
                results["steps"].append(k)
                results["flip_rate"].append(np.mean(flips))
                results["prob_delta"].append(np.mean(deltas))

        return results

    def _print_summary(self, df: pd.DataFrame):
        if df.empty: return
        corrupt_df = df[df["type"] == "corruption"]
        if len(corrupt_df) > 0:
            print(f"\n[Corruption Results]")
            conditional_df = corrupt_df[corrupt_df["clean_correct"] == True]
            flip_rate = conditional_df['answer_flipped'].mean()*100 if len(conditional_df) > 0 else 0.0
            print(f"  Answer Flip Rate (clean-conditioned):  {flip_rate:.1f}%")
            print(f"  Mean Prob Delta:   {corrupt_df['prob_delta'].mean():.4f}")

## (7) Execute Enhanced Pipeline

In [ ]:
# ======================================================================
# ENHANCED EXECUTION LOOP - ACL 2026 COMPLIANT (v4.2 - ACCURACY FIXED)
# ======================================================================
def run_enhanced_analysis(
    task,
    dataset: List,
    probe_dataset: List,
    counterfactuals: Dict[int, List],
    task_name: str,
    toolkit: MechanisticToolkit,
    config: ExperimentConfig
) -> Dict[str, Any]:
    """
    Run complete enhanced analysis including Patching, Probing, RSA, and Horizon.
    """
    results = {"task": task_name}
    print(f"\n{'#'*60}")
    print(f"# ANALYSIS: {task_name.upper()}")
    print(f"{'#'*60}")

    # Stage B: NLDD Analysis
    print("\n[Stage B] NLDD Analysis...")
    analyzer = NLDDAnalyzer(model, tokenizer, task, config)
    analyzer.calibrate(dataset, num_samples=min(50, len(dataset)))
    df_nldd = analyzer.analyze_dataset(dataset, task_name, counterfactuals=counterfactuals)

    if df_nldd.empty:
        print(f"!!! Warning: No valid NLDD samples found for {task_name}")
        return {"nldd_df": df_nldd, "error": "empty_dataframe"}

    results["nldd_df"] = df_nldd
    results["nldd_audit"] = analyzer.last_run_audit

    nldd_audit_df = pd.DataFrame([analyzer.last_run_audit]) if analyzer.last_run_audit else pd.DataFrame()
    if not nldd_audit_df.empty:
        nldd_audit_path = f"{config.results_dir}/nldd_audit_{task_name}.csv"
        nldd_audit_df.to_csv(nldd_audit_path, index=False)
        print(f"   ✓ NLDD audit saved to {nldd_audit_path}")

    # ============================================================
    # Stage B.1: Counterfactual Accuracy (FIXED)
    # ============================================================
    print("\n[Stage B.1] Counterfactual Accuracy Analysis...")

    try:
        cf_analyzer = CounterfactualAccuracyAnalyzer(model, tokenizer, task, config)
        df_cf_accuracy = cf_analyzer.analyze_dataset(dataset, counterfactuals, task_name)

        # ALWAYS store the DataFrame (even if empty)
        results["cf_accuracy_df"] = df_cf_accuracy
        results["cf_accuracy_summary_df"] = pd.DataFrame()
        results["cf_accuracy_audit"] = cf_analyzer.last_run_audit
        results["cf_accuracy_failures_df"] = cf_analyzer.last_failures_df

        audit_df = pd.DataFrame([cf_analyzer.last_run_audit]) if cf_analyzer.last_run_audit else pd.DataFrame()
        if not audit_df.empty:
            audit_csv_path = f"{config.results_dir}/accuracy_audit_{task_name}.csv"
            audit_df.to_csv(audit_csv_path, index=False)
            print(f"   ✓ Audit saved to {audit_csv_path}")

        if cf_analyzer.last_failures_df is not None and not cf_analyzer.last_failures_df.empty:
            failures_csv_path = f"{config.results_dir}/accuracy_failures_{task_name}.csv"
            cf_analyzer.last_failures_df.to_csv(failures_csv_path, index=False)
            print(f"   ✓ Failure log saved to {failures_csv_path}")

        if not df_cf_accuracy.empty:
            corrupt_df = df_cf_accuracy[df_cf_accuracy["type"] == "corruption"]

            if len(corrupt_df) > 0:
                conditional_df = corrupt_df[corrupt_df["clean_correct"] == True]
                results["cf_flip_rate_raw"] = float(corrupt_df["answer_flipped"].mean())
                results["cf_flip_rate"] = float(conditional_df["answer_flipped"].mean()) if len(conditional_df) > 0 else 0.0
                results["conditional_flip_rate"] = results["cf_flip_rate"]
                results["raw_flip_rate"] = results["cf_flip_rate_raw"]
                results["cf_flip_rate_denominator"] = int(len(conditional_df))
                results["cf_prob_delta"] = corrupt_df["prob_delta"].mean()
                if cf_analyzer.last_run_audit is not None:
                    cf_analyzer.last_run_audit["n_clean_correct_corruptions"] = int(len(conditional_df))
                    cf_analyzer.last_run_audit["conditional_flip_rate"] = float(results["cf_flip_rate"])
                    cf_analyzer.last_run_audit["raw_flip_rate"] = float(results["cf_flip_rate_raw"])

                # Calculate and print accuracy
                clean_acc = corrupt_df["clean_correct"].mean() * 100
                corrupt_acc = corrupt_df["corrupt_correct"].mean() * 100

                summary_df = pd.DataFrame([{
                    "task": task_name,
                    "model": model_manager.get_model_short_name(),
                    "n_corruptions": len(corrupt_df),
                    "clean_accuracy_pct": clean_acc,
                    "corrupt_accuracy_pct": corrupt_acc,
                    "answer_flip_rate_pct": results["cf_flip_rate"] * 100,
                    "answer_flip_rate_raw_pct": results["cf_flip_rate_raw"] * 100,
                    "n_clean_correct_corruptions": len(conditional_df),
                    "mean_prob_delta": results["cf_prob_delta"],
                }])
                results["cf_accuracy_summary_df"] = summary_df

                print(f"   ✓ Analyzed {len(corrupt_df)} corruption samples")
                print(f"   ✓ Clean accuracy:   {clean_acc:.1f}%")
                print(f"   ✓ Corrupt accuracy: {corrupt_acc:.1f}%")
                print(f"   ✓ Flip rate (clean-conditioned): {results['cf_flip_rate']:.1%}")
                print(f"   ✓ Flip rate (raw rows): {results['cf_flip_rate_raw']:.1%}")
                print(f"   ✓ Prob delta: {results['cf_prob_delta']:.3f}")

                # Save accuracy to CSV
                csv_path = f"{config.results_dir}/accuracy_{task_name}.csv"
                df_cf_accuracy.to_csv(csv_path, index=False)
                print(f"   ✓ Saved to {csv_path}")

                summary_csv_path = f"{config.results_dir}/accuracy_summary_{task_name}.csv"
                summary_df.to_csv(summary_csv_path, index=False)
                print(f"   ✓ Summary saved to {summary_csv_path}")
            else:
                print(f"   ⚠️ WARNING: No corruption samples found!")
                results["cf_flip_rate"] = 0.0
                results["cf_flip_rate_raw"] = 0.0
                results["conditional_flip_rate"] = 0.0
                results["raw_flip_rate"] = 0.0
                results["cf_flip_rate_denominator"] = 0
                results["cf_prob_delta"] = 0.0
        else:
            print(f"   ⚠️ WARNING: Accuracy analysis returned empty DataFrame!")
            print(f"   Possible causes:")
            print(f"     - Counterfactuals dict is empty: {len(counterfactuals) == 0}")
            print(f"     - Dataset is empty: {len(dataset) == 0}")
            print(f"     - All samples failed validation")
            results["cf_flip_rate"] = 0.0
            results["cf_flip_rate_raw"] = 0.0
            results["conditional_flip_rate"] = 0.0
            results["raw_flip_rate"] = 0.0
            results["cf_flip_rate_denominator"] = 0
            results["cf_prob_delta"] = 0.0

    except Exception as e:
        print(f"   ✗ ERROR in accuracy analysis: {e}")
        import traceback
        traceback.print_exc()
        results["cf_accuracy_df"] = None
        results["cf_accuracy_summary_df"] = None
        results["cf_accuracy_audit"] = None
        results["cf_accuracy_failures_df"] = None
        results["cf_flip_rate"] = 0.0
        results["cf_flip_rate_raw"] = 0.0
        results["conditional_flip_rate"] = 0.0
        results["raw_flip_rate"] = 0.0
        results["cf_flip_rate_denominator"] = 0
        results["cf_prob_delta"] = 0.0

    # Stage D: Probing
    print("\n[Stage D] Linear Probing + Failure Mode Diagnosis...")
    probe_results = toolkit.train_linear_probes(task, probe_dataset, task_name)
    results["probing"] = probe_results
    acc = probe_results.get("aggregate_accuracy", probe_results.get("mean_accuracy", 0))
    print(f"   Probe Accuracy: {acc:.3f}")

    # Stage E: RSA (in run_enhanced_analysis function)
    if config.enable_rsa:
        print("\n[Stage E] RSA Analysis...")
        rsa_results = toolkit.compute_rsa_batch(
            task,
            dataset,
            counterfactuals=counterfactuals,  # NEW: Pass counterfactuals
            n_samples=config.rsa_n_samples
        )
        results["rsa"] = rsa_results
        print(f"   Mean RSA: {rsa_results['mean_rsa']:.3f}")

    # Print per-layer breakdown
    if "per_layer_rsa" in rsa_results:
        layer_rsa = rsa_results["per_layer_rsa"]
        early = np.mean([layer_rsa.get(i, 0) for i in range(min(5, len(layer_rsa)))])
        late = np.mean([layer_rsa.get(i, 0) for i in range(max(0, len(layer_rsa)-5), len(layer_rsa))])
        print(f"   Early Layers RSA: {early:.3f}")
        print(f"   Late Layers RSA: {late:.3f}")

    # Stage E-bis: CKA
    if config.enable_cka:
        print("\n[Stage E-bis] Computing Centered Kernel Alignment (CKA)...")
        cka_results = toolkit.compute_cka_batch(task, dataset, counterfactuals=counterfactuals, n_samples=config.cka_n_samples)
        results["cka"] = cka_results
        print(f"   Mean CKA: {cka_results['mean_cka']:.3f} (Slope: {cka_results['decay_slope']:.3f})")

    # Stage F/G: Geometry
    if config.enable_geometry:
        print("\n[Stage F/G] Clean-Trajectory TAS + PCA Visualization...")
        geometry_results = toolkit.analyze_geometry(dataset, task)
        results["geometry"] = geometry_results
        print(f"   Clean-Trajectory TAS Mean: {geometry_results['tas_mean']:.3f}")

    # [NEW] Stage H: Reasoning Horizon Analysis (Figure 1 Generator)
    print("\n[Stage H] Reasoning Horizon Analysis (Figure 1 Data)...")
    horizon_metrics = toolkit.compute_horizon_metrics(task, dataset, counterfactuals, config, nldd_df=df_nldd)
    results["horizon"] = horizon_metrics
    results["horizon_raw"] = horizon_metrics.get("raw_samples", {})

    # Save Horizon Data
    if horizon_metrics["steps"]:
        n_steps = len(horizon_metrics["steps"])
        data_to_save = {
            "step_k": horizon_metrics["steps"],
            "nldd_mean": horizon_metrics["nldd_mean"],
            "nldd_sem": horizon_metrics.get("nldd_sem", horizon_metrics.get("nldd_err", [np.nan] * n_steps)),
            "nldd_err": horizon_metrics.get("nldd_err", horizon_metrics.get("nldd_sem", [np.nan] * n_steps)),
            "nldd_smoothed": horizon_metrics.get("nldd_smoothed", [np.nan] * n_steps),
            "rsa_mean": horizon_metrics["rsa_mean"],
            "rsa_sem": horizon_metrics.get("rsa_sem", horizon_metrics.get("rsa_err", [np.nan] * n_steps)),
            "rsa_err": horizon_metrics.get("rsa_err", horizon_metrics.get("rsa_sem", [np.nan] * n_steps)),
            "horizon_k_primary": [horizon_metrics.get("horizon_k_primary", np.nan)] * n_steps,
            "horizon_k_alt": [horizon_metrics.get("horizon_k_alt", np.nan)] * n_steps,
            "horizon_primary_method": [horizon_metrics.get("horizon_primary_method", "unavailable")] * n_steps,
            "horizon_alt_method": [horizon_metrics.get("horizon_alt_method", "unavailable")] * n_steps,
            "interval_method": [horizon_metrics.get("interval_method", "raw_samples_primary")] * n_steps,
            "curve_error_method": [horizon_metrics.get("curve_error_method", "summary_t_from_sem")] * n_steps,
            "tas_definition": [horizon_metrics.get("tas_definition", "clean_prefix_efficiency")] * n_steps,
        }

        # Add n_samples for CI calculation
        if "n_samples" in horizon_metrics:
            data_to_save["n_samples"] = [horizon_metrics["n_samples"]] * n_steps
        if "n_per_step" in horizon_metrics:
            data_to_save["n_per_step"] = horizon_metrics["n_per_step"]

        # Check if TAS data is present
        if "tas_mean" in horizon_metrics and len(horizon_metrics["tas_mean"]) > 0:
            data_to_save["tas_mean"] = horizon_metrics["tas_mean"]
            data_to_save["tas_sem"] = horizon_metrics.get("tas_sem", horizon_metrics.get("tas_err", [np.nan] * n_steps))
            data_to_save["tas_err"] = horizon_metrics.get("tas_err", horizon_metrics.get("tas_sem", [np.nan] * n_steps))

        horizon_df = pd.DataFrame(data_to_save)
        horizon_csv_path = f"{config.results_dir}/horizon_{task_name}.csv"
        horizon_df.to_csv(horizon_csv_path, index=False)
        print(f"✓ Horizon curve saved to {horizon_csv_path}")
    else:
        print(f"   Warning: No measured horizon rows available for {task_name}")

    # Save Main Results
    csv_path = f"{config.results_dir}/nldd_{task_name}_with_modes.csv"
    df_nldd.to_csv(csv_path, index=False)
    print(f"\n✓ Main Results saved to {csv_path}")

    return results

In [ ]:
import os
os.listdir("./results_acl_2026")


## (8) Run Complete Analysis

After defining your tasks and datasets, run:

In [ ]:
 # ============================================================================
# (8) EXECUTE COMPLETE ANALYSIS
# ============================================================================

# 1. Instantiate the Mechanistic Toolkit
mechanistic_tool = MechanisticToolkit(model, tokenizer, config)

# 2. Storage for Final Summary
all_results = {}

for benchmark_key in BENCHMARK_ORDER:
    spec = benchmark_registry[benchmark_key]
    artifact = benchmark_artifacts[benchmark_key]

    results = run_enhanced_analysis(
        task=artifact["task"],
        dataset=artifact["dataset"],
        probe_dataset=artifact["probe_dataset"],
        counterfactuals=artifact["counterfactuals"],
        task_name=spec["output_stem"],
        toolkit=mechanistic_tool,
        config=config
    )
    all_results[spec["display_name"]] = results
    artifact["results"] = results
    globals()[BENCHMARK_RESULT_VAR[benchmark_key]] = results


## (9) Statistical Analysis


In [ ]:
# ============================================================================
# ACL 2026 CONFIDENCE INTERVAL & STATISTICAL SIGNIFICANCE BLOCK
# ============================================================================
#
# This cell implements comprehensive statistical analysis following ACL guidelines:
# - BCa Bootstrap Confidence Intervals (Efron, 1987)
# - Multiple Comparison Corrections (Holm-Bonferroni)
# - Effect Sizes (Hedges' g, Cliff's δ)
# ============================================================================

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import bootstrap, mannwhitneyu, wilcoxon, shapiro
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict
import warnings
import json
import os

warnings.filterwarnings('ignore', category=RuntimeWarning)

# statsmodels is imported in Section 0; fail closed if the dependency is missing.
from statsmodels.stats.multitest import multipletests


# ============================================================================
# Core Data Classes
# ============================================================================

@dataclass
class ConfidenceIntervalResult:
    """Container for confidence interval results."""
    point_estimate: float
    ci_lower: float
    ci_upper: float
    confidence_level: float
    method: str
    n_samples: int
    se: float = None

    def __repr__(self):
        return f"{self.point_estimate:.3f} [{self.ci_lower:.3f}, {self.ci_upper:.3f}]"

    def to_dict(self):
        return {
            "mean": round(self.point_estimate, 4),
            "ci_lower": round(self.ci_lower, 4),
            "ci_upper": round(self.ci_upper, 4),
            "ci_width": round(self.ci_upper - self.ci_lower, 4),
            "se": round(self.se, 4) if self.se else None,
            "n": self.n_samples,
            "method": self.method
        }


@dataclass
class EffectSizeResult:
    """Container for effect size results."""
    metric: str
    value: float
    ci_lower: float = None
    ci_upper: float = None
    interpretation: str = ""

    def to_dict(self):
        return {
            "metric": self.metric,
            "value": round(self.value, 4),
            "ci_lower": round(self.ci_lower, 4) if self.ci_lower else None,
            "ci_upper": round(self.ci_upper, 4) if self.ci_upper else None,
            "interpretation": self.interpretation
        }


# ============================================================================
# BCa Bootstrap Implementation
# ============================================================================

class BCaBootstrap:
    """
    Bias-Corrected and Accelerated (BCa) Bootstrap for confidence intervals.
    Recommended by Efron & Tibshirani (1993) for non-symmetric distributions.
    """

    def __init__(self, n_iterations: int = 10000, confidence_level: float = 0.95, seed: int = 42):
        self.n_iterations = n_iterations
        self.confidence_level = confidence_level
        self.rng = np.random.RandomState(seed)

    def compute_ci(self, data: np.ndarray, statistic=np.mean) -> ConfidenceIntervalResult:
        """Compute BCa bootstrap confidence interval."""
        data = np.asarray(data).flatten()
        data = data[~np.isnan(data)]
        n = len(data)

        if n < 3:
            return ConfidenceIntervalResult(
                point_estimate=np.mean(data) if n > 0 else np.nan,
                ci_lower=np.nan, ci_upper=np.nan,
                confidence_level=self.confidence_level,
                method="insufficient_data", n_samples=n
            )

        point_estimate = statistic(data)

        # Generate bootstrap distribution
        boot_stats = np.array([
            statistic(self.rng.choice(data, size=n, replace=True))
            for _ in range(self.n_iterations)
        ])
        boot_stats = boot_stats[~np.isnan(boot_stats)]

        if len(boot_stats) < 100:
            # Fallback to t-interval
            return self._t_interval(data, point_estimate)

        # BCa computation
        alpha = 1 - self.confidence_level

        # Bias correction (z0)
        prop_below = np.mean(boot_stats < point_estimate)
        prop_below = np.clip(prop_below, 0.0001, 0.9999)
        z0 = stats.norm.ppf(prop_below)

        # Acceleration (jackknife)
        jackknife = np.array([statistic(np.delete(data, i)) for i in range(n)])
        jack_mean = np.mean(jackknife)
        num = np.sum((jack_mean - jackknife) ** 3)
        denom = 6 * (np.sum((jack_mean - jackknife) ** 2) ** 1.5)
        a = num / denom if abs(denom) > 1e-10 else 0

        # Adjusted percentiles
        def adjusted_percentile(z_alpha):
            num = z0 + z_alpha
            denom = 1 - a * num
            if abs(denom) < 1e-10:
                return stats.norm.cdf(z_alpha)
            return stats.norm.cdf(z0 + num / denom)

        p_lower = np.clip(adjusted_percentile(stats.norm.ppf(alpha/2)), 0.001, 0.999)
        p_upper = np.clip(adjusted_percentile(stats.norm.ppf(1-alpha/2)), 0.001, 0.999)

        ci_lower = np.percentile(boot_stats, 100 * p_lower)
        ci_upper = np.percentile(boot_stats, 100 * p_upper)

        return ConfidenceIntervalResult(
            point_estimate=point_estimate,
            ci_lower=ci_lower,
            ci_upper=ci_upper,
            confidence_level=self.confidence_level,
            method="BCa",
            n_samples=n,
            se=np.std(boot_stats)
        )

    def _t_interval(self, data, point_est):
        """Fallback parametric t-interval."""
        n = len(data)
        se = stats.sem(data)
        t_crit = stats.t.ppf(1 - (1-self.confidence_level)/2, df=n-1)
        return ConfidenceIntervalResult(
            point_estimate=point_est,
            ci_lower=point_est - t_crit * se,
            ci_upper=point_est + t_crit * se,
            confidence_level=self.confidence_level,
            method="t_interval",
            n_samples=n,
            se=se
        )

    def paired_difference_ci(self, x: np.ndarray, y: np.ndarray) -> ConfidenceIntervalResult:
        """Compute CI for paired differences."""
        x, y = np.asarray(x), np.asarray(y)
        mask = ~(np.isnan(x) | np.isnan(y))
        return self.compute_ci(x[mask] - y[mask])


# ============================================================================
# Effect Size Calculators
# ============================================================================

def hedges_g(group1: np.ndarray, group2: np.ndarray) -> EffectSizeResult:
    """Hedges' g: Small-sample corrected standardized mean difference."""
    g1, g2 = np.asarray(group1), np.asarray(group2)
    g1, g2 = g1[~np.isnan(g1)], g2[~np.isnan(g2)]
    n1, n2 = len(g1), len(g2)

    if n1 < 2 or n2 < 2:
        return EffectSizeResult("hedges_g", np.nan, interpretation="insufficient data")

    mean_diff = np.mean(g1) - np.mean(g2)
    s1, s2 = np.var(g1, ddof=1), np.var(g2, ddof=1)
    pooled_sd = np.sqrt(((n1-1)*s1 + (n2-1)*s2) / (n1+n2-2))

    if pooled_sd < 1e-10:
        return EffectSizeResult("hedges_g", 0.0, interpretation="no variance")

    d = mean_diff / pooled_sd
    df = n1 + n2 - 2
    correction = 1 - (3 / (4*df - 1))
    g = d * correction

    # Simple interpretation
    abs_g = abs(g)
    if abs_g < 0.2: interp = "negligible"
    elif abs_g < 0.5: interp = "small"
    elif abs_g < 0.8: interp = "medium"
    else: interp = "large"

    return EffectSizeResult(
        metric="hedges_g", value=g, interpretation=interp
    )

def cliffs_delta(group1: np.ndarray, group2: np.ndarray) -> EffectSizeResult:
    """Cliff's delta: Non-parametric effect size."""
    g1, g2 = np.asarray(group1), np.asarray(group2)
    g1, g2 = g1[~np.isnan(g1)], g2[~np.isnan(g2)]
    n1, n2 = len(g1), len(g2)

    if n1 < 1 or n2 < 1:
        return EffectSizeResult("cliffs_delta", np.nan, interpretation="insufficient data")

    greater = np.sum(g1[:, None] > g2[None, :])
    less = np.sum(g1[:, None] < g2[None, :])
    delta = (greater - less) / (n1 * n2)

    abs_delta = abs(delta)
    if abs_delta < 0.147: interp = "negligible"
    elif abs_delta < 0.33: interp = "small"
    elif abs_delta < 0.474: interp = "medium"
    else: interp = "large"

    return EffectSizeResult(
        metric="cliffs_delta", value=delta, interpretation=interp
    )


# ============================================================================
# Main Statistical Analysis Class
# ============================================================================

class ACLConfidenceIntervalAnalyzer:
    def __init__(self, confidence_level=0.95, n_bootstrap=10000, alpha=0.05, seed=42):
        self.confidence_level = confidence_level
        self.n_bootstrap = n_bootstrap
        self.alpha = alpha
        self.bootstrap = BCaBootstrap(n_bootstrap, confidence_level, seed)
        self.rng = np.random.RandomState(seed)

    def analyze_metric(self, data: np.ndarray, metric_name: str = "metric") -> Dict:
        data = np.asarray(data).flatten()
        data = data[~np.isnan(data)]
        if len(data) < 3: return {"metric": metric_name, "error": "insufficient samples"}

        ci = self.bootstrap.compute_ci(data)
        return {
            "metric": metric_name,
            "mean": round(float(np.mean(data)), 4),
            "std": round(float(np.std(data, ddof=1)), 4),
            "ci": ci.to_dict()
        }

    def _analyze_scalar_horizon_samples(self, samples: List[float], metric_name: str) -> Dict:
        data = np.asarray(samples, dtype=float).flatten()
        data = data[np.isfinite(data)]
        if len(data) < 3:
            return {
                "mean": float(np.mean(data)) if len(data) > 0 else np.nan,
                "ci_lower": np.nan,
                "ci_upper": np.nan,
                "interval_method": "insufficient_data",
                "n": int(len(data)),
            }

        ci = self.bootstrap.compute_ci(data)
        return {
            "mean": float(np.mean(data)),
            "ci_lower": float(ci.ci_lower),
            "ci_upper": float(ci.ci_upper),
            "interval_method": ci.method,
            "n": int(len(data)),
        }

    def _analyze_rsa_horizon_pairs(self, clean: np.ndarray, corrupt: np.ndarray) -> Dict:
        clean = np.asarray(clean)
        corrupt = np.asarray(corrupt)
        n = min(len(clean), len(corrupt))
        if n < 3:
            return {
                "mean": np.nan,
                "ci_lower": np.nan,
                "ci_upper": np.nan,
                "interval_method": "insufficient_data",
                "n": int(n),
            }

        clean = clean[:n]
        corrupt = corrupt[:n]
        rsa_calc = RSACalculator(
            dissimilarity_metric="correlation",
            bootstrap_iterations=max(100, self.n_bootstrap),
            enable_noise_ceiling=False,
            enable_bootstrap=False,
            random_seed=42,
            device="cpu"
        )

        def rsa_stat(sample_indices):
            idx = np.asarray(sample_indices, dtype=int)
            if idx.size < 3:
                return np.nan
            try:
                result = rsa_calc.compute_rsa(clean[idx], corrupt[idx], return_detailed=False)
                return float(result["rsa_correlation"])
            except Exception:
                return np.nan

        base_idx = np.arange(n, dtype=int)
        point_est = rsa_stat(base_idx)
        boot_stats = []
        for _ in range(self.n_bootstrap):
            sample_idx = self.rng.choice(base_idx, size=n, replace=True)
            stat_val = rsa_stat(sample_idx)
            if np.isfinite(stat_val):
                boot_stats.append(stat_val)
        boot_stats = np.asarray(boot_stats, dtype=float)

        if boot_stats.size < 100:
            return {
                "mean": float(point_est) if np.isfinite(point_est) else np.nan,
                "ci_lower": np.nan,
                "ci_upper": np.nan,
                "interval_method": "insufficient_bootstrap",
                "n": int(n),
            }

        prop_below = np.mean(boot_stats < point_est)
        prop_below = np.clip(prop_below, 0.0001, 0.9999)
        z0 = stats.norm.ppf(prop_below)

        jackknife = []
        for i in range(n):
            jack_val = rsa_stat(np.delete(base_idx, i))
            if np.isfinite(jack_val):
                jackknife.append(jack_val)
        jackknife = np.asarray(jackknife, dtype=float)

        if jackknife.size < 3:
            ci_lower = float(np.percentile(boot_stats, 2.5))
            ci_upper = float(np.percentile(boot_stats, 97.5))
            return {
                "mean": float(point_est) if np.isfinite(point_est) else np.nan,
                "ci_lower": ci_lower,
                "ci_upper": ci_upper,
                "interval_method": "percentile_bootstrap",
                "n": int(n),
            }

        jack_mean = np.mean(jackknife)
        num = np.sum((jack_mean - jackknife) ** 3)
        denom = 6 * (np.sum((jack_mean - jackknife) ** 2) ** 1.5)
        accel = num / denom if abs(denom) > 1e-10 else 0.0

        alpha = 1 - self.confidence_level
        z_lower = stats.norm.ppf(alpha / 2)
        z_upper = stats.norm.ppf(1 - alpha / 2)

        def adjusted_percentile(z_alpha):
            num = z0 + z_alpha
            denom_inner = 1 - accel * num
            if abs(denom_inner) < 1e-10:
                return stats.norm.cdf(z_alpha)
            return stats.norm.cdf(z0 + num / denom_inner)

        p_lower = np.clip(adjusted_percentile(z_lower), 0.001, 0.999)
        p_upper = np.clip(adjusted_percentile(z_upper), 0.001, 0.999)
        ci_lower = float(np.percentile(boot_stats, 100 * p_lower))
        ci_upper = float(np.percentile(boot_stats, 100 * p_upper))
        return {
            "mean": float(point_est) if np.isfinite(point_est) else np.nan,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
            "interval_method": "BCa",
            "n": int(n),
        }

    def analyze_horizon_raw_samples(self, horizon_raw: Dict, task_name: str) -> Dict:
        results = {"task": task_name, "by_step": {}, "interval_method": "BCa"}
        if not horizon_raw:
            results["error"] = "missing_raw_samples"
            return results

        for step_k, samples in horizon_raw.get("nldd", {}).items():
            step = int(step_k)
            if step not in results["by_step"]:
                results["by_step"][step] = {}
            results["by_step"][step]["nldd"] = self._analyze_scalar_horizon_samples(samples, "nldd")

        for step_k, samples in horizon_raw.get("tas", {}).items():
            step = int(step_k)
            if step not in results["by_step"]:
                results["by_step"][step] = {}
            results["by_step"][step]["tas"] = self._analyze_scalar_horizon_samples(samples, "tas")

        for step_k, pair_data in horizon_raw.get("rsa", {}).items():
            step = int(step_k)
            if step not in results["by_step"]:
                results["by_step"][step] = {}
            clean = pair_data.get("clean", []) if isinstance(pair_data, dict) else []
            corrupt = pair_data.get("corrupt", []) if isinstance(pair_data, dict) else []
            results["by_step"][step]["rsa"] = self._analyze_rsa_horizon_pairs(clean, corrupt)

        return results

    def analyze_horizon_data(self, horizon_df: pd.DataFrame, task_name: str) -> Dict:
        results = {"task": task_name, "by_step": {}, "interval_method": "summary_t_from_sem"}
        for metric in ["nldd", "rsa", "tas"]:
            mean_col = f"{metric}_mean"
            sem_col = f"{metric}_sem"
            err_col = f"{metric}_err"
            if mean_col not in horizon_df.columns: continue

            for _, row in horizon_df.iterrows():
                mean_val = row.get(mean_col, np.nan)
                if pd.isna(mean_val):
                    continue

                k = int(row["step_k"])
                sem_val = row.get(sem_col, row.get(err_col, 0))
                if pd.isna(sem_val):
                    sem_val = 0
                n = row.get("n_per_step", row.get("n_samples", 20))
                if pd.isna(n):
                    n = row.get("n_samples", 20)
                n = int(max(1, n))

                # Reconstruct CI from summary SEM when raw per-step samples are unavailable.
                t_crit = stats.t.ppf(1 - self.alpha/2, df=max(1, n-1))
                ci_lower = mean_val - t_crit * sem_val
                ci_upper = mean_val + t_crit * sem_val

                if k not in results["by_step"]: results["by_step"][k] = {}
                results["by_step"][k][metric] = {
                    "mean": mean_val,
                    "ci_lower": ci_lower,
                    "ci_upper": ci_upper,
                    "interval_method": "summary_t_from_sem",
                    "n": n,
                }
        return results

    def analyze_nldd_results(self, nldd_df: pd.DataFrame, task_name: str) -> Dict:
        results = {"task": task_name}
        corruption_data = nldd_df[nldd_df["type"] == "corruption"]["nldd"].values
        results["corruption"] = self.analyze_metric(corruption_data, "NLDD_corruption")
        return results

    def generate_latex_table(self, results_dict: Dict) -> str:
        lines = [
            r"\begin{table}[t]", r"\centering", r"\small",
            r"\caption{Faithfulness Degradation Results with 95\% BCa Bootstrap CIs and effective sample counts}",
            r"\begin{tabular}{lccc}", r"\toprule", r"Task & NLDD & RSA & Probe Acc \\", r"\midrule"
        ]
        for task, res in results_dict.items():
            nldd_res = res.get("nldd", {}).get("corruption", {})
            nldd_val = f"{nldd_res.get('mean',0):.1f}" if nldd_res else "—"
            if "ci" in nldd_res:
                nldd_val += f" [{nldd_res['ci']['ci_lower']:.1f}, {nldd_res['ci']['ci_upper']:.1f}]"
            nldd_counts = res.get("nldd_counts", {})
            if nldd_counts:
                n_eff = nldd_counts.get("n_rows_valid", nldd_counts.get("n_rows_total", None))
                n_total = nldd_counts.get("n_rows_total", None)
                if n_eff is not None and n_total is not None:
                    nldd_val += f" (n={int(n_eff)}/{int(n_total)})"
                elif n_eff is not None:
                    nldd_val += f" (n={int(n_eff)})"

            rsa_val = f"{res.get('rsa_summary', {}).get('mean', 0):.3f}"
            probe_val = f"{res.get('probe_accuracy', 0)*100:.1f}"

            lines.append(f"{task} & {nldd_val} & {rsa_val} & {probe_val} \\\\")

        lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
        return "\n".join(lines)

# ============================================================================
# Integration Function
# ============================================================================

def run_statistical_analysis(all_results, config, output_dir="./results_acl_2026"):
    print("\n" + "="*70)
    print("ACL 2026 STATISTICAL ANALYSIS")
    print("="*70)

    analyzer = ACLConfidenceIntervalAnalyzer(
        confidence_level=config.confidence_level,
        n_bootstrap=config.n_bootstrap,
        alpha=1 - config.confidence_level,
        seed=config.seed
    )

    statistical_results = {}

    for task_name, task_results in all_results.items():
        print(f"\n# Analyzing: {task_name}")
        task_stats = {"task": task_name}

        # 1. NLDD
        if "nldd_df" in task_results:
            nldd_stats = analyzer.analyze_nldd_results(task_results["nldd_df"], task_name)
            task_stats["nldd"] = nldd_stats
            if task_results.get("nldd_audit") is not None:
                task_stats["nldd_counts"] = dict(task_results["nldd_audit"])

            # FIXED: Check for 'ci' presence and correct key usage ('mean' instead of 'point_estimate')
            if "corruption" in nldd_stats and "ci" in nldd_stats["corruption"]:
                ci = nldd_stats["corruption"]["ci"]
                print(f"   NLDD (95% CI): {ci['mean']:.2f} [{ci['ci_lower']:.2f}, {ci['ci_upper']:.2f}]")
            elif "error" in nldd_stats.get("corruption", {}):
                print(f"   NLDD: Insufficient data for CI ({nldd_stats['corruption']['error']})")

        # 2. Horizon
        horizon_path = f"{output_dir}/horizon_{task_name.lower().replace('-', '_').replace(' ', '_')}.csv"
        try:
            horizon_raw = task_results.get("horizon_raw", task_results.get("horizon", {}).get("raw_samples", {}))
            if horizon_raw:
                task_stats["horizon"] = analyzer.analyze_horizon_raw_samples(horizon_raw, task_name)
            elif os.path.exists(horizon_path):
                horizon_df = pd.read_csv(horizon_path)
                task_stats["horizon"] = analyzer.analyze_horizon_data(horizon_df, task_name)
        except Exception as e:
            print(f"   Warning processing horizon data: {e}")

        # 3. Aggregates
        if "rsa" in task_results:
            task_stats["rsa_summary"] = {"mean": task_results["rsa"].get("mean_rsa", 0)}
        if "probing" in task_results:
            # Handle potentially different probing return structures
            probe_data = task_results["probing"]
            acc = probe_data.get("aggregate_accuracy", probe_data.get("mean_accuracy", 0))
            task_stats["probe_accuracy"] = acc
        if task_results.get("cf_accuracy_summary_df") is not None and isinstance(task_results.get("cf_accuracy_summary_df"), pd.DataFrame) and not task_results["cf_accuracy_summary_df"].empty:
            task_stats["counterfactual_counts"] = task_results["cf_accuracy_summary_df"].iloc[0].to_dict()

        statistical_results[task_name] = task_stats

    # Generate Output
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}/statistical_analysis.json", "w") as f:
        # Custom converter for numpy types
        def convert(o):
            if isinstance(o, np.generic): return o.item()
            raise TypeError
        json.dump(statistical_results, f, indent=2, default=convert)

    latex = analyzer.generate_latex_table(statistical_results)
    with open(f"{output_dir}/results_table.tex", "w") as f:
        f.write(latex)

    print(f"\n✓ Analysis saved to {output_dir}/statistical_analysis.json")
    print(f"✓ LaTeX Table saved to {output_dir}/results_table.tex")
    print(f"\n{latex}")

# Auto-Run if results exist
if 'all_results' in globals() and all_results:
    run_statistical_analysis(all_results, config, config.results_dir)

## (10) Visualization


visual 1 should work if not do visual 2

In [ ]:
# ============================================================================
# SECTION 10: ACL 2026 VISUALIZATION SUITE (FINAL: FIXED LEGEND)
# ============================================================================

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import os
import pandas as pd
from scipy import stats
import numpy as np

# --- 1. GLOBAL ACL 2026 CONFIGURATION ---
# IBM Design Library Palette (Magenta/Blue/Orange triad)
COLOR_PALETTE = [
    "#DC267F",  # 0: Magenta  -> NLDD (Faithfulness)
    "#648FFF",  # 1: Blue     -> RSA (Alignment) / Corrupt
    "#FE6100",  # 2: Orange   -> TAS (Efficiency) / Clean
    "#785EF0"   # 3: Purple   -> Extra
]

# Check for LaTeX installation
def is_latex_installed():
    return shutil.which("latex") is not None

LATEX_INSTALLED = is_latex_installed()
if not LATEX_INSTALLED:
    print("⚠️ LaTeX not found. Falling back to default matplotlib rendering.")

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "text.usetex": LATEX_INSTALLED,
    "mathtext.fontset": "stix",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.figsize": (3.33, 2.5),
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.format": "pdf",
    "axes.prop_cycle": mpl.cycler(
        color=COLOR_PALETTE,
        linestyle=["-", "--", "-.", ":"],
        marker=["s", "^", "o", "D"]
    ),
    "lines.linewidth": 1.5,
    "lines.markersize": 3.5,
    "grid.alpha": 0.3,
    "grid.linestyle": ":",
})


def get_benchmark_specs_in_order() -> List[Tuple[str, Dict[str, Any]]]:
    if 'benchmark_registry' in globals() and 'BENCHMARK_ORDER' in globals():
        return [(key, benchmark_registry[key]) for key in BENCHMARK_ORDER if key in benchmark_registry]
    return [
        ("dyck", {"display_name": "Dyck-n", "output_stem": "dyck"}),
        ("pronto_qa", {"display_name": "ProntoQA", "output_stem": "pronto_qa"}),
        ("gsm8k", {"display_name": "GSM8K", "output_stem": "gsm8k"}),
        ("strategyqa", {"display_name": "StrategyQA", "output_stem": "strategyqa"}),
    ]


def get_benchmark_display_names() -> List[str]:
    return [spec["display_name"] for _, spec in get_benchmark_specs_in_order()]


def get_benchmark_output_stems() -> List[str]:
    return [spec["output_stem"] for _, spec in get_benchmark_specs_in_order()]


def get_benchmark_horizon_csv_pairs() -> List[Tuple[str, str]]:
    return [
        (f"{config.results_dir}/horizon_{spec['output_stem']}.csv", spec["display_name"])
        for _, spec in get_benchmark_specs_in_order()
    ]


def get_task_palette(n: int) -> List[str]:
    return [COLOR_PALETTE[i % len(COLOR_PALETTE)] for i in range(n)]


# --- SHARED HORIZON DETECTION HELPER ---
def detect_reasoning_horizon(df, nldd_col="nldd_mean", method="steepest_drop"):
    """
    Detect reasoning horizon k* from a horizon DataFrame.
    
    Args:
        df: DataFrame with columns 'step_k' and nldd_col
        nldd_col: column name for NLDD values
        method: "steepest_drop" (argmin of diff, canonical) or "peak" (argmax)
    
    Returns:
        int: detected horizon step k*
    """
    from scipy.ndimage import gaussian_filter1d
    
    df_temp = df.copy().sort_values("step_k").reset_index(drop=True)
    smoothed = gaussian_filter1d(df_temp[nldd_col].values, sigma=1.0)
    df_temp["nldd_mean_smoothed"] = smoothed
    
    valid_mask = (df_temp["step_k"] > 1) & (df_temp["step_k"] < df_temp["step_k"].max())
    valid_df = df_temp[valid_mask]
    
    if valid_df.empty:
        return int(df["step_k"].iloc[-1]) if len(df) > 0 else 0
    
    if method == "peak":
        idx = valid_df["nldd_mean_smoothed"].idxmax()
    else:  # "steepest_drop"
        idx = valid_df["nldd_mean_smoothed"].diff().fillna(0).idxmin()
    
    return int(valid_df.loc[idx, "step_k"])


# --- 2. GRAPH 1: Reasoning Horizon ---
def plot_reasoning_horizon_acl(csv_path, task_name, config, ylim_nldd=None):
    if not os.path.exists(csv_path):
        print(f"Skipping {task_name}: CSV not found.")
        return

    df = pd.read_csv(csv_path)
    if df.empty: return

    # Scale Fix
    if df["nldd_mean"].abs().max() < 1.5:
        df["nldd_mean"] *= 100
        for col in ["nldd_sem", "nldd_err"]:
            if col in df.columns: df[col] *= 100
    df = df.sort_values("step_k").reset_index(drop=True)

    fig, ax1 = plt.subplots()
    ax2 = ax1.twinx()
    steps = df["step_k"]

    color_nldd = COLOR_PALETTE[0]
    ax1.plot(steps, df["nldd_mean"], marker='s', label='NLDD', color=color_nldd, clip_on=False)

    err_col = 'nldd_err' if 'nldd_err' in df.columns else 'nldd_sem'
    if err_col in df.columns:
        ax1.fill_between(steps, df["nldd_mean"] - df[err_col], df["nldd_mean"] + df[err_col], color=color_nldd, alpha=0.15)

    color_rsa = COLOR_PALETTE[1]
    ax2.plot(steps, df["rsa_mean"], marker='^', label='RSA', color=color_rsa, clip_on=False)

    rsa_err = 'rsa_err' if 'rsa_err' in df.columns else 'rsa_sem'
    if rsa_err in df.columns:
        ax2.fill_between(steps, df["rsa_mean"] - df[rsa_err], df["rsa_mean"] + df[rsa_err], color=color_rsa, alpha=0.15)

    if "tas_mean" in df.columns:
        color_tas = COLOR_PALETTE[2]
        ax2.plot(steps, df["tas_mean"], marker='o', label='TAS (efficiency)', color=color_tas, clip_on=False)

        tas_err = 'tas_sem' if 'tas_sem' in df.columns else 'tas_err'
        if tas_err in df.columns:
            ax2.fill_between(steps, df["tas_mean"] - df[tas_err], df["tas_mean"] + df[tas_err], color=color_tas, alpha=0.12)

    ax1.set_xlabel("Reasoning Depth ($k$)")
    ax1.set_ylabel(r"NLDD (\%)" if LATEX_INSTALLED else "NLDD (%)", color=color_nldd)
    ax1.tick_params(axis='y', labelcolor=color_nldd)
    ax1.grid(True)

    ax2.set_ylabel("RSA / TAS Efficiency")
    ax2.tick_params(axis='y')
    ax2.set_ylim(0.0, 1.05)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, labels1 + labels2, loc='lower center', bbox_to_anchor=(0.5, 1.05), ncol=3, frameon=False)

    ax1.set_xlim(df["step_k"].min(), df["step_k"].max())

    if ylim_nldd is not None:
        ax1.set_ylim(ylim_nldd)

    if "horizon_k_primary" in df.columns and df["horizon_k_primary"].notna().any():
        horizon_step = df["horizon_k_primary"].dropna().iloc[0]
    else:
        horizon_step = detect_reasoning_horizon(df, method=config.horizon_method if hasattr(config, 'horizon_method') else "steepest_drop")

    if pd.notna(horizon_step):
        ax1.axvline(x=horizon_step, color='black', linestyle=':', linewidth=2.0, alpha=0.9)
        text_y = ax1.get_ylim()[1] - 0.05 * (ax1.get_ylim()[1] - ax1.get_ylim()[0])
        ax1.text(horizon_step + 0.15, text_y, f'$k^*={int(horizon_step)}$', ha='left', va='top', fontsize=9, fontweight='bold')

    save_path = f"./results_acl_2026/fig1_horizon_{task_name.lower()}.pdf"
    plt.savefig(save_path)
    print(f"✓ Saved plot: {save_path}")
    plt.show()
    plt.close(fig)


# --- 3. GRAPH 2: Accuracy Comparison ---
def plot_accuracy_comparison_acl(all_results):
    tasks = get_benchmark_display_names()
    clean_accs, corrupt_accs, clean_cis, corrupt_cis = [], [], [], []
    valid_tasks = []

    for task in tasks:
        if task in all_results and all_results[task].get("cf_accuracy_df") is not None:
            df = all_results[task]["cf_accuracy_df"]
            cdf = df[df["type"] == "corruption"]
            if len(cdf) == 0:
                continue

            valid_tasks.append(task)
            n = len(cdf)
            c_acc = cdf["clean_correct"].mean() * 100
            corr_acc = cdf["corrupt_correct"].mean() * 100
            clean_accs.append(c_acc)
            corrupt_accs.append(corr_acc)

            t_crit = stats.t.ppf(0.975, n-1)
            c_se = np.sqrt((c_acc/100)*(1-c_acc/100)/n) * 100
            corr_se = np.sqrt((corr_acc/100)*(1-corr_acc/100)/n) * 100
            clean_cis.append(t_crit * c_se)
            corrupt_cis.append(t_crit * corr_se)

    if not valid_tasks:
        return

    fig, ax = plt.subplots()
    x = np.arange(len(valid_tasks))
    width = 0.35

    rects1 = ax.bar(x - width/2, clean_accs, width, yerr=clean_cis, capsize=3, label='Clean', color=COLOR_PALETTE[2], edgecolor='black', linewidth=0.5)
    rects2 = ax.bar(x + width/2, corrupt_accs, width, yerr=corrupt_cis, capsize=3, label='Corrupt', color=COLOR_PALETTE[1], edgecolor='black', linewidth=0.5)

    ax.set_ylabel(r'Accuracy (\%)' if LATEX_INSTALLED else 'Accuracy (%)')
    ax.set_xticks(x)
    ax.set_xticklabels(valid_tasks)
    ax.set_ylim(0, 115)
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.05), ncol=2, frameon=False)

    ax.bar_label(rects1, fmt='%.0f', padding=2, fontsize=8)
    ax.bar_label(rects2, fmt='%.0f', padding=2, fontsize=8)

    sns.despine()
    ax.yaxis.grid(True)

    save_path = './results_acl_2026/accuracy_comparison.pdf'
    plt.savefig(save_path)
    print(f"✓ Saved plot: {save_path}")
    plt.show()
    plt.close(fig)


# --- 4. GRAPH 3: Probability Delta ---
def plot_probability_delta_acl(all_results):
    tasks = get_benchmark_display_names()
    combined_df = []

    for task in tasks:
        if task in all_results and all_results[task].get("cf_accuracy_df") is not None:
            df = all_results[task]["cf_accuracy_df"]
            cdf = df[df["type"] == "corruption"].copy()
            if len(cdf) > 0:
                cdf["task"] = task
                combined_df.append(cdf[["task", "prob_delta"]])

    if combined_df:
        combined_df = pd.concat(combined_df, ignore_index=True)
        task_order = [task for task in tasks if task in combined_df["task"].unique()]

        fig, ax = plt.subplots()
        sns.violinplot(
            data=combined_df,
            x="task",
            y="prob_delta",
            order=task_order,
            inner="quartile",
            palette=get_task_palette(len(task_order)),
            linewidth=1.0,
            cut=0,
            ax=ax,
        )

        ax.axhline(0, color='black', linestyle=':', linewidth=1.0, alpha=0.8)
        ax.set_ylabel(r'$\Delta$ Probability' if LATEX_INSTALLED else 'Delta Probability')
        ax.set_xlabel('')

        sns.despine()
        ax.yaxis.grid(True)

        save_path = './results_acl_2026/prob_delta_distribution.pdf'
        plt.savefig(save_path)
        print(f"✓ Saved plot: {save_path}")
        plt.show()
        plt.close(fig)
    else:
        print("⚠️ No data for probability delta.")


visual 2

In [ ]:
# Deprecated duplicate visualization suite removed.
# The canonical plotting helpers are defined earlier in Section 10.


In [ ]:
# ============================================================================
# EXECUTE ACL VISUALIZATIONS
# ============================================================================

# 1. Horizon Plots (with synchronized NLDD y-axis across all tasks)
print("Generating Horizon Plots (synchronized axes)...")

horizon_csvs = get_benchmark_horizon_csv_pairs()

# --- Pass 1: Compute shared NLDD y-axis limits across all tasks ---
global_nldd_min = float('inf')
global_nldd_max = float('-inf')

for csv_path, _ in horizon_csvs:
    if not os.path.exists(csv_path): continue
    df_tmp = pd.read_csv(csv_path)
    if df_tmp.empty: continue
    if df_tmp["nldd_mean"].abs().max() < 1.5:
        df_tmp["nldd_mean"] *= 100
        for col in ["nldd_sem", "nldd_err"]:
            if col in df_tmp.columns: df_tmp[col] *= 100
    err_col = 'nldd_err' if 'nldd_err' in df_tmp.columns else 'nldd_sem'
    err_vals = df_tmp[err_col] if err_col in df_tmp.columns else 0
    local_min = (df_tmp["nldd_mean"] - err_vals).min()
    local_max = (df_tmp["nldd_mean"] + err_vals).max()
    global_nldd_min = min(global_nldd_min, local_min)
    global_nldd_max = max(global_nldd_max, local_max)

if global_nldd_min < float('inf'):
    nldd_range = global_nldd_max - global_nldd_min
    shared_ylim = (global_nldd_min - 0.15 * nldd_range, global_nldd_max + 0.45 * nldd_range)
    print(f"  Shared NLDD y-axis: [{shared_ylim[0]:.1f}, {shared_ylim[1]:.1f}]")
else:
    shared_ylim = None

for csv_path, task_name in horizon_csvs:
    plot_reasoning_horizon_acl(csv_path, task_name, config, ylim_nldd=shared_ylim)

print("Generating Accuracy Plots...")
plot_accuracy_comparison_acl(all_results)

print("Generating Probability Delta Plots...")
plot_probability_delta_acl(all_results)


In [ ]:
# ===========================================================================
# ACL Robustness Analyzer (Statistical Significance Module)
# ===========================================================================
class ACLRobustnessAnalyzer:
    """
    Generates the 'Full Stats Report' required for ACL submission.
    Calculates statistical significance of the faithfulness decay (NLDD)
    and the alignment drop (RSA) at the Reasoning Horizon (k*).
    """
    def __init__(self, df: pd.DataFrame, task_name: str):
        self.df = df
        self.task_name = task_name

    def generate_full_report(self, horizon_k: int) -> Dict:
        """
        Compares metrics at step k* (Horizon) vs the earliest available depth.
        """
        print(f"\n[Robustness Report] {self.task_name} (Horizon k*={horizon_k})")

        # 1. Get data at the earliest available depth and at the detected horizon.
        # Horizon CSVs are stored using one-based step_k.
        baseline_k = self.df['step_k'].min()

        row_base = self.df[self.df['step_k'] == baseline_k].iloc[0]
        row_horz = self.df[self.df['step_k'] == horizon_k].iloc[0]

        stats_report = {}

        # 2. Analyze Metrics (NLDD, RSA, TAS)
        for metric, label in [('nldd', 'Faithfulness'), ('rsa', 'Alignment'), ('tas', 'Efficiency')]:
            if f'{metric}_mean' not in self.df.columns: continue

            val_base = row_base[f'{metric}_mean']
            val_horz = row_horz[f'{metric}_mean']
            if pd.isna(val_base) or pd.isna(val_horz):
                continue

            err_base = row_base.get(f'{metric}_sem', row_base.get(f'{metric}_err', 0))
            err_horz = row_horz.get(f'{metric}_sem', row_horz.get(f'{metric}_err', 0))
            err_base = 0 if pd.isna(err_base) else err_base
            err_horz = 0 if pd.isna(err_horz) else err_horz

            # T-test approximation using row-level SEM and per-step sample counts.
            n_base = row_base.get('n_per_step', row_base.get('n_samples', 20))
            n_horz = row_horz.get('n_per_step', row_horz.get('n_samples', 20))
            n_base = int(max(2, n_base)) if not pd.isna(n_base) else 20
            n_horz = int(max(2, n_horz)) if not pd.isna(n_horz) else 20
            se_diff = np.sqrt(err_base**2 + err_horz**2)
            t_stat = abs(val_base - val_horz) / (se_diff + 1e-9)

            # Degrees of freedom approx (Welch-Satterthwaite would be better but N is equal)
            df_stat = max(1, n_base + n_horz - 2)
            p_val = stats.t.sf(t_stat, df_stat) * 2  # two-sided

            # Effect Size (Cohen's d approx)
            # We treat SEM * sqrt(N) as SD
            sd_pooled = np.sqrt(((err_base * np.sqrt(n_base))**2 + (err_horz * np.sqrt(n_horz))**2) / 2)
            cohens_d = abs(val_base - val_horz) / (sd_pooled + 1e-9)

            stats_report[metric] = {
                'baseline': val_base,
                'horizon': val_horz,
                'delta': val_horz - val_base,
                'p_value': p_val,
                'cohens_d': cohens_d,
                'significant': p_val < 0.05
            }

            sig_mark = "*" if p_val < 0.05 else "ns"
            if p_val < 0.001: sig_mark = "***"
            elif p_val < 0.01: sig_mark = "**"

            print(f"   > {label}: {val_base:.1f} -> {val_horz:.1f} (d={cohens_d:.2f}, p={p_val:.1e} {sig_mark})")

        return stats_report

In [ ]:
# ============================================================
# RUN ROBUSTNESS ANALYSIS ON ALL TASKS
# ============================================================

print("\n" + "="*70)
print("RUNNING ROBUSTNESS ANALYSIS")
print("="*70)

tasks_info = {task_name: csv_path for csv_path, task_name in get_benchmark_horizon_csv_pairs()}
robustness_results = {}

for task_name, csv_path in tasks_info.items():
    print(f"\n{'#'*60}")
    print(f"# {task_name}")
    print(f"{'#'*60}")

    df = pd.read_csv(csv_path)

    if df["nldd_mean"].abs().max() < 2.0:
        print(f"  Converting NLDD to percentage scale...")
        cols = [c for c in df.columns if 'nldd' in c]
        df[cols] *= 100

    if 'n_samples' not in df.columns:
        df['n_samples'] = 20
    if 'n_per_step' not in df.columns:
        df['n_per_step'] = df['n_samples']

    from scipy import stats

    for m in ['nldd', 'rsa', 'tas']:
        mean_col = f'{m}_mean'
        sem_col = f'{m}_sem' if f'{m}_sem' in df.columns else f'{m}_err'

        if mean_col in df.columns and sem_col in df.columns:
            n_vals = df['n_per_step'].fillna(df['n_samples']).clip(lower=2)
            t_crit = stats.t.ppf(0.975, df=n_vals - 1)
            df[f'{m}_ci_lower'] = df[mean_col] - (t_crit * df[sem_col])
            df[f'{m}_ci_upper'] = df[mean_col] + (t_crit * df[sem_col])

    _method = config.horizon_method if hasattr(config, 'horizon_method') else "steepest_drop"
    if 'horizon_k_primary' in df.columns and df['horizon_k_primary'].notna().any():
        horizon_k = int(df['horizon_k_primary'].dropna().iloc[0])
    else:
        horizon_k = detect_reasoning_horizon(df, method=_method)

    alt_label = "peak" if _method == "steepest_drop" else "steepest_drop"
    if 'horizon_k_alt' in df.columns and df['horizon_k_alt'].notna().any():
        horizon_k_alt = int(df['horizon_k_alt'].dropna().iloc[0])
    else:
        horizon_k_alt = detect_reasoning_horizon(df, method=alt_label)

    print(f"\n  Detected horizon: k*={horizon_k} ({_method})")
    print(f"  Robustness check: k*={horizon_k_alt} ({alt_label})")

    analyzer = ACLRobustnessAnalyzer(df, task_name)
    results = analyzer.generate_full_report(horizon_k)

    robustness_results[task_name] = {
        'horizon': horizon_k,
        'robustness': results
    }

print("\n" + "="*70)
print("✅ ROBUSTNESS ANALYSIS COMPLETE")
print("="*70)


## (11) Pruning Cost-Benefit Analysis

Evaluates the trade-off between token savings and accuracy retention when truncating reasoning chains at the detected horizon k*.
Produces `pruning_analysis.csv` (per-sample) and `pruning_summary.csv` (aggregated).

In [ ]:
# ============================================================================
# SECTION 11: PRUNING COST-BENEFIT ANALYSIS
# ============================================================================

class PruningAnalyzer:
    """
    Evaluates the cost-benefit of truncating reasoning chains at the
    detected reasoning horizon k*.

    For each task x model combination:
    1. Detect k* from horizon CSV via detect_reasoning_horizon()
    2. Build pruned chains: steps[:k*]
    3. Evaluate accuracy on pruned vs full chains
    4. Count tokens (full vs pruned)
    5. Output CSV with token savings, agreement, crash rate, and retention ratio metrics
    """

    def __init__(self, model, tokenizer, config):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
        self.device = next(model.parameters()).device
        self.last_task_audit = {}
        self.last_failures_df = pd.DataFrame()

    def count_tokens(self, text: str) -> int:
        """Count tokens in a text string."""
        return len(self.tokenizer.encode(text, add_special_tokens=False))

    def _build_replacement_steps(self, sample, cf) -> List[str]:
        """Rebuild the full replacement chain used during horizon estimation."""
        return build_replacement_steps(sample, cf)

    def estimate_horizon_cost(self, task, dataset: List, counterfactuals: Dict[int, List]) -> Dict[str, float]:
        """Estimate the token cost of the clean+corrupt prompts used to locate k*."""
        total_tokens = 0
        n_clean_evals = 0
        n_corrupt_evals = 0

        if not counterfactuals:
            return {
                "estimation_tokens": 0,
                "n_clean_evals": 0,
                "n_corrupt_evals": 0,
                "mean_estimation_tokens_per_eval": 0.0,
            }

        for idx, sample in enumerate(dataset):
            sample_cf_rows = get_analysis_counterfactual_rows(
                sample,
                counterfactuals.get(idx) if counterfactuals is not None else None,
                config=self.config,
                include_controls=False
            )
            if not sample_cf_rows:
                continue

            clean_prompt = task.build_prompt(sample.input, sample.steps)
            total_tokens += self.count_tokens(clean_prompt)
            n_clean_evals += 1

            for cf_row in sample_cf_rows:
                replacement_steps = cf_row["replacement_steps"]
                corrupt_prompt = task.build_prompt(sample.input, replacement_steps)
                total_tokens += self.count_tokens(corrupt_prompt)
                n_corrupt_evals += 1

        total_evals = n_clean_evals + n_corrupt_evals
        return {
            "estimation_tokens": int(total_tokens),
            "n_clean_evals": int(n_clean_evals),
            "n_corrupt_evals": int(n_corrupt_evals),
            "mean_estimation_tokens_per_eval": float(total_tokens / total_evals) if total_evals > 0 else 0.0,
        }

    def analyze_task(
        self,
        task,
        dataset: List,
        horizon_k: int,
        task_name: str,
        model_name: str
    ) -> pd.DataFrame:
        """
        Run pruning analysis for a single task.

        Args:
            task: ReasoningTask instance
            dataset: List[TaskSample] - samples to evaluate
            horizon_k: int - detected k* for this task
            task_name: str
            model_name: str

        Returns:
            DataFrame with per-sample pruning results
        """
        cf_analyzer = CounterfactualAccuracyAnalyzer(
            self.model, self.tokenizer, task, self.config
        )

        results = []
        failure_rows = []
        skipped_short_chains = 0
        failed_samples = 0
        attempted_samples = 0

        for idx, sample in enumerate(tqdm(dataset, desc=f"Pruning: {task_name}")):
            try:
                # Skip samples with fewer steps than k*
                if len(sample.steps) <= horizon_k:
                    skipped_short_chains += 1
                    continue

                attempted_samples += 1
                # --- Full chain ---
                full_prompt = task.build_prompt(sample.input, sample.steps)
                full_tokens = self.count_tokens(full_prompt)
                full_correct, full_prob, _ = cf_analyzer._get_predicted_answer(
                    full_prompt, sample
                )

                # --- Pruned chain (truncate to first k* steps) ---
                pruned_steps = sample.steps[:horizon_k]
                pruned_prompt = task.build_prompt(sample.input, pruned_steps)
                pruned_tokens = self.count_tokens(pruned_prompt)
                pruned_correct, pruned_prob, _ = cf_analyzer._get_predicted_answer(
                    pruned_prompt, sample
                )

                # --- Metrics ---
                token_savings = full_tokens - pruned_tokens
                token_savings_pct = (token_savings / full_tokens * 100) if full_tokens > 0 else 0

                results.append({
                    "task": task_name,
                    "model": model_name,
                    "sample_idx": idx,
                    "n_steps_full": len(sample.steps),
                    "n_steps_pruned": len(pruned_steps),
                    "k_star": horizon_k,
                    "full_tokens": full_tokens,
                    "pruned_tokens": pruned_tokens,
                    "token_savings": token_savings,
                    "token_savings_pct": token_savings_pct,
                    "full_correct": full_correct,
                    "pruned_correct": pruned_correct,
                    "full_prob": full_prob,
                    "pruned_prob": pruned_prob,
                    "prediction_unchanged": int(full_correct == pruned_correct),
                    "causal_crash": int(full_correct and not pruned_correct),
                })

            except Exception as e:
                failed_samples += 1
                failure_rows.append({
                    "task": task_name,
                    "model": model_name,
                    "sample_idx": idx,
                    "horizon_k": horizon_k,
                    "failure_scope": "sample",
                    "exception_type": type(e).__name__,
                    "message": str(e)[:200],
                })
                continue

        self.last_failures_df = pd.DataFrame(failure_rows)
        self.last_task_audit = {
            "task": task_name,
            "model": model_name,
            "k_star": int(horizon_k),
            "n_dataset_samples": int(len(dataset)),
            "n_attempted_pruning_samples": int(attempted_samples),
            "n_successful_pruning_samples": int(len(results)),
            "n_skipped_shorter_than_horizon": int(skipped_short_chains),
            "n_pruning_failures": int(failed_samples),
        }
        if failed_samples:
            print(f"   [Pruning Audit] failures={failed_samples}, skipped_short={skipped_short_chains}")
        return pd.DataFrame(results)

    def run_full_analysis(
        self,
        tasks_data: Dict[str, Dict],
        model_name: str
    ) -> pd.DataFrame:
        """
        Run pruning analysis across all tasks.

        Args:
            tasks_data: {task_name: {"task": obj, "dataset": samples, "horizon_csv": path}}
            model_name: str - current model name

        Returns:
            Combined DataFrame; also saves CSVs to results_dir
        """
        all_results = []
        cost_by_task = {}
        audit_rows = []

        for task_name, data in tasks_data.items():
            task = data["task"]
            dataset = data["dataset"]
            csv_path = data["horizon_csv"]
            counterfactuals = data.get("counterfactuals", {})

            # Detect k* from horizon CSV
            if not os.path.exists(csv_path):
                print(f"WARNING: No horizon CSV for {task_name} at {csv_path}, skipping")
                continue

            horizon_df = pd.read_csv(csv_path)
            if horizon_df["nldd_mean"].abs().max() < 1.5:
                horizon_df["nldd_mean"] *= 100

            _method = self.config.horizon_method if hasattr(self.config, 'horizon_method') else "steepest_drop"
            if "horizon_k_primary" in horizon_df.columns and horizon_df["horizon_k_primary"].notna().any():
                horizon_k = int(horizon_df["horizon_k_primary"].dropna().iloc[0])
                horizon_source = "stored"
            else:
                horizon_k = detect_reasoning_horizon(horizon_df, method=_method)
                horizon_source = _method

            if pd.isna(horizon_k):
                print(f"WARNING: No valid horizon detected for {task_name}, skipping")
                continue

            print(f"\n{'='*60}")
            print(f"Pruning Analysis: {task_name} (k*={horizon_k}, source={horizon_source})")
            print(f"{'='*60}")

            cost_by_task[task_name] = self.estimate_horizon_cost(task, dataset, counterfactuals)
            df = self.analyze_task(task, dataset, horizon_k, task_name, model_name)
            if self.last_task_audit:
                audit_rows.append(self.last_task_audit)
            if self.last_failures_df is not None and not self.last_failures_df.empty:
                failures_path = f"{self.config.results_dir}/pruning_failures_{task_name}.csv"
                self.last_failures_df.to_csv(failures_path, index=False)
                print(f"Saved: {failures_path}")
            if not df.empty:
                all_results.append(df)

        if not all_results:
            print("WARNING: No pruning results generated.")
            return pd.DataFrame()

        combined = pd.concat(all_results, ignore_index=True)

        # --- Print Summary ---
        print(f"\n{'='*70}")
        print("PRUNING COST-BENEFIT SUMMARY")
        print(f"{'='*70}")

        summary = combined.groupby("task").agg(
            k_star=("k_star", "first"),
            n_samples=("sample_idx", "count"),
            mean_full_tokens=("full_tokens", "mean"),
            mean_pruned_tokens=("pruned_tokens", "mean"),
            mean_token_savings=("token_savings", "mean"),
            mean_token_savings_pct=("token_savings_pct", "mean"),
            full_accuracy=("full_correct", "mean"),
            pruned_accuracy=("pruned_correct", "mean"),
            prediction_unchanged_rate=("prediction_unchanged", "mean"),
            causal_crash_rate=("causal_crash", "mean"),
        ).reset_index()

        summary["full_accuracy_pct"] = summary["full_accuracy"] * 100
        summary["pruned_accuracy_pct"] = summary["pruned_accuracy"] * 100
        summary["prediction_unchanged_pct"] = summary["prediction_unchanged_rate"] * 100
        summary["causal_crash_pct"] = summary["causal_crash_rate"] * 100
        summary["accuracy_retention_ratio_pct"] = np.where(
            summary["full_accuracy"] > 0,
            summary["pruned_accuracy"] / summary["full_accuracy"] * 100,
            np.nan,
        )
        summary["mean_token_savings_per_query"] = summary["mean_token_savings"]

        cost_df = pd.DataFrame([
            {"task": task_name, **cost_info}
            for task_name, cost_info in cost_by_task.items()
        ])
        if not cost_df.empty:
            summary = summary.merge(cost_df, on="task", how="left")
        else:
            summary["estimation_tokens"] = 0
            summary["n_clean_evals"] = 0
            summary["n_corrupt_evals"] = 0
            summary["mean_estimation_tokens_per_eval"] = 0.0

        audit_df = pd.DataFrame(audit_rows)
        if not audit_df.empty:
            summary = summary.merge(audit_df, on=["task", "k_star"], how="left")

        summary["break_even_queries"] = np.where(
            summary["mean_token_savings_per_query"] > 0,
            np.ceil(summary["estimation_tokens"] / summary["mean_token_savings_per_query"]),
            np.nan,
        )

        print(summary[[
            "task", "k_star", "n_samples",
            "mean_token_savings_pct",
            "full_accuracy_pct", "pruned_accuracy_pct",
            "prediction_unchanged_pct", "accuracy_retention_ratio_pct",
            "causal_crash_pct", "break_even_queries"
        ]].to_string(index=False))

        # --- Save ---
        output_path = f"{self.config.results_dir}/pruning_analysis.csv"
        combined.to_csv(output_path, index=False)

        summary_path = f"{self.config.results_dir}/pruning_summary.csv"
        summary.to_csv(summary_path, index=False)

        print(f"\nSaved: {output_path}")
        print(f"Saved: {summary_path}")

        return combined


In [ ]:
# ============================================================================
# EXECUTE PRUNING COST-BENEFIT ANALYSIS
# ============================================================================

print("\n" + "="*70)
print("RUNNING PRUNING COST-BENEFIT ANALYSIS")
print("="*70)

pruning_analyzer = PruningAnalyzer(model, tokenizer, config)

pruning_tasks = {
    spec["display_name"]: {
        "task": benchmark_artifacts[benchmark_key]["task"],
        "dataset": benchmark_artifacts[benchmark_key]["dataset"],
        "counterfactuals": benchmark_artifacts[benchmark_key]["counterfactuals"],
        "horizon_csv": f"{config.results_dir}/horizon_{spec['output_stem']}.csv",
    }
    for benchmark_key, spec in get_benchmark_specs_in_order()
}

pruning_results_df = pruning_analyzer.run_full_analysis(
    pruning_tasks,
    model_name=model_manager.get_model_short_name()
)


## (12) Dataset Export (Gated: Upon Acceptance)

In [ ]:
# ============================================================================
# SECTION 12: DATASET EXPORT (Gated: Upon Acceptance)
# ============================================================================

EXPORT_ENABLED = False  # Set to True upon acceptance

if EXPORT_ENABLED:
    import shutil
    export_dir = "./release_data"
    os.makedirs(export_dir, exist_ok=True)

    for task_name in get_benchmark_output_stems():
        for pattern in [
            f"nldd_{task_name}_with_modes.csv",
            f"horizon_{task_name}.csv",
            f"accuracy_{task_name}.csv",
            f"accuracy_summary_{task_name}.csv",
            f"accuracy_audit_{task_name}.csv",
            f"accuracy_failures_{task_name}.csv",
            f"pruning_failures_{task_name}.csv"
        ]:
            src = f"{config.results_dir}/{pattern}"
            if os.path.exists(src):
                shutil.copy2(src, f"{export_dir}/{pattern}")
                print(f"  Exported: {pattern}")

    for fname in ["pruning_analysis.csv", "pruning_summary.csv"]:
        src = f"{config.results_dir}/{fname}"
        if os.path.exists(src):
            shutil.copy2(src, f"{export_dir}/{fname}")
            print(f"  Exported: {fname}")

    for fname in ["statistical_analysis.json", "results_table.tex"]:
        src = f"{config.results_dir}/{fname}"
        if os.path.exists(src):
            shutil.copy2(src, f"{export_dir}/{fname}")
            print(f"  Exported: {fname}")

    print(f"\nExported release data to {export_dir}/")
else:
    print("Dataset export is gated until acceptance. Set EXPORT_ENABLED=True to release.")
